# DDI-Edge Cold-Start Evaluation

This notebook constructs and evaluates the CHEERS R-GCN DDI-edge cold-start experiment.

The experiment compares:

- **G0:** DDI edges only.
- **G3:** DDI edges plus biomedical knowledge-graph relations.

A cold drug has all of its DDI edges removed from the training and message-passing DDI graph. In G3, non-DDI biomedical edges for cold drugs are retained.

> **Scope:** This is a **DDI-edge cold-start** experiment, not a fully inductive unseen-node experiment. Cold drugs remain graph nodes and the model architecture contains learned node embeddings.

The cold cohort is fixed using **split seed 42**. Model seeds **42, 43, and 44** measure model-training variation conditional on this single fixed cohort.


In [ ]:
from pathlib import Path

BASE = Path("/workspace/primekg_ddi_rgcn")

print("PROJECT:", BASE)

print("\nProcessed data:")
for p in sorted((BASE / "data" / "processed").glob("*")):
    print(p)

print("\nR-GCN tensors:")
for p in sorted(
    (BASE / "data" / "processed" / "rgcn_tensors").glob("*")
):
    print(p)

print("\nPossible split/result files:")
for p in sorted(BASE.rglob("*")):
    if p.is_file():
        name = p.name.lower()

        if any(
            keyword in name
            for keyword in [
                "split",
                "train_pair",
                "val_pair",
                "test_pair",
                "ddi_pair",
            ]
        ):
            print(p)
            

In [ ]:
import torch
from pathlib import Path

BASE = Path("/workspace/primekg_ddi_rgcn")
TENSOR_DIR = BASE / "data" / "processed" / "rgcn_tensors"

files_to_check = [
    "ddi_train.pt",
    "ddi_val.pt",
    "ddi_test.pt",
    "drug_node_ids.pt",
]

for filename in files_to_check:

    path = TENSOR_DIR / filename
    obj = torch.load(path, map_location="cpu")

    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    print("Python type:", type(obj))

    if torch.is_tensor(obj):
        print("Shape:", tuple(obj.shape))
        print("dtype:", obj.dtype)

        print("\nFirst 10 entries:")
        print(obj[:10])

        if obj.numel() > 0:
            print("\nMinimum value:", obj.min().item())
            print("Maximum value:", obj.max().item())

    elif isinstance(obj, dict):
        print("Keys:", list(obj.keys()))

        for key, value in obj.items():
            if torch.is_tensor(value):
                print(
                    f"{key}: shape={tuple(value.shape)}, "
                    f"dtype={value.dtype}"
                )
            else:
                print(f"{key}: {type(value)}")

## 1. Dataset and DDI Degree Inspection

Inspect the fixed DDI train/validation/test splits, candidate drug set, total DDI degree, and biomedical context available to candidate drugs.


In [ ]:
import torch
import numpy as np
from pathlib import Path

BASE = Path("/workspace/primekg_ddi_rgcn")
TENSOR_DIR = BASE / "data" / "processed" / "rgcn_tensors"

# ------------------------------------------------------------
# Load fixed DDI splits
# ------------------------------------------------------------

train_pairs = torch.load(
    TENSOR_DIR / "ddi_train.pt",
    map_location="cpu"
)["pair_index"]

val_pairs = torch.load(
    TENSOR_DIR / "ddi_val.pt",
    map_location="cpu"
)["pair_index"]

test_pairs = torch.load(
    TENSOR_DIR / "ddi_test.pt",
    map_location="cpu"
)["pair_index"]

drug_node_ids = torch.load(
    TENSOR_DIR / "drug_node_ids.pt",
    map_location="cpu"
)["drug_node_ids"]

print("Train pairs:", train_pairs.shape[1])
print("Val pairs:  ", val_pairs.shape[1])
print("Test pairs: ", test_pairs.shape[1])
print("Drugs:      ", len(drug_node_ids))


# ------------------------------------------------------------
# Combine all known DDI pairs
# ------------------------------------------------------------

all_pairs = torch.cat(
    [train_pairs, val_pairs, test_pairs],
    dim=1
)

print("\nAll known DDI pairs:", all_pairs.shape[1])


# ------------------------------------------------------------
# Calculate total DDI degree for every drug
# ------------------------------------------------------------

all_endpoints = all_pairs.reshape(-1)

unique_nodes, counts = torch.unique(
    all_endpoints,
    return_counts=True
)

degree_map = {
    int(node): int(count)
    for node, count in zip(unique_nodes, counts)
}

degrees = np.array(
    [
        degree_map.get(int(node), 0)
        for node in drug_node_ids
    ]
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nDDI DEGREE SUMMARY")
print("=" * 60)

print("Minimum degree:", degrees.min())
print("Maximum degree:", degrees.max())
print("Mean degree:   ", degrees.mean())
print("Median degree: ", np.median(degrees))

print("\nPercentiles:")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(
        f"{p:>2}th percentile:",
        np.percentile(degrees, p)
    )


# ------------------------------------------------------------
# Count drugs by degree threshold
# ------------------------------------------------------------

print("\nDRUG COUNTS BY MINIMUM DEGREE")
print("=" * 60)

for threshold in [1, 2, 5, 10, 20, 50, 100, 200, 500]:
    count = int((degrees >= threshold).sum())

    print(
        f"degree >= {threshold:>3}: "
        f"{count:>4} drugs "
        f"({100 * count / len(degrees):.2f}%)"
    )

In [ ]:
import torch
from pathlib import Path

BASE = Path("/workspace/primekg_ddi_rgcn")
TENSOR_DIR = BASE / "data" / "processed" / "rgcn_tensors"

g3 = torch.load(
    TENSOR_DIR / "G3.pt",
    map_location="cpu"
)

print("=" * 70)
print("G3 STRUCTURE")
print("=" * 70)

print("Python type:", type(g3))

if isinstance(g3, dict):

    print("\nKeys:")
    for key in g3.keys():
        print(" -", key)

    print("\nContents:")
    for key, value in g3.items():

        if torch.is_tensor(value):
            print(
                f"{key:30s}"
                f" shape={tuple(value.shape)},"
                f" dtype={value.dtype}"
            )

            if value.numel() > 0:
                print(
                    f"{'':30s}"
                    f" min={value.min().item()},"
                    f" max={value.max().item()}"
                )

        else:
            print(
                f"{key:30s}",
                type(value),
                value if isinstance(
                    value,
                    (int, float, str, list, tuple)
                ) else ""
            )

In [ ]:
# ============================================================
# STEP 5 — Biomedical context coverage for candidate drugs
# ============================================================

import torch
import numpy as np

# G3 was loaded in the previous cell
edge_index = g3["edge_index"]
edge_type = g3["edge_type"]

# ------------------------------------------------------------
# Relation 0 = DDI
# Relations 1-14 = biomedical context
# ------------------------------------------------------------

biomedical_mask = edge_type != 0

biomedical_edges = edge_index[:, biomedical_mask]
biomedical_types = edge_type[biomedical_mask]

print("Total G3 directed edges:       ", edge_index.shape[1])
print("DDI directed edges:            ", int((edge_type == 0).sum()))
print("Biomedical directed edges:     ", biomedical_edges.shape[1])

print("\nBiomedical relation IDs present:")
print(torch.unique(biomedical_types).tolist())


# ------------------------------------------------------------
# Count biomedical edges touching each node
# ------------------------------------------------------------

biomedical_endpoints = biomedical_edges.reshape(-1)

bio_nodes, bio_counts = torch.unique(
    biomedical_endpoints,
    return_counts=True
)

bio_degree_map = {
    int(node): int(count)
    for node, count in zip(bio_nodes, bio_counts)
}

drug_bio_degrees = np.array(
    [
        bio_degree_map.get(int(node), 0)
        for node in drug_node_ids
    ]
)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nBIOMEDICAL CONTEXT DEGREE FOR DRUGS")
print("=" * 60)

print("Minimum:", drug_bio_degrees.min())
print("Maximum:", drug_bio_degrees.max())
print("Mean:   ", drug_bio_degrees.mean())
print("Median: ", np.median(drug_bio_degrees))

print("\nPercentiles:")

for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(
        f"{p:>2}th percentile:",
        np.percentile(drug_bio_degrees, p)
    )


# ------------------------------------------------------------
# Number of drugs with biomedical context
# ------------------------------------------------------------

print("\nDRUG COUNTS BY MINIMUM BIOMEDICAL DEGREE")
print("=" * 60)

for threshold in [1, 2, 5, 10, 20, 50, 100]:

    count = int(
        (drug_bio_degrees >= threshold).sum()
    )

    print(
        f"bio degree >= {threshold:>3}: "
        f"{count:>4} drugs "
        f"({100 * count / len(drug_node_ids):.2f}%)"
    )


# ------------------------------------------------------------
# Joint DDI + biomedical coverage
# ------------------------------------------------------------

print("\nJOINT ELIGIBILITY")
print("=" * 60)

for ddi_threshold in [10, 20, 50, 100]:

    for bio_threshold in [1, 5, 10]:

        eligible = (
            (degrees >= ddi_threshold)
            &
            (drug_bio_degrees >= bio_threshold)
        )

        count = int(eligible.sum())

        print(
            f"DDI >= {ddi_threshold:>3}, "
            f"Bio >= {bio_threshold:>2}: "
            f"{count:>4} drugs "
            f"({100 * count / len(drug_node_ids):.2f}%)"
        )

    print()

In [ ]:
# ============================================================
# STEP 6 — Split-specific DDI degree per drug
# ============================================================

import numpy as np
import torch


def calculate_degree_map(pair_index):
    """
    Count how many DDI pairs each node participates in.
    """

    endpoints = pair_index.reshape(-1)

    nodes, counts = torch.unique(
        endpoints,
        return_counts=True
    )

    return {
        int(node): int(count)
        for node, count in zip(nodes, counts)
    }


# ------------------------------------------------------------
# Degree maps for each existing split
# ------------------------------------------------------------

train_degree_map = calculate_degree_map(train_pairs)
val_degree_map = calculate_degree_map(val_pairs)
test_degree_map = calculate_degree_map(test_pairs)


train_degrees = np.array([
    train_degree_map.get(int(node), 0)
    for node in drug_node_ids
])

val_degrees = np.array([
    val_degree_map.get(int(node), 0)
    for node in drug_node_ids
])

test_degrees = np.array([
    test_degree_map.get(int(node), 0)
    for node in drug_node_ids
])


# ------------------------------------------------------------
# Integrity check
# ------------------------------------------------------------

reconstructed_total = (
    train_degrees
    + val_degrees
    + test_degrees
)

print("TOTAL DEGREE INTEGRITY CHECK")
print("=" * 60)

print(
    "Matches previous total degree:",
    np.array_equal(
        reconstructed_total,
        degrees
    )
)

print(
    "Maximum difference:",
    np.abs(
        reconstructed_total - degrees
    ).max()
)


# ------------------------------------------------------------
# Split summaries
# ------------------------------------------------------------

print("\nSPLIT-SPECIFIC DEGREE SUMMARY")
print("=" * 60)

for name, values in [
    ("Train", train_degrees),
    ("Validation", val_degrees),
    ("Test", test_degrees),
]:

    print(f"\n{name}")

    print("  Minimum:", values.min())
    print("  Maximum:", values.max())
    print("  Mean:   ", values.mean())
    print("  Median: ", np.median(values))

    print(
        "  Drugs with >= 1 pair:",
        int((values >= 1).sum())
    )

    print(
        "  Drugs with >= 5 pairs:",
        int((values >= 5).sum())
    )

    print(
        "  Drugs with >= 10 pairs:",
        int((values >= 10).sum())
    )

    print(
        "  Drugs with >= 20 pairs:",
        int((values >= 20).sum())
    )


# ------------------------------------------------------------
# Candidate cold-start eligibility
# ------------------------------------------------------------

base_eligible = (
    (degrees >= 20)
    &
    (drug_bio_degrees >= 5)
)

print("\nCOLD-START CANDIDATE ANALYSIS")
print("=" * 60)

print(
    "Base eligible:",
    int(base_eligible.sum())
)

for min_test in [1, 2, 5, 10, 20]:

    mask = (
        base_eligible
        &
        (test_degrees >= min_test)
    )

    print(
        f"Base eligible + test degree >= {min_test:>2}: "
        f"{int(mask.sum()):>4} drugs"
    )
    

## 2. Cold-Drug Cohort Construction

Construct the fixed context-available cold-drug cohort.

Eligibility uses DDI-degree and biomedical-context requirements. Because total DDI degree and existing test DDI degree contribute to cohort eligibility, this cohort construction is **not test-blind**. This should be treated as an evaluation-design limitation rather than training-label leakage.


In [ ]:
# ============================================================
# STEP 7 — Examine eligible cold-start population
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Locked eligibility criteria
# ------------------------------------------------------------

MIN_TOTAL_DDI_DEGREE = 20
MIN_BIO_DEGREE = 5
MIN_TEST_DEGREE = 10

eligible_mask = (
    (degrees >= MIN_TOTAL_DDI_DEGREE)
    &
    (drug_bio_degrees >= MIN_BIO_DEGREE)
    &
    (test_degrees >= MIN_TEST_DEGREE)
)

eligible_indices = np.where(eligible_mask)[0]

eligible_drug_ids = (
    drug_node_ids[eligible_indices]
    .cpu()
    .numpy()
)


# ------------------------------------------------------------
# Build analysis table
# ------------------------------------------------------------

eligible_df = pd.DataFrame({
    "drug_node_id": eligible_drug_ids,
    "total_ddi_degree": degrees[eligible_indices],
    "train_ddi_degree": train_degrees[eligible_indices],
    "val_ddi_degree": val_degrees[eligible_indices],
    "test_ddi_degree": test_degrees[eligible_indices],
    "bio_degree": drug_bio_degrees[eligible_indices],
})


print("ELIGIBLE COLD-START POPULATION")
print("=" * 65)

print("Number of eligible drugs:", len(eligible_df))

print("\nDDI degree:")
print(
    eligible_df["total_ddi_degree"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

print("\nBiomedical degree:")
print(
    eligible_df["bio_degree"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)

print("\nTest DDI degree:")
print(
    eligible_df["test_ddi_degree"]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90
        ]
    )
)


In [ ]:
# ============================================================
# STEP 8 — Select cold-start drugs using stratified sampling
# ============================================================

import numpy as np
import pandas as pd

SPLIT_SEED = 42
COLD_FRACTION = 0.10

# ------------------------------------------------------------
# Create DDI-degree quartiles
# ------------------------------------------------------------

eligible_df = eligible_df.copy()

eligible_df["degree_stratum"] = pd.qcut(
    eligible_df["total_ddi_degree"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

print("ELIGIBLE DRUGS BY DDI-DEGREE STRATUM")
print("=" * 65)

print(
    eligible_df
    .groupby(
        "degree_stratum",
        observed=True
    )
    .agg(
        drugs=("drug_node_id", "count"),
        min_degree=("total_ddi_degree", "min"),
        median_degree=("total_ddi_degree", "median"),
        max_degree=("total_ddi_degree", "max"),
    )
)


# ------------------------------------------------------------
# Sample approximately 10% independently from each stratum
# ------------------------------------------------------------

cold_parts = []

for i, stratum in enumerate(["Q1", "Q2", "Q3", "Q4"]):

    group = eligible_df[
        eligible_df["degree_stratum"] == stratum
    ]

    n_select = round(
        len(group) * COLD_FRACTION
    )

    sampled = group.sample(
        n=n_select,
        random_state=SPLIT_SEED + i
    )

    cold_parts.append(sampled)


cold_df = pd.concat(
    cold_parts,
    ignore_index=True
)

# Sort only for reproducible display/storage.
cold_df = cold_df.sort_values(
    "drug_node_id"
).reset_index(drop=True)


# ------------------------------------------------------------
# Basic checks
# ------------------------------------------------------------

print("\nCOLD-START SELECTION")
print("=" * 65)

print("Split seed:", SPLIT_SEED)
print("Requested fraction:", COLD_FRACTION)
print("Eligible drugs:", len(eligible_df))
print("Selected cold drugs:", len(cold_df))

print(
    "Actual fraction:",
    len(cold_df) / len(eligible_df)
)


print("\nSELECTED DRUGS BY STRATUM")
print("=" * 65)

print(
    cold_df["degree_stratum"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# Compare eligible population vs selected cold population
# ------------------------------------------------------------

print("\nELIGIBLE vs COLD-START")
print("=" * 65)

comparison = pd.DataFrame({

    "Eligible median": [
        eligible_df["total_ddi_degree"].median(),
        eligible_df["bio_degree"].median(),
        eligible_df["test_ddi_degree"].median(),
    ],

    "Cold median": [
        cold_df["total_ddi_degree"].median(),
        cold_df["bio_degree"].median(),
        cold_df["test_ddi_degree"].median(),
    ],

    "Eligible mean": [
        eligible_df["total_ddi_degree"].mean(),
        eligible_df["bio_degree"].mean(),
        eligible_df["test_ddi_degree"].mean(),
    ],

    "Cold mean": [
        cold_df["total_ddi_degree"].mean(),
        cold_df["bio_degree"].mean(),
        cold_df["test_ddi_degree"].mean(),
    ],
},
index=[
    "Total DDI degree",
    "Biomedical degree",
    "Test DDI degree",
])

print(comparison)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert cold_df["drug_node_id"].is_unique

assert set(
    cold_df["drug_node_id"]
).issubset(
    set(eligible_df["drug_node_id"])
)

assert (
    cold_df["total_ddi_degree"]
    >= MIN_TOTAL_DDI_DEGREE
).all()

assert (
    cold_df["bio_degree"]
    >= MIN_BIO_DEGREE
).all()

assert (
    cold_df["test_ddi_degree"]
    >= MIN_TEST_DEGREE
).all()

print("\nSelection assertions passed.")

## 3. Leakage-Safe Cold/Warm Partition

Partition the existing fixed splits into warm-only, one-sided cold/warm, and two-sided cold/cold subsets.

All DDI edges touching a selected cold drug are removed from DDI training/message passing.


In [ ]:
# ============================================================
# STEP 9 — Measure effect of cold drugs on existing DDI splits
# ============================================================

import torch
import numpy as np

cold_drug_ids = torch.tensor(
    cold_df["drug_node_id"].to_numpy(),
    dtype=torch.long
)

print("Number of selected cold drugs:", len(cold_drug_ids))


# ------------------------------------------------------------
# Helper: identify pairs touching cold-start drugs
# ------------------------------------------------------------

def pairs_touching_cold(pair_index, cold_ids):
    """
    Returns a boolean mask of shape [num_pairs].

    True means at least one endpoint of the DDI pair
    is a selected cold-start drug.
    """

    src = pair_index[0]
    dst = pair_index[1]

    src_is_cold = torch.isin(src, cold_ids)
    dst_is_cold = torch.isin(dst, cold_ids)

    return src_is_cold | dst_is_cold


# ------------------------------------------------------------
# Calculate masks
# ------------------------------------------------------------

train_cold_mask = pairs_touching_cold(
    train_pairs,
    cold_drug_ids
)

val_cold_mask = pairs_touching_cold(
    val_pairs,
    cold_drug_ids
)

test_cold_mask = pairs_touching_cold(
    test_pairs,
    cold_drug_ids
)


# ------------------------------------------------------------
# Count affected pairs
# ------------------------------------------------------------

def report_split(name, pair_index, mask):

    total = pair_index.shape[1]
    affected = int(mask.sum())
    unaffected = total - affected

    print(f"\n{name}")
    print("-" * 50)
    print(f"Total pairs:       {total:,}")
    print(f"Touch cold drugs:  {affected:,}")
    print(f"Do not touch cold: {unaffected:,}")
    print(
        f"Affected fraction: "
        f"{100 * affected / total:.2f}%"
    )


print("\nIMPACT ON EXISTING DDI SPLITS")
print("=" * 65)

report_split(
    "TRAIN",
    train_pairs,
    train_cold_mask
)

report_split(
    "VALIDATION",
    val_pairs,
    val_cold_mask
)

report_split(
    "TEST",
    test_pairs,
    test_cold_mask
)


# ------------------------------------------------------------
# One-sided vs two-sided cold pairs
# ------------------------------------------------------------

def cold_endpoint_breakdown(pair_index, cold_ids):

    src_cold = torch.isin(
        pair_index[0],
        cold_ids
    )

    dst_cold = torch.isin(
        pair_index[1],
        cold_ids
    )

    one_sided = src_cold ^ dst_cold
    two_sided = src_cold & dst_cold

    return (
        int(one_sided.sum()),
        int(two_sided.sum())
    )


print("\nCOLD-PAIR BREAKDOWN")
print("=" * 65)

for name, pairs in [
    ("Train", train_pairs),
    ("Validation", val_pairs),
    ("Test", test_pairs),
]:

    one_sided, two_sided = (
        cold_endpoint_breakdown(
            pairs,
            cold_drug_ids
        )
    )

    print(
        f"{name:10s} "
        f"one-sided cold = {one_sided:>8,} | "
        f"two-sided cold = {two_sided:>8,}"
    )


# ------------------------------------------------------------
# Estimate remaining training data
# ------------------------------------------------------------

remaining_train_pairs = (
    train_pairs[:, ~train_cold_mask]
)

print("\nTRAINING GRAPH AFTER REMOVAL")
print("=" * 65)

print(
    "Original training pairs:",
    f"{train_pairs.shape[1]:,}"
)

print(
    "Pairs removed:",
    f"{int(train_cold_mask.sum()):,}"
)

print(
    "Pairs remaining:",
    f"{remaining_train_pairs.shape[1]:,}"
)

print(
    "Training data retained:",
    f"{100 * remaining_train_pairs.shape[1] / train_pairs.shape[1]:.2f}%"
)


In [ ]:
# ============================================================
# STEP 10 — Construct in-memory cold/warm partitions
#            and verify them
# ============================================================

# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

cold_train_pairs = train_pairs[:, train_cold_mask]
warm_train_pairs = train_pairs[:, ~train_cold_mask]


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

cold_val_pairs = val_pairs[:, val_cold_mask]
warm_val_pairs = val_pairs[:, ~val_cold_mask]


# ------------------------------------------------------------
# Test: separate one-sided and two-sided cold
# ------------------------------------------------------------

test_src_cold = torch.isin(
    test_pairs[0],
    cold_drug_ids
)

test_dst_cold = torch.isin(
    test_pairs[1],
    cold_drug_ids
)

test_one_sided_mask = (
    test_src_cold ^ test_dst_cold
)

test_two_sided_mask = (
    test_src_cold & test_dst_cold
)

test_warm_mask = (
    ~test_src_cold & ~test_dst_cold
)


cold_test_one_sided = (
    test_pairs[:, test_one_sided_mask]
)

cold_test_two_sided = (
    test_pairs[:, test_two_sided_mask]
)

warm_test_pairs = (
    test_pairs[:, test_warm_mask]
)


# ------------------------------------------------------------
# Helper
# ------------------------------------------------------------

def contains_cold(pair_index, cold_ids):

    if pair_index.shape[1] == 0:
        return False

    return bool(
        torch.isin(
            pair_index.reshape(-1),
            cold_ids
        ).any()
    )


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("FINAL IN-MEMORY PARTITIONS")
print("=" * 65)

print("\nTRAIN")
print(
    "Warm training pairs:",
    f"{warm_train_pairs.shape[1]:,}"
)
print(
    "Removed cold-touching pairs:",
    f"{cold_train_pairs.shape[1]:,}"
)

print("\nVALIDATION")
print(
    "Warm validation pairs:",
    f"{warm_val_pairs.shape[1]:,}"
)
print(
    "Cold-touching validation pairs:",
    f"{cold_val_pairs.shape[1]:,}"
)

print("\nTEST")
print(
    "Warm test pairs:",
    f"{warm_test_pairs.shape[1]:,}"
)
print(
    "One-sided cold test pairs:",
    f"{cold_test_one_sided.shape[1]:,}"
)
print(
    "Two-sided cold test pairs:",
    f"{cold_test_two_sided.shape[1]:,}"
)


# ------------------------------------------------------------
# Mandatory leakage assertions
# ------------------------------------------------------------

print("\nLEAKAGE CHECKS")
print("=" * 65)

assert not contains_cold(
    warm_train_pairs,
    cold_drug_ids
)

print(
    "PASS: no selected cold drug occurs "
    "in warm training DDI pairs."
)


assert not contains_cold(
    warm_val_pairs,
    cold_drug_ids
)

print(
    "PASS: no selected cold drug occurs "
    "in warm validation DDI pairs."
)


# Every one-sided test pair must contain exactly one cold drug

one_src = torch.isin(
    cold_test_one_sided[0],
    cold_drug_ids
)

one_dst = torch.isin(
    cold_test_one_sided[1],
    cold_drug_ids
)

assert torch.all(
    one_src ^ one_dst
)

print(
    "PASS: every primary cold test pair "
    "contains exactly one cold drug."
)


# Every two-sided test pair must contain two cold drugs

two_src = torch.isin(
    cold_test_two_sided[0],
    cold_drug_ids
)

two_dst = torch.isin(
    cold_test_two_sided[1],
    cold_drug_ids
)

assert torch.all(
    two_src & two_dst
)

print(
    "PASS: every secondary cold test pair "
    "contains two cold drugs."
)


# Partition must reconstruct the original test set

reconstructed_test_count = (
    warm_test_pairs.shape[1]
    + cold_test_one_sided.shape[1]
    + cold_test_two_sided.shape[1]
)

assert (
    reconstructed_test_count
    == test_pairs.shape[1]
)

print(
    "PASS: test partitions reconstruct "
    "the complete original test set."
)


# Training partition must reconstruct original train set

assert (
    warm_train_pairs.shape[1]
    + cold_train_pairs.shape[1]
    == train_pairs.shape[1]
)

print(
    "PASS: training partitions reconstruct "
    "the complete original training set."
)


print("\nAll Step 10 assertions passed.")


## 4. Cold-Start Message-Passing Graphs

Construct G0-cold and G3-cold graphs.

G0 contains only warm-training DDI edges. G3 contains the same warm-training DDI edges while retaining the original non-DDI biomedical context.


In [ ]:
# ============================================================
# STEP 11 — Build cold-start G0/G3 message-passing graphs
#            IN MEMORY ONLY
# ============================================================

import torch

# ------------------------------------------------------------
# Load original graphs
# ------------------------------------------------------------

g0 = torch.load(
    TENSOR_DIR / "G0.pt",
    map_location="cpu"
)

g3 = torch.load(
    TENSOR_DIR / "G3.pt",
    map_location="cpu"
)


# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------

def build_cold_graph(graph, cold_ids):
    """
    Remove every relation-0 (DDI) edge touching a cold-start
    drug while preserving all non-DDI biomedical edges.
    """

    edge_index = graph["edge_index"]
    edge_type = graph["edge_type"]

    is_ddi = edge_type == 0

    src_cold = torch.isin(
        edge_index[0],
        cold_ids
    )

    dst_cold = torch.isin(
        edge_index[1],
        cold_ids
    )

    touches_cold = src_cold | dst_cold

    # Remove ONLY DDI edges touching cold drugs.
    remove_mask = is_ddi & touches_cold

    keep_mask = ~remove_mask

    cold_graph = {
        "edge_index": edge_index[:, keep_mask].clone(),
        "edge_type": edge_type[keep_mask].clone(),
        "num_nodes": graph["num_nodes"],
        "num_relations": graph["num_relations"],
    }

    return cold_graph, remove_mask


# ------------------------------------------------------------
# Construct graphs
# ------------------------------------------------------------

g0_cold, g0_removed_mask = build_cold_graph(
    g0,
    cold_drug_ids
)

g3_cold, g3_removed_mask = build_cold_graph(
    g3,
    cold_drug_ids
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("COLD-START MESSAGE-PASSING GRAPHS")
print("=" * 70)

for name, original, cold, removed in [
    ("G0", g0, g0_cold, g0_removed_mask),
    ("G3", g3, g3_cold, g3_removed_mask),
]:

    print(f"\n{name}")
    print("-" * 50)

    print(
        "Original directed edges:",
        f"{original['edge_index'].shape[1]:,}"
    )

    print(
        "Removed directed edges:",
        f"{int(removed.sum()):,}"
    )

    print(
        "Remaining directed edges:",
        f"{cold['edge_index'].shape[1]:,}"
    )

    print(
        "Remaining DDI edges:",
        f"{int((cold['edge_type'] == 0).sum()):,}"
    )

    print(
        "Remaining biomedical edges:",
        f"{int((cold['edge_type'] != 0).sum()):,}"
    )

In [ ]:
# ============================================================
# STEP 11B — Mandatory graph leakage checks
# ============================================================

EXPECTED_WARM_DIRECTED_DDI = (
    warm_train_pairs.shape[1] * 2
)

print("GRAPH INTEGRITY / LEAKAGE CHECKS")
print("=" * 70)

print(
    "Expected directed warm DDI edges:",
    f"{EXPECTED_WARM_DIRECTED_DDI:,}"
)


for name, graph in [
    ("G0-cold", g0_cold),
    ("G3-cold", g3_cold),
]:

    edge_index = graph["edge_index"]
    edge_type = graph["edge_type"]

    ddi_mask = edge_type == 0

    ddi_edges = edge_index[:, ddi_mask]

    cold_in_ddi = torch.isin(
        ddi_edges.reshape(-1),
        cold_drug_ids
    ).any()

    print(f"\n{name}")

    print(
        "Directed DDI edges:",
        f"{ddi_edges.shape[1]:,}"
    )

    print(
        "Cold drug appears in DDI edges:",
        bool(cold_in_ddi)
    )

    assert (
        ddi_edges.shape[1]
        == EXPECTED_WARM_DIRECTED_DDI
    )

    assert not bool(cold_in_ddi)

    print(
        "PASS: DDI graph contains only "
        "warm-training DDI edges."
    )


# ------------------------------------------------------------
# G3 biomedical context must remain unchanged
# ------------------------------------------------------------

original_g3_bio = int(
    (g3["edge_type"] != 0).sum()
)

cold_g3_bio = int(
    (g3_cold["edge_type"] != 0).sum()
)

print("\nG3 BIOMEDICAL CONTEXT")
print("-" * 50)

print(
    "Original biomedical directed edges:",
    f"{original_g3_bio:,}"
)

print(
    "Cold G3 biomedical directed edges:",
    f"{cold_g3_bio:,}"
)

assert original_g3_bio == cold_g3_bio

print(
    "PASS: all G3 biomedical edges were preserved."
)


# ------------------------------------------------------------
# Verify selected cold drugs still have biomedical context
# in G3-cold
# ------------------------------------------------------------

g3_bio_mask = g3_cold["edge_type"] != 0

g3_bio_edges = (
    g3_cold["edge_index"][:, g3_bio_mask]
)

cold_bio_counts = []

for node in cold_drug_ids.tolist():

    count = int(
        (
            (g3_bio_edges[0] == node)
            |
            (g3_bio_edges[1] == node)
        ).sum()
    )

    cold_bio_counts.append(count)


print("\nCOLD-DRUG BIOMEDICAL CONTEXT IN G3-COLD")
print("-" * 50)

print(
    "Cold drugs:",
    len(cold_bio_counts)
)

print(
    "Minimum biomedical degree:",
    min(cold_bio_counts)
)

print(
    "Median biomedical degree:",
    float(torch.tensor(
        cold_bio_counts,
        dtype=torch.float
    ).median())
)

print(
    "Maximum biomedical degree:",
    max(cold_bio_counts)
)

print(
    "Cold drugs with zero biomedical context:",
    sum(x == 0 for x in cold_bio_counts)
)

assert all(
    x >= MIN_BIO_DEGREE
    for x in cold_bio_counts
)

print(
    "PASS: every selected cold drug retains "
    "the required biomedical context."
)

print("\nAll Step 11 assertions passed.")

## 5. Frozen Experiment Artifacts

Save the selected cold cohort, cold-start graphs, split partitions, and experiment manifest. Reload checks are used to verify the frozen artifacts.


In [ ]:
# ============================================================
# STEP 12 — Save reproducible cold-start experiment artifacts
# ============================================================

from pathlib import Path
import json
import torch
import pandas as pd


# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

COLD_DIR = (
    BASE
    / "data"
    / "processed"
    / "cold_start"
    / f"split_seed_{SPLIT_SEED}"
)

COLD_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Saving cold-start experiment to:")
print(COLD_DIR)


# ------------------------------------------------------------
# Save message-passing graphs
# ------------------------------------------------------------

torch.save(
    g0_cold,
    COLD_DIR / "G0_cold.pt"
)

torch.save(
    g3_cold,
    COLD_DIR / "G3_cold.pt"
)


# ------------------------------------------------------------
# Save training / validation pairs
# ------------------------------------------------------------

torch.save(
    {"pair_index": warm_train_pairs},
    COLD_DIR / "warm_train.pt"
)

torch.save(
    {"pair_index": warm_val_pairs},
    COLD_DIR / "warm_val.pt"
)


# ------------------------------------------------------------
# Save test partitions
# ------------------------------------------------------------

torch.save(
    {"pair_index": cold_test_one_sided},
    COLD_DIR / "cold_test_one_sided.pt"
)

torch.save(
    {"pair_index": cold_test_two_sided},
    COLD_DIR / "cold_test_two_sided.pt"
)

torch.save(
    {"pair_index": warm_test_pairs},
    COLD_DIR / "warm_test.pt"
)


# ------------------------------------------------------------
# Save selected cold drug IDs
# ------------------------------------------------------------

torch.save(
    {"cold_drug_ids": cold_drug_ids},
    COLD_DIR / "cold_drug_ids.pt"
)


# ------------------------------------------------------------
# Save human-readable cold-drug table
# ------------------------------------------------------------

cold_df.to_csv(
    COLD_DIR / "cold_drugs.csv",
    index=False
)


# ------------------------------------------------------------
# Experiment manifest
# ------------------------------------------------------------

manifest = {

    "experiment": "ddi_edge_cold_start",

    "split_seed": int(SPLIT_SEED),

    "cold_fraction_requested": float(
        COLD_FRACTION
    ),

    "num_candidate_drugs": int(
        len(drug_node_ids)
    ),

    "num_eligible_drugs": int(
        len(eligible_df)
    ),

    "num_cold_drugs": int(
        len(cold_df)
    ),

    "eligibility": {
        "min_total_ddi_degree":
            int(MIN_TOTAL_DDI_DEGREE),

        "min_biomedical_degree":
            int(MIN_BIO_DEGREE),

        "min_test_ddi_degree":
            int(MIN_TEST_DEGREE),
    },

    "ddi_pairs": {

        "original_train":
            int(train_pairs.shape[1]),

        "warm_train":
            int(warm_train_pairs.shape[1]),

        "removed_train_touching_cold":
            int(cold_train_pairs.shape[1]),

        "warm_validation":
            int(warm_val_pairs.shape[1]),

        "cold_validation":
            int(cold_val_pairs.shape[1]),

        "warm_test":
            int(warm_test_pairs.shape[1]),

        "cold_test_one_sided":
            int(cold_test_one_sided.shape[1]),

        "cold_test_two_sided":
            int(cold_test_two_sided.shape[1]),
    },

    "message_passing": {

        "g0_directed_edges":
            int(
                g0_cold[
                    "edge_index"
                ].shape[1]
            ),

        "g3_directed_edges":
            int(
                g3_cold[
                    "edge_index"
                ].shape[1]
            ),

        "directed_warm_ddi_edges":
            int(EXPECTED_WARM_DIRECTED_DDI),

        "g3_biomedical_directed_edges":
            int(
                (
                    g3_cold["edge_type"] != 0
                ).sum()
            ),
    },

    "protocol": {

        "primary_evaluation":
            "one-sided cold test DDIs",

        "secondary_evaluation":
            "two-sided cold test DDIs",

        "early_stopping":
            "warm-only validation DDIs",

        "cold_drug_training_ddi_edges":
            "all removed",

        "cold_drug_biomedical_edges":
            "preserved in G3",

        "claim_scope":
            "DDI-edge cold-start; not fully unseen-node inductive evaluation",
    },
}


with open(
    COLD_DIR / "manifest.json",
    "w"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


# ------------------------------------------------------------
# Report files
# ------------------------------------------------------------

print("\nSAVED FILES")
print("=" * 70)

for path in sorted(COLD_DIR.iterdir()):

    size_mb = (
        path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"{path.name:35s}"
        f"{size_mb:10.2f} MB"
    )
    

In [ ]:
# ============================================================
# STEP 12B — Reload saved artifacts and verify
# ============================================================

print("RELOAD VERIFICATION")
print("=" * 70)


saved_g0 = torch.load(
    COLD_DIR / "G0_cold.pt",
    map_location="cpu"
)

saved_g3 = torch.load(
    COLD_DIR / "G3_cold.pt",
    map_location="cpu"
)

saved_train = torch.load(
    COLD_DIR / "warm_train.pt",
    map_location="cpu"
)["pair_index"]

saved_val = torch.load(
    COLD_DIR / "warm_val.pt",
    map_location="cpu"
)["pair_index"]

saved_test_one = torch.load(
    COLD_DIR / "cold_test_one_sided.pt",
    map_location="cpu"
)["pair_index"]

saved_test_two = torch.load(
    COLD_DIR / "cold_test_two_sided.pt",
    map_location="cpu"
)["pair_index"]

saved_cold_ids = torch.load(
    COLD_DIR / "cold_drug_ids.pt",
    map_location="cpu"
)["cold_drug_ids"]


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert torch.equal(
    saved_cold_ids,
    cold_drug_ids
)

assert torch.equal(
    saved_train,
    warm_train_pairs
)

assert torch.equal(
    saved_val,
    warm_val_pairs
)

assert torch.equal(
    saved_test_one,
    cold_test_one_sided
)

assert torch.equal(
    saved_test_two,
    cold_test_two_sided
)


assert (
    saved_g0["edge_index"].shape[1]
    == 1_855_564
)

assert (
    int(
        (saved_g0["edge_type"] == 0).sum()
    )
    == 1_855_564
)


assert (
    int(
        (saved_g3["edge_type"] == 0).sum()
    )
    == 1_855_564
)

assert (
    int(
        (saved_g3["edge_type"] != 0).sum()
    )
    == 136_568
)


# ------------------------------------------------------------
# Final leakage verification after disk reload
# ------------------------------------------------------------

for name, graph in [
    ("G0-cold", saved_g0),
    ("G3-cold", saved_g3),
]:

    ddi_edges = graph["edge_index"][
        :,
        graph["edge_type"] == 0
    ]

    leakage = torch.isin(
        ddi_edges.reshape(-1),
        saved_cold_ids
    ).any()

    assert not bool(leakage)

    print(
        f"PASS: {name} contains no "
        f"cold-drug DDI message-passing edges."
    )


print(
    "\nPASS: saved artifacts exactly match "
    "the verified in-memory experiment."
)

print("\nCold-start dataset is ready.")


In [ ]:
# ============================================================
# STEP 13 — Locate existing R-GCN training code
# ============================================================

from pathlib import Path

BASE = Path("/workspace/primekg_ddi_rgcn")
NOTEBOOK_DIR = BASE / "notebooks"

print("NOTEBOOKS")
print("=" * 70)

for path in sorted(NOTEBOOK_DIR.glob("*.ipynb")):
    print(path.name)
    

In [ ]:
# ============================================================
# STEP 14 — Inspect original R-GCN training code
# ============================================================

import json
from pathlib import Path

NOTEBOOK_PATH = (
    Path("/workspace/primekg_ddi_rgcn")
    / "notebooks"
    / "04_train_rgcn.ipynb"
)

with open(
    NOTEBOOK_PATH,
    "r",
    encoding="utf-8"
) as f:
    nb = json.load(f)

print("NOTEBOOK:")
print(NOTEBOOK_PATH)
print()

for i, cell in enumerate(nb["cells"]):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell["source"])

    print("=" * 80)
    print(f"CODE CELL {i}")
    print("=" * 80)

    # Print only cells likely related to the model/training/evaluation
    keywords = [
        "RGCN",
        "RGCNConv",
        "train",
        "optimizer",
        "negative",
        "decoder",
        "score",
        "early",
        "patience",
        "epoch",
        "evaluate",
        "loss",
    ]

    if any(
        keyword.lower() in source.lower()
        for keyword in keywords
    ):
        print(source)
        print()

## 6. Leakage-Safe Negative Sampling

Construct warm-only DDI supervision negatives.

The complete known-positive DDI mask is used so removed cold-training DDIs, validation DDIs, and test DDIs cannot accidentally become sampled negatives.


In [ ]:
# ============================================================
# STEP 15 — Build leakage-safe warm-only negative pools
# ============================================================

import torch

# ------------------------------------------------------------
# Load full known-positive mask
# IMPORTANT: this includes train + val + test positives
# ------------------------------------------------------------

positive_data = torch.load(
    TENSOR_DIR / "ddi_known_positive_mask.pt",
    map_location="cpu"
)

known_positive_mask = positive_data[
    "known_positive_mask"
]

print("Known-positive mask shape:",
      tuple(known_positive_mask.shape))


# ------------------------------------------------------------
# Identify warm drugs
# ------------------------------------------------------------

is_cold_drug = torch.isin(
    drug_node_ids,
    cold_drug_ids
)

warm_drug_node_ids = drug_node_ids[
    ~is_cold_drug
]

print("All drugs:", len(drug_node_ids))
print("Cold drugs:", len(cold_drug_ids))
print("Warm drugs:", len(warm_drug_node_ids))

assert len(warm_drug_node_ids) == (
    len(drug_node_ids) - len(cold_drug_ids)
)


# ------------------------------------------------------------
# Convert warm global node IDs -> local drug indices
#
# known_positive_mask uses LOCAL drug indices
# from 0 ... 4277.
# ------------------------------------------------------------

warm_local_ids = torch.searchsorted(
    drug_node_ids,
    warm_drug_node_ids
)

assert torch.equal(
    drug_node_ids[warm_local_ids],
    warm_drug_node_ids
)


# ------------------------------------------------------------
# Generate every possible WARM-WARM undirected pair
# ------------------------------------------------------------

warm_candidate_positions = torch.triu_indices(
    len(warm_local_ids),
    len(warm_local_ids),
    offset=1
)

warm_candidate_local = torch.stack(
    [
        warm_local_ids[
            warm_candidate_positions[0]
        ],
        warm_local_ids[
            warm_candidate_positions[1]
        ]
    ],
    dim=0
)

print(
    "\nPossible warm-warm pairs:",
    f"{warm_candidate_local.shape[1]:,}"
)


# ------------------------------------------------------------
# Remove ALL known positive DDIs
#
# This uses the ORIGINAL complete known-positive mask.
# Therefore removed cold-start positives are still recognized
# as positives and can NEVER become negatives.
# ------------------------------------------------------------

is_known_positive = known_positive_mask[
    warm_candidate_local[0],
    warm_candidate_local[1]
]

warm_negative_pool_local = (
    warm_candidate_local[:, ~is_known_positive]
)

print(
    "Known positive warm-warm pairs:",
    f"{int(is_known_positive.sum()):,}"
)

print(
    "Available warm-warm negative pairs:",
    f"{warm_negative_pool_local.shape[1]:,}"
)


# ------------------------------------------------------------
# Sample fixed TRAIN negatives
#
# Keep the same 1:1 scale as the original training setup:
# one negative for each warm training positive.
# ------------------------------------------------------------

num_train_negatives = warm_train_pairs.shape[1]

train_neg_generator = torch.Generator()
train_neg_generator.manual_seed(SPLIT_SEED)

train_perm = torch.randperm(
    warm_negative_pool_local.shape[1],
    generator=train_neg_generator
)

train_selected = train_perm[
    :num_train_negatives
]

cold_train_negative_local = (
    warm_negative_pool_local[:, train_selected]
)


# ------------------------------------------------------------
# Remove those negatives before selecting validation negatives
# ------------------------------------------------------------

remaining_selected = train_perm[
    num_train_negatives:
]

remaining_negative_pool_local = (
    warm_negative_pool_local[
        :,
        remaining_selected
    ]
)


# ------------------------------------------------------------
# Sample fixed warm VALIDATION negatives
# ------------------------------------------------------------

num_val_negatives = warm_val_pairs.shape[1]

assert (
    remaining_negative_pool_local.shape[1]
    >= num_val_negatives
)

cold_val_negative_local = (
    remaining_negative_pool_local[
        :,
        :num_val_negatives
    ]
)


# ------------------------------------------------------------
# Convert local drug indices -> global/shared node IDs
# ------------------------------------------------------------

cold_train_negative_pairs = drug_node_ids[
    cold_train_negative_local
]

cold_val_negative_pairs = drug_node_ids[
    cold_val_negative_local
]


# ------------------------------------------------------------
# Basic report
# ------------------------------------------------------------

print("\nCOLD-START NEGATIVE DATA")
print("=" * 70)

print(
    "Train positives:",
    f"{warm_train_pairs.shape[1]:,}"
)

print(
    "Train negatives:",
    f"{cold_train_negative_pairs.shape[1]:,}"
)

print(
    "Validation positives:",
    f"{warm_val_pairs.shape[1]:,}"
)

print(
    "Validation negatives:",
    f"{cold_val_negative_pairs.shape[1]:,}"
)


In [ ]:
# ============================================================
# STEP 15B — Negative-sampling safety checks
# ============================================================

print("NEGATIVE-SAMPLING SAFETY CHECKS")
print("=" * 70)


def global_pairs_to_local(pair_index):
    a = torch.searchsorted(
        drug_node_ids,
        pair_index[0]
    )

    b = torch.searchsorted(
        drug_node_ids,
        pair_index[1]
    )

    assert torch.equal(
        drug_node_ids[a],
        pair_index[0]
    )

    assert torch.equal(
        drug_node_ids[b],
        pair_index[1]
    )

    return a, b


# ------------------------------------------------------------
# 1. No cold drugs in training negatives
# ------------------------------------------------------------

train_negative_has_cold = torch.isin(
    cold_train_negative_pairs.reshape(-1),
    cold_drug_ids
).any()

assert not bool(train_negative_has_cold)

print(
    "PASS: no cold drug occurs in training negatives."
)


# ------------------------------------------------------------
# 2. No cold drugs in validation negatives
# ------------------------------------------------------------

val_negative_has_cold = torch.isin(
    cold_val_negative_pairs.reshape(-1),
    cold_drug_ids
).any()

assert not bool(val_negative_has_cold)

print(
    "PASS: no cold drug occurs in validation negatives."
)


# ------------------------------------------------------------
# 3. Training negatives are not known positives
# ------------------------------------------------------------

train_a, train_b = global_pairs_to_local(
    cold_train_negative_pairs
)

train_positive_overlap = known_positive_mask[
    train_a,
    train_b
].sum().item()

assert train_positive_overlap == 0

print(
    "PASS: training negatives contain zero known DDIs."
)


# ------------------------------------------------------------
# 4. Validation negatives are not known positives
# ------------------------------------------------------------

val_a, val_b = global_pairs_to_local(
    cold_val_negative_pairs
)

val_positive_overlap = known_positive_mask[
    val_a,
    val_b
].sum().item()

assert val_positive_overlap == 0

print(
    "PASS: validation negatives contain zero known DDIs."
)


# ------------------------------------------------------------
# 5. No duplicate training negatives
# ------------------------------------------------------------

num_drugs = len(drug_node_ids)

train_keys = (
    train_a.to(torch.int64) * num_drugs
    + train_b.to(torch.int64)
)

assert (
    torch.unique(train_keys).numel()
    == train_keys.numel()
)

print(
    "PASS: training negatives contain no duplicates."
)


# ------------------------------------------------------------
# 6. No duplicate validation negatives
# ------------------------------------------------------------

val_keys = (
    val_a.to(torch.int64) * num_drugs
    + val_b.to(torch.int64)
)

assert (
    torch.unique(val_keys).numel()
    == val_keys.numel()
)

print(
    "PASS: validation negatives contain no duplicates."
)


# ------------------------------------------------------------
# 7. Train and validation negatives must be disjoint
# ------------------------------------------------------------

train_val_overlap = torch.isin(
    val_keys,
    train_keys
).sum().item()

assert train_val_overlap == 0

print(
    "PASS: train and validation negatives are disjoint."
)


# ------------------------------------------------------------
# 8. Positive/negative counts remain balanced
# ------------------------------------------------------------

assert (
    cold_train_negative_pairs.shape[1]
    == warm_train_pairs.shape[1]
)

assert (
    cold_val_negative_pairs.shape[1]
    == warm_val_pairs.shape[1]
)

print(
    "PASS: positive/negative counts are balanced."
)


print("\nAll Step 15 negative-sampling checks passed.")

In [ ]:
# ============================================================
# STEP 16 — Save verified cold-start negative sets
# ============================================================

TRAIN_NEG_PATH = (
    COLD_DIR / "warm_train_negatives.pt"
)

VAL_NEG_PATH = (
    COLD_DIR / "warm_val_negatives.pt"
)


torch.save(
    {
        "pair_index": cold_train_negative_pairs,
        "seed": SPLIT_SEED,
        "description": (
            "Warm-warm unobserved DDI pairs used as "
            "cold-start training negatives."
        )
    },
    TRAIN_NEG_PATH
)


torch.save(
    {
        "pair_index": cold_val_negative_pairs,
        "seed": SPLIT_SEED,
        "description": (
            "Warm-warm unobserved DDI pairs used as "
            "cold-start validation negatives."
        )
    },
    VAL_NEG_PATH
)


print("SAVED COLD-START NEGATIVES")
print("=" * 70)

print("Training:")
print(TRAIN_NEG_PATH)

print("\nValidation:")
print(VAL_NEG_PATH)

In [ ]:
# ============================================================
# STEP 16B — Reload and verify saved negatives
# ============================================================

saved_train_neg = torch.load(
    TRAIN_NEG_PATH,
    map_location="cpu"
)["pair_index"]

saved_val_neg = torch.load(
    VAL_NEG_PATH,
    map_location="cpu"
)["pair_index"]


# Exact equality with the verified in-memory tensors
assert torch.equal(
    saved_train_neg,
    cold_train_negative_pairs
)

assert torch.equal(
    saved_val_neg,
    cold_val_negative_pairs
)


# No cold drugs
assert not bool(
    torch.isin(
        saved_train_neg.reshape(-1),
        cold_drug_ids
    ).any()
)

assert not bool(
    torch.isin(
        saved_val_neg.reshape(-1),
        cold_drug_ids
    ).any()
)


# Check against COMPLETE known-positive mask again
train_a = torch.searchsorted(
    drug_node_ids,
    saved_train_neg[0]
)

train_b = torch.searchsorted(
    drug_node_ids,
    saved_train_neg[1]
)

val_a = torch.searchsorted(
    drug_node_ids,
    saved_val_neg[0]
)

val_b = torch.searchsorted(
    drug_node_ids,
    saved_val_neg[1]
)


assert (
    known_positive_mask[
        train_a,
        train_b
    ].sum().item()
    == 0
)

assert (
    known_positive_mask[
        val_a,
        val_b
    ].sum().item()
    == 0
)


print("RELOAD VERIFICATION")
print("=" * 70)

print(
    "Training negatives:",
    f"{saved_train_neg.shape[1]:,}"
)

print(
    "Validation negatives:",
    f"{saved_val_neg.shape[1]:,}"
)

print(
    "Cold drugs in training negatives:",
    False
)

print(
    "Cold drugs in validation negatives:",
    False
)

print(
    "Known positives in training negatives:",
    0
)

print(
    "Known positives in validation negatives:",
    0
)

print(
    "\nPASS: cold-start negative sets were "
    "saved and reloaded correctly."
)


## 7. Compute and Graph Integrity Checks

Verify the CUDA environment and perform graph-integrity checks before training.

Historical failed CUDA diagnostic cells have been omitted from this validated notebook; they remain preserved in the original experimental notebook.


In [ ]:
# ============================================================
# STEP 17 — Inspect GPU availability
# ============================================================

import subprocess

result = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=index,name,memory.total,memory.used,memory.free,utilization.gpu",
        "--format=csv,noheader,nounits"
    ],
    capture_output=True,
    text=True
)

print("GPU STATUS")
print("=" * 80)
print(
    "index | name | total MB | used MB | free MB | utilization %"
)
print("-" * 80)

for line in result.stdout.strip().split("\n"):
    print(line)

print("\nDo not start training yet.")


In [ ]:
# ============================================================
# STEP 18 — Verify GPU 1 from the current notebook kernel
# ============================================================

import torch

print("PYTORCH GPU CHECK")
print("=" * 70)

print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(
        f"cuda:{i} -> "
        f"{torch.cuda.get_device_name(i)}"
    )

# ------------------------------------------------------------
# Select GPU 1 explicitly
# ------------------------------------------------------------

TRAIN_GPU_INDEX = 1
device = torch.device(
    f"cuda:{TRAIN_GPU_INDEX}"
)

print("\nSelected training device:", device)
print(
    "Selected GPU:",
    torch.cuda.get_device_name(device)
)

# ------------------------------------------------------------
# Tiny allocation test
# ------------------------------------------------------------

test_tensor = torch.zeros(
    1024,
    device=device
)

print(
    "Test tensor device:",
    test_tensor.device
)

assert test_tensor.device.index == TRAIN_GPU_INDEX

del test_tensor

torch.cuda.empty_cache()

# ------------------------------------------------------------
# Memory report
# ------------------------------------------------------------

free_bytes, total_bytes = torch.cuda.mem_get_info(
    device
)

print(
    "\nFree GPU memory:",
    f"{free_bytes / 1024**3:.2f} GB"
)

print(
    "Total GPU memory:",
    f"{total_bytes / 1024**3:.2f} GB"
)

print(
    "\nPASS: GPU 1 is accessible from "
    "the current notebook kernel."
)


In [ ]:
# ============================================================
# STEP 19 — Exact cold-graph integrity verification
# ============================================================

import torch

print("EXACT COLD-GRAPH INTEGRITY CHECK")
print("=" * 70)


# ------------------------------------------------------------
# Reload frozen artifacts from disk
# ------------------------------------------------------------

check_g0 = torch.load(
    COLD_DIR / "G0_cold.pt",
    map_location="cpu"
)

check_g3 = torch.load(
    COLD_DIR / "G3_cold.pt",
    map_location="cpu"
)

check_warm_train = torch.load(
    COLD_DIR / "warm_train.pt",
    map_location="cpu"
)["pair_index"]


NUM_NODES = check_g0["num_nodes"]


# ------------------------------------------------------------
# Build EXACT expected directed DDI graph
#
# Each undirected warm training pair:
#
# A -- B
#
# must appear as:
#
# A -> B
# B -> A
# ------------------------------------------------------------

expected_directed_ddi = torch.cat(
    [
        check_warm_train,
        check_warm_train.flip(0)
    ],
    dim=1
)

print(
    "Warm undirected training pairs:",
    f"{check_warm_train.shape[1]:,}"
)

print(
    "Expected directed DDI edges:",
    f"{expected_directed_ddi.shape[1]:,}"
)


# ------------------------------------------------------------
# Encode edges as unique integer keys
# This lets us compare edge SETS independent of ordering.
# ------------------------------------------------------------

def directed_edge_keys(edge_index, num_nodes):

    return (
        edge_index[0].to(torch.int64)
        * num_nodes
        + edge_index[1].to(torch.int64)
    )


expected_keys = torch.sort(
    directed_edge_keys(
        expected_directed_ddi,
        NUM_NODES
    )
).values


# ------------------------------------------------------------
# Check G0 DDI edges
# ------------------------------------------------------------

g0_ddi_mask = (
    check_g0["edge_type"] == 0
)

g0_ddi_edges = check_g0[
    "edge_index"
][:, g0_ddi_mask]

g0_keys = torch.sort(
    directed_edge_keys(
        g0_ddi_edges,
        NUM_NODES
    )
).values


assert torch.equal(
    g0_keys,
    expected_keys
)

print(
    "\nPASS: G0-cold DDI edges exactly equal "
    "the directed warm training pairs."
)


# ------------------------------------------------------------
# G0 must contain ONLY DDI edges
# ------------------------------------------------------------

assert bool(
    torch.all(
        check_g0["edge_type"] == 0
    )
)

print(
    "PASS: G0-cold contains no biomedical edges."
)


# ------------------------------------------------------------
# Check G3 DDI edges
# ------------------------------------------------------------

g3_ddi_mask = (
    check_g3["edge_type"] == 0
)

g3_ddi_edges = check_g3[
    "edge_index"
][:, g3_ddi_mask]

g3_keys = torch.sort(
    directed_edge_keys(
        g3_ddi_edges,
        NUM_NODES
    )
).values


assert torch.equal(
    g3_keys,
    expected_keys
)

print(
    "PASS: G3-cold DDI edges exactly equal "
    "the directed warm training pairs."
)


# ------------------------------------------------------------
# Verify G3 biomedical graph is EXACTLY preserved
# ------------------------------------------------------------

original_g3 = torch.load(
    TENSOR_DIR / "G3.pt",
    map_location="cpu"
)

original_bio_mask = (
    original_g3["edge_type"] != 0
)

cold_bio_mask = (
    check_g3["edge_type"] != 0
)


original_bio_edges = original_g3[
    "edge_index"
][:, original_bio_mask]

original_bio_types = original_g3[
    "edge_type"
][original_bio_mask]


cold_bio_edges = check_g3[
    "edge_index"
][:, cold_bio_mask]

cold_bio_types = check_g3[
    "edge_type"
][cold_bio_mask]


assert torch.equal(
    original_bio_edges,
    cold_bio_edges
)

assert torch.equal(
    original_bio_types,
    cold_bio_types
)

print(
    "PASS: G3 biomedical edge_index is "
    "exactly preserved."
)

print(
    "PASS: G3 biomedical edge_type is "
    "exactly preserved."
)


# ------------------------------------------------------------
# Duplicate check
# ------------------------------------------------------------

assert (
    torch.unique(expected_keys).numel()
    == expected_keys.numel()
)

assert (
    torch.unique(g0_keys).numel()
    == g0_keys.numel()
)

assert (
    torch.unique(g3_keys).numel()
    == g3_keys.numel()
)

print(
    "PASS: directed DDI graph contains "
    "no duplicate edges."
)


# ------------------------------------------------------------
# Final counts
# ------------------------------------------------------------

print("\nFINAL VERIFIED COUNTS")
print("-" * 70)

print(
    "Warm train pairs:",
    f"{check_warm_train.shape[1]:,}"
)

print(
    "G0 directed DDI:",
    f"{g0_ddi_edges.shape[1]:,}"
)

print(
    "G3 directed DDI:",
    f"{g3_ddi_edges.shape[1]:,}"
)

print(
    "G3 biomedical:",
    f"{cold_bio_edges.shape[1]:,}"
)

print(
    "G3 total:",
    f"{check_g3['edge_index'].shape[1]:,}"
)

print(
    "\nPASS: exact cold-start graph integrity "
    "verification completed."
)


In [ ]:
# ============================================================
# STEP 20 — Initialize G0-cold R-GCN, model seed 42
# ============================================================

import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv


# ------------------------------------------------------------
# Experiment configuration
# ------------------------------------------------------------

MODEL_SEED = 42

COLD_CONFIG = {
    "split_seed": 42,
    "model_seed": MODEL_SEED,

    "num_nodes": 13094,
    "num_relations": 15,

    "embedding_dim": 128,
    "hidden_dim": 128,
    "num_rgcn_layers": 2,
    "dropout": 0.2,

    "learning_rate": 1e-3,
    "weight_decay": 1e-5,

    "max_epochs": 500,
    "early_stopping_patience": 10,

    "train_positives_per_epoch": 100_000,
    "negative_ratio": 1,

    "device": str(device),

    "protocol": "DDI-edge cold-start",
}


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

def reset_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


reset_seed(MODEL_SEED)


# ------------------------------------------------------------
# Same model architecture as original experiment
# ------------------------------------------------------------

class RGCNDDIModel(nn.Module):

    def __init__(
        self,
        num_nodes,
        num_relations,
        embedding_dim=128,
        hidden_dim=128,
        dropout=0.2
    ):
        super().__init__()

        self.node_embedding = nn.Embedding(
            num_nodes,
            embedding_dim
        )

        self.conv1 = RGCNConv(
            embedding_dim,
            hidden_dim,
            num_relations
        )

        self.conv2 = RGCNConv(
            hidden_dim,
            hidden_dim,
            num_relations
        )

        self.dropout = dropout

        # Symmetric DistMult-style DDI decoder
        self.ddi_relation = nn.Parameter(
            torch.empty(hidden_dim)
        )

        self.reset_parameters()

    def reset_parameters(self):

        nn.init.xavier_uniform_(
            self.node_embedding.weight
        )

        self.conv1.reset_parameters()
        self.conv2.reset_parameters()

        nn.init.ones_(
            self.ddi_relation
        )

    def encode(
        self,
        edge_index,
        edge_type
    ):

        x = self.node_embedding.weight

        x = self.conv1(
            x,
            edge_index,
            edge_type
        )

        x = F.relu(x)

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = self.conv2(
            x,
            edge_index,
            edge_type
        )

        return x

    def decode(
        self,
        z,
        pair_index
    ):

        src = pair_index[0]
        dst = pair_index[1]

        return (
            z[src]
            * self.ddi_relation
            * z[dst]
        ).sum(dim=-1)


# ------------------------------------------------------------
# Fresh G0-cold model
# ------------------------------------------------------------

reset_seed(MODEL_SEED)

cold_model = RGCNDDIModel(
    num_nodes=COLD_CONFIG["num_nodes"],
    num_relations=COLD_CONFIG["num_relations"],
    embedding_dim=COLD_CONFIG["embedding_dim"],
    hidden_dim=COLD_CONFIG["hidden_dim"],
    dropout=COLD_CONFIG["dropout"]
).to(device)


total_params = sum(
    p.numel()
    for p in cold_model.parameters()
)


print("G0-COLD MODEL")
print("=" * 70)

print(cold_model)

print(
    "\nTotal parameters:",
    f"{total_params:,}"
)

print(
    "Model seed:",
    MODEL_SEED
)

print(
    "Split seed:",
    COLD_CONFIG["split_seed"]
)

print(
    "Device:",
    next(cold_model.parameters()).device
)

print(
    "Decoder shape:",
    tuple(cold_model.ddi_relation.shape)
)

print(
    "\nPASS: fresh G0-cold model initialized."
)

## 8. Fresh CUDA Session and Frozen-Artifact Reload

Select the intended GPU before importing PyTorch in the fresh kernel and reload the frozen cold-start experiment artifacts.


In [ ]:
# ============================================================
# STEP 23 — Select physical GPU 1 BEFORE importing torch
# ============================================================

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

print(
    "CUDA_VISIBLE_DEVICES =",
    os.environ["CUDA_VISIBLE_DEVICES"]
)

print(
    "\nPhysical GPU 1 will appear to PyTorch as cuda:0."
)


In [ ]:
# ============================================================
# STEP 24 — Verify fresh CUDA context
# ============================================================

import torch

print("FRESH CUDA CHECK")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())

assert torch.cuda.is_available()
assert torch.cuda.device_count() == 1


device = torch.device("cuda:0")

print(
    "Visible GPU:",
    torch.cuda.get_device_name(0)
)


# ------------------------------------------------------------
# Basic CUDA operation
# ------------------------------------------------------------

a = torch.ones(
    (10, 10),
    device=device
)

b = a + 1

assert b[0, 0].item() == 2.0

print(
    "Basic CUDA operation: PASS"
)


# ------------------------------------------------------------
# cuBLAS matrix multiplication
# ------------------------------------------------------------

x = torch.randn(
    (128, 128),
    device=device
)

y = torch.randn(
    (128, 128),
    device=device
)

z = x @ y

torch.cuda.synchronize()

assert bool(
    torch.isfinite(z).all()
)

print(
    "cuBLAS matrix multiplication: PASS"
)


# ------------------------------------------------------------
# Memory
# ------------------------------------------------------------

free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)

print(
    "\nFree memory:",
    f"{free_bytes / 1024**3:.2f} GB"
)

print(
    "Total memory:",
    f"{total_bytes / 1024**3:.2f} GB"
)


del a, b, x, y, z

torch.cuda.empty_cache()


print(
    "\nPASS: fresh CUDA context is healthy."
)

In [ ]:
# ============================================================
# STEP 25 — Reload frozen cold-start experiment
# ============================================================

from pathlib import Path
import torch

PROJECT_DIR = Path("/workspace/primekg_ddi_rgcn")

TENSOR_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "rgcn_tensors"
)

COLD_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "cold_start"
    / "split_seed_42"
)

device = torch.device("cuda:0")


# ------------------------------------------------------------
# Frozen graphs
# ------------------------------------------------------------

g0_cold = torch.load(
    COLD_DIR / "G0_cold.pt",
    map_location="cpu"
)

g3_cold = torch.load(
    COLD_DIR / "G3_cold.pt",
    map_location="cpu"
)


# ------------------------------------------------------------
# Frozen positive partitions
# ------------------------------------------------------------

warm_train_pairs = torch.load(
    COLD_DIR / "warm_train.pt",
    map_location="cpu"
)["pair_index"]

warm_val_pairs = torch.load(
    COLD_DIR / "warm_val.pt",
    map_location="cpu"
)["pair_index"]

warm_test_pairs = torch.load(
    COLD_DIR / "warm_test.pt",
    map_location="cpu"
)["pair_index"]

cold_test_one_sided = torch.load(
    COLD_DIR / "cold_test_one_sided.pt",
    map_location="cpu"
)["pair_index"]

cold_test_two_sided = torch.load(
    COLD_DIR / "cold_test_two_sided.pt",
    map_location="cpu"
)["pair_index"]


# ------------------------------------------------------------
# Frozen negatives
# ------------------------------------------------------------

warm_train_negatives = torch.load(
    COLD_DIR / "warm_train_negatives.pt",
    map_location="cpu"
)["pair_index"]

warm_val_negatives = torch.load(
    COLD_DIR / "warm_val_negatives.pt",
    map_location="cpu"
)["pair_index"]


# ------------------------------------------------------------
# Cold drug IDs
# ------------------------------------------------------------

cold_drug_ids = torch.load(
    COLD_DIR / "cold_drug_ids.pt",
    map_location="cpu"
)

# Handle either saved tensor or {"drug_node_ids": tensor}
if isinstance(cold_drug_ids, dict):
    if "drug_node_ids" in cold_drug_ids:
        cold_drug_ids = cold_drug_ids["drug_node_ids"]
    elif "cold_drug_ids" in cold_drug_ids:
        cold_drug_ids = cold_drug_ids["cold_drug_ids"]
    else:
        raise KeyError(
            f"Unknown cold_drug_ids keys: "
            f"{list(cold_drug_ids.keys())}"
        )


# ------------------------------------------------------------
# Basic verification
# ------------------------------------------------------------

assert warm_train_pairs.shape[1] == 927_782
assert warm_val_pairs.shape[1] == 115_889
assert warm_test_pairs.shape[1] == 115_787

assert cold_test_one_sided.shape[1] == 17_176
assert cold_test_two_sided.shape[1] == 651

assert warm_train_negatives.shape[1] == 927_782
assert warm_val_negatives.shape[1] == 115_889

assert cold_drug_ids.numel() == 196

assert g0_cold["edge_index"].shape[1] == 1_855_564
assert g3_cold["edge_index"].shape[1] == 1_992_132


print("FROZEN COLD-START EXPERIMENT RELOADED")
print("=" * 70)

print("Device:", device)

print("\nCold drugs:")
print(f"  {cold_drug_ids.numel():,}")

print("\nTraining:")
print(
    f"  positives = {warm_train_pairs.shape[1]:,}"
)
print(
    f"  negatives = {warm_train_negatives.shape[1]:,}"
)

print("\nValidation:")
print(
    f"  positives = {warm_val_pairs.shape[1]:,}"
)
print(
    f"  negatives = {warm_val_negatives.shape[1]:,}"
)

print("\nTest:")
print(
    f"  warm      = {warm_test_pairs.shape[1]:,}"
)
print(
    f"  one-sided = {cold_test_one_sided.shape[1]:,}"
)
print(
    f"  two-sided = {cold_test_two_sided.shape[1]:,}"
)

print("\nGraphs:")
print(
    f"  G0-cold = "
    f"{g0_cold['edge_index'].shape[1]:,} edges"
)
print(
    f"  G3-cold = "
    f"{g3_cold['edge_index'].shape[1]:,} edges"
)

print(
    "\nPASS: frozen experiment successfully reloaded."
)


## 9. R-GCN Model and Training Protocol

Define the R-GCN DDI model and training procedure.

Training uses warm-only DDI supervision and warm-only validation for early stopping. Cold drugs receive no positive or negative DDI supervision. In G3 they can still participate in biomedical message passing.


In [ ]:
# ============================================================
# STEP 26 — Fresh G0-cold R-GCN forward smoke test
# ============================================================

import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

MODEL_SEED = 42

NUM_NODES = 13_094
NUM_RELATIONS = 15

EMBEDDING_DIM = 128
HIDDEN_DIM = 128
DROPOUT = 0.2


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

def reset_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


reset_seed(MODEL_SEED)


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

class RGCNDDIModel(nn.Module):

    def __init__(
        self,
        num_nodes,
        num_relations,
        embedding_dim=128,
        hidden_dim=128,
        dropout=0.2
    ):
        super().__init__()

        self.node_embedding = nn.Embedding(
            num_nodes,
            embedding_dim
        )

        self.conv1 = RGCNConv(
            embedding_dim,
            hidden_dim,
            num_relations
        )

        self.conv2 = RGCNConv(
            hidden_dim,
            hidden_dim,
            num_relations
        )

        self.dropout = dropout

        self.ddi_relation = nn.Parameter(
            torch.empty(hidden_dim)
        )

        self.reset_parameters()

    def reset_parameters(self):

        nn.init.xavier_uniform_(
            self.node_embedding.weight
        )

        self.conv1.reset_parameters()
        self.conv2.reset_parameters()

        nn.init.ones_(
            self.ddi_relation
        )

    def encode(
        self,
        edge_index,
        edge_type
    ):

        x = self.node_embedding.weight

        x = self.conv1(
            x,
            edge_index,
            edge_type
        )

        x = F.relu(x)

        x = F.dropout(
            x,
            p=self.dropout,
            training=self.training
        )

        x = self.conv2(
            x,
            edge_index,
            edge_type
        )

        return x

    def decode(
        self,
        z,
        pair_index
    ):

        src = pair_index[0]
        dst = pair_index[1]

        return (
            z[src]
            * self.ddi_relation
            * z[dst]
        ).sum(dim=-1)


# ------------------------------------------------------------
# Create fresh model
# ------------------------------------------------------------

model = RGCNDDIModel(
    num_nodes=NUM_NODES,
    num_relations=NUM_RELATIONS,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT
).to(device)


total_params = sum(
    p.numel()
    for p in model.parameters()
)


print("MODEL INITIALIZATION")
print("=" * 70)

print("Model seed:", MODEL_SEED)
print("Device:", next(model.parameters()).device)
print("Parameters:", f"{total_params:,}")

assert total_params == 2_200_704

print("PASS: model initialized correctly.")


# ------------------------------------------------------------
# Move G0-cold graph to GPU
# ------------------------------------------------------------

edge_index = (
    g0_cold["edge_index"]
    .to(device)
)

edge_type = (
    g0_cold["edge_type"]
    .to(device)
)


print("\nGRAPH")
print("=" * 70)

print(
    "Directed edges:",
    f"{edge_index.shape[1]:,}"
)

print(
    "edge_index device:",
    edge_index.device
)

print(
    "edge_type device:",
    edge_type.device
)


# ------------------------------------------------------------
# Tiny supervised sample
# ------------------------------------------------------------

SMOKE_SIZE = 1024

positive_sample = (
    warm_train_pairs[:, :SMOKE_SIZE]
)

negative_sample = (
    warm_train_negatives[:, :SMOKE_SIZE]
)

smoke_pairs = torch.cat(
    [
        positive_sample,
        negative_sample
    ],
    dim=1
).to(device)


# ------------------------------------------------------------
# Forward pass
# ------------------------------------------------------------

print("\nFORWARD PASS")
print("=" * 70)

model.eval()

with torch.no_grad():

    node_z = model.encode(
        edge_index,
        edge_type
    )

    logits = model.decode(
        node_z,
        smoke_pairs
    )

torch.cuda.synchronize()


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

assert node_z.shape == (
    NUM_NODES,
    HIDDEN_DIM
)

assert logits.shape == (
    SMOKE_SIZE * 2,
)

assert bool(
    torch.isfinite(node_z).all()
)

assert bool(
    torch.isfinite(logits).all()
)


print(
    "Node representation shape:",
    tuple(node_z.shape)
)

print(
    "Logit shape:",
    tuple(logits.shape)
)

print(
    "Node representations finite:",
    bool(torch.isfinite(node_z).all())
)

print(
    "Logits finite:",
    bool(torch.isfinite(logits).all())
)


# ------------------------------------------------------------
# GPU memory
# ------------------------------------------------------------

print("\nGPU MEMORY")
print("=" * 70)

print(
    "Allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    "Reserved:",
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)

print(
    "Free:",
    f"{free_bytes / 1024**3:.2f} GB"
)


print(
    "\nPASS: G0-cold R-GCN forward "
    "smoke test completed."
)

In [ ]:
# ============================================================
# STEP 27 — G0-cold backward-pass smoke test
# ============================================================

import torch
import torch.nn.functional as F


print("G0-COLD BACKWARD SMOKE TEST")
print("=" * 70)


# ------------------------------------------------------------
# IMPORTANT:
# Return model to training mode so dropout behaves exactly
# as it will during real training.
# ------------------------------------------------------------

model.train()


# ------------------------------------------------------------
# Fresh optimizer — same hyperparameters as original experiment
# ------------------------------------------------------------

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)


# ------------------------------------------------------------
# Small balanced supervised batch
# ------------------------------------------------------------

SMOKE_SIZE = 1024

positive_pairs = (
    warm_train_pairs[:, :SMOKE_SIZE]
    .to(device)
)

negative_pairs = (
    warm_train_negatives[:, :SMOKE_SIZE]
    .to(device)
)

training_pairs = torch.cat(
    [
        positive_pairs,
        negative_pairs
    ],
    dim=1
)


labels = torch.cat(
    [
        torch.ones(
            SMOKE_SIZE,
            device=device
        ),
        torch.zeros(
            SMOKE_SIZE,
            device=device
        )
    ]
)


# ------------------------------------------------------------
# Verify no cold drug accidentally appears
# ------------------------------------------------------------

cold_gpu = cold_drug_ids.to(device)

assert not bool(
    torch.isin(
        training_pairs.reshape(-1),
        cold_gpu
    ).any()
)

print(
    "Cold drugs in supervised smoke batch:",
    False
)


# ------------------------------------------------------------
# Forward + loss
# ------------------------------------------------------------

optimizer.zero_grad(
    set_to_none=True
)

z_train = model.encode(
    edge_index,
    edge_type
)

logits_train = model.decode(
    z_train,
    training_pairs
)

loss = F.binary_cross_entropy_with_logits(
    logits_train,
    labels
)


assert bool(torch.isfinite(loss))

print(
    "Loss before backward:",
    float(loss.item())
)


# ------------------------------------------------------------
# Backward
# ------------------------------------------------------------

loss.backward()

torch.cuda.synchronize()


# ------------------------------------------------------------
# Gradient checks
# ------------------------------------------------------------

parameters_with_grad = 0
nonfinite_gradient_tensors = 0
total_grad_norm_sq = 0.0


for name, parameter in model.named_parameters():

    if parameter.grad is None:
        continue

    parameters_with_grad += 1

    grad = parameter.grad

    if not bool(torch.isfinite(grad).all()):
        nonfinite_gradient_tensors += 1

    total_grad_norm_sq += (
        grad.detach()
        .float()
        .norm(2)
        .item()
        ** 2
    )


total_grad_norm = (
    total_grad_norm_sq ** 0.5
)


assert parameters_with_grad > 0
assert nonfinite_gradient_tensors == 0
assert total_grad_norm > 0


print(
    "Parameter tensors with gradients:",
    parameters_with_grad
)

print(
    "Non-finite gradient tensors:",
    nonfinite_gradient_tensors
)

print(
    "Total gradient norm:",
    f"{total_grad_norm:.6f}"
)


# ------------------------------------------------------------
# Perform exactly ONE optimizer step
# ------------------------------------------------------------

optimizer.step()

torch.cuda.synchronize()


# ------------------------------------------------------------
# Check model parameters remain finite
# ------------------------------------------------------------

all_parameters_finite = all(
    bool(torch.isfinite(p).all())
    for p in model.parameters()
)

assert all_parameters_finite


print(
    "Parameters finite after optimizer step:",
    all_parameters_finite
)


# ------------------------------------------------------------
# GPU memory
# ------------------------------------------------------------

print("\nGPU MEMORY")
print("-" * 70)

print(
    "Allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    "Reserved:",
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)

print(
    "Free:",
    f"{free_bytes / 1024**3:.2f} GB"
)


print(
    "\nPASS: forward, BCE loss, backward, gradients, "
    "and optimizer step completed."
)

In [ ]:
# ============================================================
# STEP 28 — Create pristine G0-cold seed-42 training model
# ============================================================

MODEL_SEED = 42

reset_seed(MODEL_SEED)

model = RGCNDDIModel(
    num_nodes=NUM_NODES,
    num_relations=NUM_RELATIONS,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

# Confirm pristine setup
total_params = sum(
    p.numel()
    for p in model.parameters()
)

assert total_params == 2_200_704

print("REAL TRAINING MODEL READY")
print("=" * 70)
print("Experiment: G0-cold")
print("Split seed:", 42)
print("Model seed:", MODEL_SEED)
print("Device:", device)
print("Parameters:", f"{total_params:,}")
print("Learning rate:", 1e-3)
print("Weight decay:", 1e-5)
print("Max epochs:", 500)
print("Early stopping patience:", 10)
print("Training positives available:", f"{warm_train_pairs.shape[1]:,}")
print("Training negatives available:", f"{warm_train_negatives.shape[1]:,}")
print("Sampled positives/epoch:", "100,000")
print("Sampled negatives/epoch:", "100,000")

print(
    "\nPASS: pristine G0-cold seed-42 model "
    "is ready for real training."
)

In [ ]:
# ============================================================
# STEP 29 — Define real training + warm-only validation
# ============================================================

import copy
import time
import torch
import torch.nn.functional as F

TRAIN_SAMPLE_SIZE = 100_000
MAX_EPOCHS = 500
PATIENCE = 10


def train_one_epoch(
    model,
    optimizer,
    edge_index,
    edge_type,
    train_positive_pairs,
    train_negative_pairs,
    epoch,
    model_seed,
    device
):
    model.train()

    # Same paired sampling logic as the original experiment:
    # deterministic per model seed + epoch.
    generator = torch.Generator(device="cpu")
    generator.manual_seed(model_seed + epoch)

    pos_perm = torch.randperm(
        train_positive_pairs.shape[1],
        generator=generator
    )[:TRAIN_SAMPLE_SIZE]

    neg_perm = torch.randperm(
        train_negative_pairs.shape[1],
        generator=generator
    )[:TRAIN_SAMPLE_SIZE]

    pos_batch = train_positive_pairs[:, pos_perm].to(device)
    neg_batch = train_negative_pairs[:, neg_perm].to(device)

    pair_batch = torch.cat(
        [pos_batch, neg_batch],
        dim=1
    )

    labels = torch.cat(
        [
            torch.ones(
                TRAIN_SAMPLE_SIZE,
                device=device
            ),
            torch.zeros(
                TRAIN_SAMPLE_SIZE,
                device=device
            )
        ]
    )

    optimizer.zero_grad(set_to_none=True)

    z = model.encode(
        edge_index,
        edge_type
    )

    logits = model.decode(
        z,
        pair_batch
    )

    loss = F.binary_cross_entropy_with_logits(
        logits,
        labels
    )

    loss.backward()
    optimizer.step()

    return float(loss.item())


@torch.no_grad()
def validation_loss(
    model,
    edge_index,
    edge_type,
    val_positive_pairs,
    val_negative_pairs,
    device
):
    model.eval()

    z = model.encode(
        edge_index,
        edge_type
    )

    pos = val_positive_pairs.to(device)
    neg = val_negative_pairs.to(device)

    pairs = torch.cat(
        [pos, neg],
        dim=1
    )

    labels = torch.cat(
        [
            torch.ones(
                pos.shape[1],
                device=device
            ),
            torch.zeros(
                neg.shape[1],
                device=device
            )
        ]
    )

    logits = model.decode(
        z,
        pairs
    )

    loss = F.binary_cross_entropy_with_logits(
        logits,
        labels
    )

    return float(loss.item())


print("TRAINING FUNCTIONS READY")
print("=" * 70)
print("Training sample per epoch:", f"{TRAIN_SAMPLE_SIZE:,} positives")
print("Training negatives per epoch:", f"{TRAIN_SAMPLE_SIZE:,}")
print("Validation positives:", f"{warm_val_pairs.shape[1]:,}")
print("Validation negatives:", f"{warm_val_negatives.shape[1]:,}")
print("Validation protocol: warm-only")
print("\nPASS: real cold-start training functions defined.")

In [ ]:
# ============================================================
# STEP 29B — Run epoch 1 only
# ============================================================

print("G0-COLD — REAL TRAINING EPOCH 1")
print("=" * 70)

start_time = time.time()

train_loss_epoch1 = train_one_epoch(
    model=model,
    optimizer=optimizer,
    edge_index=edge_index,
    edge_type=edge_type,
    train_positive_pairs=warm_train_pairs,
    train_negative_pairs=warm_train_negatives,
    epoch=1,
    model_seed=MODEL_SEED,
    device=device
)

torch.cuda.synchronize()

after_train = time.time()

val_loss_epoch1 = validation_loss(
    model=model,
    edge_index=edge_index,
    edge_type=edge_type,
    val_positive_pairs=warm_val_pairs,
    val_negative_pairs=warm_val_negatives,
    device=device
)

torch.cuda.synchronize()

end_time = time.time()


assert torch.isfinite(
    torch.tensor(train_loss_epoch1)
)

assert torch.isfinite(
    torch.tensor(val_loss_epoch1)
)


print(
    "Training loss:",
    f"{train_loss_epoch1:.6f}"
)

print(
    "Warm validation loss:",
    f"{val_loss_epoch1:.6f}"
)

print(
    "Training time:",
    f"{after_train - start_time:.2f} sec"
)

print(
    "Validation time:",
    f"{end_time - after_train:.2f} sec"
)

print(
    "Total epoch time:",
    f"{end_time - start_time:.2f} sec"
)

print("\nGPU MEMORY")
print("-" * 70)

print(
    "Allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    "Reserved:",
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

free_bytes, total_bytes = torch.cuda.mem_get_info()

print(
    "Free:",
    f"{free_bytes / 1024**3:.2f} GB"
)

print(
    "\nPASS: real G0-cold epoch 1 completed."
)

In [ ]:
# ============================================================
# STEP 30 — Continue G0-cold seed-42 training
#            Epoch 2 -> max 500
# ============================================================

import copy
import time
import math

print("G0-COLD — FULL TRAINING")
print("=" * 70)

print("Continuing from completed epoch 1.")
print("Maximum epoch:", MAX_EPOCHS)
print("Early stopping patience:", PATIENCE)
print("Model seed:", MODEL_SEED)
print("Validation: warm-only")
print()


# ------------------------------------------------------------
# Epoch 1 is already completed.
# Initialize early-stopping state from its validation result.
# ------------------------------------------------------------

best_val_loss = val_loss_epoch1
best_epoch = 1

best_model_state = copy.deepcopy(
    model.state_dict()
)

patience_counter = 0


history = [
    {
        "epoch": 1,
        "train_loss": train_loss_epoch1,
        "val_loss": val_loss_epoch1
    }
]


training_start = time.time()


# ------------------------------------------------------------
# Continue from epoch 2
# ------------------------------------------------------------

for epoch in range(2, MAX_EPOCHS + 1):

    epoch_start = time.time()

    train_loss = train_one_epoch(
        model=model,
        optimizer=optimizer,
        edge_index=edge_index,
        edge_type=edge_type,
        train_positive_pairs=warm_train_pairs,
        train_negative_pairs=warm_train_negatives,
        epoch=epoch,
        model_seed=MODEL_SEED,
        device=device
    )

    val_loss = validation_loss(
        model=model,
        edge_index=edge_index,
        edge_type=edge_type,
        val_positive_pairs=warm_val_pairs,
        val_negative_pairs=warm_val_negatives,
        device=device
    )

    torch.cuda.synchronize()

    epoch_seconds = (
        time.time() - epoch_start
    )


    # --------------------------------------------------------
    # Safety checks
    # --------------------------------------------------------

    if not math.isfinite(train_loss):
        raise RuntimeError(
            f"Non-finite training loss "
            f"at epoch {epoch}: {train_loss}"
        )

    if not math.isfinite(val_loss):
        raise RuntimeError(
            f"Non-finite validation loss "
            f"at epoch {epoch}: {val_loss}"
        )


    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss
        }
    )


    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    improved = (
        val_loss < best_val_loss
    )

    if improved:

        best_val_loss = val_loss
        best_epoch = epoch

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        patience_counter = 0

    else:

        patience_counter += 1


    # --------------------------------------------------------
    # Progress output
    # --------------------------------------------------------

    if (
        epoch <= 10
        or epoch % 10 == 0
        or improved
    ):

        marker = "*" if improved else ""

        print(
            f"Epoch {epoch:03d} | "
            f"train={train_loss:.6f} | "
            f"val={val_loss:.6f} | "
            f"best={best_val_loss:.6f} "
            f"(epoch {best_epoch}) | "
            f"patience={patience_counter}/{PATIENCE} | "
            f"{epoch_seconds:.2f}s "
            f"{marker}"
        )


    # --------------------------------------------------------
    # Stop if validation has not improved
    # --------------------------------------------------------

    if patience_counter >= PATIENCE:

        print()
        print(
            f"EARLY STOPPING at epoch {epoch}."
        )

        break


# ------------------------------------------------------------
# Restore BEST model, not final model
# ------------------------------------------------------------

model.load_state_dict(
    best_model_state
)

model.eval()


training_seconds = (
    time.time() - training_start
)

last_epoch = history[-1]["epoch"]


print("\n" + "=" * 70)
print("G0-COLD TRAINING COMPLETE")
print("=" * 70)

print(
    "Last epoch executed:",
    last_epoch
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best warm validation loss:",
    f"{best_val_loss:.6f}"
)

print(
    "Total epochs represented:",
    len(history)
)

print(
    "Continuation training time:",
    f"{training_seconds:.2f} sec"
)

print(
    "Best model restored:",
    True
)


# ------------------------------------------------------------
# Final parameter safety check
# ------------------------------------------------------------

assert all(
    bool(torch.isfinite(p).all())
    for p in model.parameters()
)

print(
    "All restored parameters finite:",
    True
)


print("\nGPU MEMORY")
print("-" * 70)

print(
    "Allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    "Reserved:",
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)

print(
    "Free:",
    f"{free_bytes / 1024**3:.2f} GB"
)

print(
    "\nPASS: G0-cold seed-42 training "
    "completed and best model restored."
)

In [ ]:
# ============================================================
# STEP 31 — Save G0-cold seed-42 checkpoint + history
# ============================================================

from pathlib import Path
import json
import torch


# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------

COLD_CHECKPOINT_DIR = (
    PROJECT_DIR
    / "checkpoints"
    / "cold_start"
    / "split_seed_42"
    / "G0"
    / "model_seed_42"
)

COLD_RESULT_DIR = (
    PROJECT_DIR
    / "results"
    / "cold_start"
    / "split_seed_42"
    / "G0"
    / "model_seed_42"
)

COLD_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

COLD_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Checkpoint
# ------------------------------------------------------------

checkpoint_path = (
    COLD_CHECKPOINT_DIR
    / "best_model.pt"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),

        "experiment": "ddi_edge_cold_start",
        "graph_variant": "G0",

        "split_seed": 42,
        "model_seed": MODEL_SEED,

        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,

        "num_nodes": NUM_NODES,
        "num_relations": NUM_RELATIONS,

        "embedding_dim": EMBEDDING_DIM,
        "hidden_dim": HIDDEN_DIM,
        "dropout": DROPOUT,

        "learning_rate": 1e-3,
        "weight_decay": 1e-5,

        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,

        "train_sample_size": TRAIN_SAMPLE_SIZE,

        "validation_protocol": "warm-only",

        "cold_start_scope": (
            "DDI-edge cold-start; cold drugs receive "
            "no supervised DDI positives or negatives"
        ),
    },
    checkpoint_path
)


# ------------------------------------------------------------
# Training history
# ------------------------------------------------------------

history_path = (
    COLD_RESULT_DIR
    / "training_history.json"
)

with open(
    history_path,
    "w"
) as f:

    json.dump(
        history,
        f,
        indent=2
    )


# ------------------------------------------------------------
# Training summary
# ------------------------------------------------------------

summary = {
    "experiment": "ddi_edge_cold_start",
    "graph_variant": "G0",

    "split_seed": 42,
    "model_seed": MODEL_SEED,

    "best_epoch": int(best_epoch),
    "best_val_loss": float(best_val_loss),

    "last_epoch": int(last_epoch),

    "warm_train_positive_count":
        int(warm_train_pairs.shape[1]),

    "warm_train_negative_count":
        int(warm_train_negatives.shape[1]),

    "warm_validation_positive_count":
        int(warm_val_pairs.shape[1]),

    "warm_validation_negative_count":
        int(warm_val_negatives.shape[1]),

    "cold_drug_count":
        int(cold_drug_ids.numel()),

    "validation_protocol":
        "warm-only",

    "test_evaluated":
        False,
}


summary_path = (
    COLD_RESULT_DIR
    / "training_summary.json"
)

with open(
    summary_path,
    "w"
) as f:

    json.dump(
        summary,
        f,
        indent=2
    )


print("G0-COLD SEED-42 ARTIFACTS SAVED")
print("=" * 70)

print(
    "Checkpoint:"
)

print(
    checkpoint_path
)

print(
    "\nTraining history:"
)

print(
    history_path
)

print(
    "\nTraining summary:"
)

print(
    summary_path
)

In [ ]:
# ============================================================
# STEP 31B — Reload checkpoint verification
# ============================================================

saved_checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu"
)

assert (
    saved_checkpoint["graph_variant"]
    == "G0"
)

assert (
    saved_checkpoint["split_seed"]
    == 42
)

assert (
    saved_checkpoint["model_seed"]
    == 42
)

assert (
    saved_checkpoint["best_epoch"]
    == best_epoch
)

assert abs(
    saved_checkpoint["best_val_loss"]
    - best_val_loss
) < 1e-12


# ------------------------------------------------------------
# Compare every saved parameter against current best model
# ------------------------------------------------------------

current_state = {
    k: v.detach().cpu()
    for k, v
    in model.state_dict().items()
}

saved_state = (
    saved_checkpoint[
        "model_state_dict"
    ]
)

assert (
    current_state.keys()
    == saved_state.keys()
)

for name in current_state:

    assert torch.equal(
        current_state[name],
        saved_state[name]
    ), f"Mismatch: {name}"


# ------------------------------------------------------------
# Verify history
# ------------------------------------------------------------

with open(
    history_path,
    "r"
) as f:

    saved_history = json.load(f)

assert len(saved_history) == 500

assert saved_history[0]["epoch"] == 1
assert saved_history[-1]["epoch"] == 500


print("CHECKPOINT RELOAD VERIFICATION")
print("=" * 70)

print(
    "Graph:",
    saved_checkpoint["graph_variant"]
)

print(
    "Split seed:",
    saved_checkpoint["split_seed"]
)

print(
    "Model seed:",
    saved_checkpoint["model_seed"]
)

print(
    "Best epoch:",
    saved_checkpoint["best_epoch"]
)

print(
    "Best validation loss:",
    f"{saved_checkpoint['best_val_loss']:.6f}"
)

print(
    "History epochs:",
    len(saved_history)
)

print(
    "Exact parameter match:",
    True
)

print(
    "\nPASS: G0-cold seed-42 checkpoint "
    "and history are safely stored."
)


## 10. Filtered Ranking Protocol

Prepare the candidate-drug set and full known-positive mask.

The primary evaluation orients each one-sided test interaction as:

**cold drug → warm drug**

Filtered ranking removes known positive DDIs and the query drug itself from competing candidates, while restoring the true target before ranking.


In [ ]:
# ============================================================
# STEP 32 — Verify candidate drugs + complete positive mask
# ============================================================

import torch

drug_data = torch.load(
    TENSOR_DIR / "drug_node_ids.pt",
    map_location="cpu"
)

if isinstance(drug_data, dict):
    drug_node_ids = drug_data["drug_node_ids"]
else:
    drug_node_ids = drug_data


mask_data = torch.load(
    TENSOR_DIR / "ddi_known_positive_mask.pt",
    map_location="cpu"
)

if isinstance(mask_data, dict):
    known_positive_mask = mask_data["known_positive_mask"]
else:
    known_positive_mask = mask_data


print("FILTERED-RANKING DATA CHECK")
print("=" * 70)

print(
    "Candidate drugs:",
    f"{drug_node_ids.numel():,}"
)

print(
    "Known-positive mask shape:",
    tuple(known_positive_mask.shape)
)

print(
    "Mask dtype:",
    known_positive_mask.dtype
)


# ------------------------------------------------------------
# Basic structure
# ------------------------------------------------------------

assert drug_node_ids.numel() == 4_278

assert known_positive_mask.shape == (
    4_278,
    4_278
)

assert known_positive_mask.dtype == torch.bool

assert torch.all(
    drug_node_ids[:-1]
    < drug_node_ids[1:]
)

print(
    "PASS: candidate drug IDs are strictly sorted."
)


# ------------------------------------------------------------
# Mask properties
# ------------------------------------------------------------

assert torch.equal(
    known_positive_mask,
    known_positive_mask.T
)

print(
    "PASS: known-positive mask is symmetric."
)


diagonal_positive_count = int(
    torch.diagonal(
        known_positive_mask
    ).sum().item()
)

assert diagonal_positive_count == 0

print(
    "PASS: known-positive mask has no self-DDIs."
)


directed_positive_count = int(
    known_positive_mask.sum().item()
)

upper_positive_count = int(
    torch.triu(
        known_positive_mask,
        diagonal=1
    ).sum().item()
)


print(
    "\nDirected positive entries:",
    f"{directed_positive_count:,}"
)

print(
    "Unique undirected positives:",
    f"{upper_positive_count:,}"
)


assert directed_positive_count == 2_672_628
assert upper_positive_count == 1_336_314


# ------------------------------------------------------------
# Reconstruct ALL known DDI pairs from original fixed splits
# ------------------------------------------------------------

original_train = torch.load(
    TENSOR_DIR / "ddi_train.pt",
    map_location="cpu"
)["pair_index"]

original_val = torch.load(
    TENSOR_DIR / "ddi_val.pt",
    map_location="cpu"
)["pair_index"]

original_test = torch.load(
    TENSOR_DIR / "ddi_test.pt",
    map_location="cpu"
)["pair_index"]


all_known_pairs = torch.cat(
    [
        original_train,
        original_val,
        original_test
    ],
    dim=1
)

assert all_known_pairs.shape[1] == 1_336_314


# ------------------------------------------------------------
# Convert global node IDs -> local drug indices
# ------------------------------------------------------------

src_local = torch.searchsorted(
    drug_node_ids,
    all_known_pairs[0]
)

dst_local = torch.searchsorted(
    drug_node_ids,
    all_known_pairs[1]
)


# Make sure searchsorted really found each global ID
assert torch.equal(
    drug_node_ids[src_local],
    all_known_pairs[0]
)

assert torch.equal(
    drug_node_ids[dst_local],
    all_known_pairs[1]
)


# ------------------------------------------------------------
# Every original DDI must exist in BOTH mask directions
# ------------------------------------------------------------

forward_present = known_positive_mask[
    src_local,
    dst_local
]

reverse_present = known_positive_mask[
    dst_local,
    src_local
]

assert bool(forward_present.all())
assert bool(reverse_present.all())


print(
    "PASS: every train/val/test DDI exists "
    "in the positive mask."
)

print(
    "PASS: every known DDI is represented "
    "in both directions."
)


# ------------------------------------------------------------
# Exactness argument:
#
# We have 1,336,314 unique split pairs.
# The upper triangle contains exactly 1,336,314 positives.
# Every split pair is present.
#
# Therefore there are no additional unknown pairs marked
# positive in the mask.
# ------------------------------------------------------------

assert (
    upper_positive_count
    == all_known_pairs.shape[1]
)

print(
    "PASS: mask contains no additional "
    "positive DDI pairs."
)


print("\n" + "=" * 70)
print(
    "PASS: complete known-positive mask "
    "is exact and ready for filtered ranking."
)


In [ ]:
# ============================================================
# STEP 33 — Orient primary test pairs: COLD -> WARM
# ============================================================

cold_ids_cpu = cold_drug_ids.cpu()

src = cold_test_one_sided[0]
dst = cold_test_one_sided[1]

src_is_cold = torch.isin(
    src,
    cold_ids_cpu
)

dst_is_cold = torch.isin(
    dst,
    cold_ids_cpu
)

# Exactly one endpoint must be cold
assert bool(
    (src_is_cold ^ dst_is_cold).all()
)

cold_query = torch.where(
    src_is_cold,
    src,
    dst
)

warm_target = torch.where(
    src_is_cold,
    dst,
    src
)

primary_cold_pairs = torch.stack(
    [
        cold_query,
        warm_target
    ],
    dim=0
)


# ------------------------------------------------------------
# Verify orientation
# ------------------------------------------------------------

assert primary_cold_pairs.shape == (
    2,
    17_176
)

assert bool(
    torch.isin(
        primary_cold_pairs[0],
        cold_ids_cpu
    ).all()
)

assert not bool(
    torch.isin(
        primary_cold_pairs[1],
        cold_ids_cpu
    ).any()
)


print("PRIMARY COLD-START TEST ORIENTATION")
print("=" * 70)

print(
    "Pairs:",
    f"{primary_cold_pairs.shape[1]:,}"
)

print(
    "Cold queries:",
    f"{torch.isin(primary_cold_pairs[0], cold_ids_cpu).sum().item():,}"
)

print(
    "Cold targets:",
    f"{torch.isin(primary_cold_pairs[1], cold_ids_cpu).sum().item():,}"
)

print(
    "\nPASS: every primary pair is oriented "
    "cold query -> warm target."
)

In [ ]:
# ============================================================
# STEP 34 — Filtered ranking evaluator
# ============================================================

@torch.no_grad()
def evaluate_filtered_ranking(
    model,
    edge_index,
    edge_type,
    evaluation_pairs,
    drug_node_ids,
    known_positive_mask,
    device,
    batch_size=128
):
    model.eval()

    # --------------------------------------------------------
    # Encode graph once
    # --------------------------------------------------------

    z = model.encode(
        edge_index,
        edge_type
    )

    candidate_global = (
        drug_node_ids.to(device)
    )

    candidate_z = z[
        candidate_global
    ]

    relation = model.ddi_relation

    # Keep filtering mask on CPU.
    # Each row is only 4,278 booleans.
    positive_mask_cpu = (
        known_positive_mask.cpu()
    )

    drug_ids_cpu = (
        drug_node_ids.cpu()
    )

    eval_pairs_cpu = (
        evaluation_pairs.cpu()
    )

    ranks = []

    n_queries = (
        eval_pairs_cpu.shape[1]
    )


    # --------------------------------------------------------
    # Batched queries
    # --------------------------------------------------------

    for start in range(
        0,
        n_queries,
        batch_size
    ):

        end = min(
            start + batch_size,
            n_queries
        )

        query_global_cpu = (
            eval_pairs_cpu[0, start:end]
        )

        target_global_cpu = (
            eval_pairs_cpu[1, start:end]
        )


        # Global graph IDs -> local drug IDs
        query_local = torch.searchsorted(
            drug_ids_cpu,
            query_global_cpu
        )

        target_local = torch.searchsorted(
            drug_ids_cpu,
            target_global_cpu
        )


        # Exact membership verification
        assert torch.equal(
            drug_ids_cpu[query_local],
            query_global_cpu
        )

        assert torch.equal(
            drug_ids_cpu[target_local],
            target_global_cpu
        )


        query_global = (
            query_global_cpu.to(device)
        )

        target_local_gpu = (
            target_local.to(device)
        )


        # ----------------------------------------------------
        # DistMult scores against ALL 4,278 drug candidates
        #
        # [B, 128] @ [128, 4278]
        # -> [B, 4278]
        # ----------------------------------------------------

        query_repr = (
            z[query_global]
            * relation
        )

        scores = (
            query_repr
            @ candidate_z.T
        )


        # Save target scores BEFORE filtering
        row_ids = torch.arange(
            end - start,
            device=device
        )

        target_scores = scores[
            row_ids,
            target_local_gpu
        ].clone()


        # ----------------------------------------------------
        # Filter:
        #   1. all other known positive DDIs
        #   2. self candidate
        #   3. restore actual target
        # ----------------------------------------------------

        filter_mask_cpu = (
            positive_mask_cpu[
                query_local
            ].clone()
        )

        # self
        filter_mask_cpu[
            torch.arange(end - start),
            query_local
        ] = True

        filter_mask = (
            filter_mask_cpu.to(device)
        )

        scores[
            filter_mask
        ] = -torch.inf

        # Restore target
        scores[
            row_ids,
            target_local_gpu
        ] = target_scores


        # ----------------------------------------------------
        # Rank:
        #
        # 1 + number of candidates scoring strictly higher
        #
        # This matches the original evaluator's rank rule.
        # ----------------------------------------------------

        batch_ranks = (
            1
            + (
                scores
                > target_scores.unsqueeze(1)
            ).sum(dim=1)
        )

        ranks.append(
            batch_ranks.cpu()
        )


    ranks = torch.cat(
        ranks
    ).to(torch.float64)


    metrics = {
        "num_queries":
            int(ranks.numel()),

        "mean_reciprocal_rank":
            float((1.0 / ranks).mean()),

        "hits_at_1":
            float((ranks <= 1).double().mean()),

        "hits_at_5":
            float((ranks <= 5).double().mean()),

        "hits_at_10":
            float((ranks <= 10).double().mean()),

        "mean_rank":
            float(ranks.mean()),

        "median_rank":
            float(ranks.median()),
    }

    return metrics, ranks


print("PASS: filtered ranking evaluator defined.")

In [ ]:
# ============================================================
# STEP 34B — 100-query evaluator smoke test
# ============================================================

SMOKE_EVAL_SIZE = 100

smoke_eval_pairs = (
    primary_cold_pairs[
        :,
        :SMOKE_EVAL_SIZE
    ]
)


smoke_metrics, smoke_ranks = (
    evaluate_filtered_ranking(
        model=model,
        edge_index=edge_index,
        edge_type=edge_type,
        evaluation_pairs=smoke_eval_pairs,
        drug_node_ids=drug_node_ids,
        known_positive_mask=known_positive_mask,
        device=device,
        batch_size=50
    )
)


assert smoke_metrics["num_queries"] == 100

assert bool(
    (smoke_ranks >= 1).all()
)

assert bool(
    (smoke_ranks <= 4_278).all()
)


print("100-QUERY COLD EVALUATION SMOKE TEST")
print("=" * 70)

for key, value in smoke_metrics.items():

    if key == "num_queries":
        print(
            f"{key}: {value:,}"
        )

    else:
        print(
            f"{key}: {value:.6f}"
        )


print(
    "\nMinimum rank:",
    int(smoke_ranks.min())
)

print(
    "Maximum rank:",
    int(smoke_ranks.max())
)

print(
    "\nPASS: filtered cold-query evaluation "
    "works correctly."
)

## 11. Seed-42 Primary Evaluation

Evaluate and save the initial G0 and G3 seed-42 cold→warm results.

These cells are retained as part of the original experimental workflow. Final reported results must use the validated checkpoint-based reproduction section appended later in this notebook.


In [ ]:
# ============================================================
# STEP 35 — Full primary G0-cold seed-42 evaluation
# ============================================================

import time

print("G0-COLD SEED-42 — PRIMARY COLD-START EVALUATION")
print("=" * 75)

print(
    "Evaluation direction: cold drug -> warm target"
)

print(
    "Queries:",
    f"{primary_cold_pairs.shape[1]:,}"
)

print(
    "Candidate drugs:",
    f"{drug_node_ids.numel():,}"
)

print(
    "Filtering: all known DDIs + self"
)

print()


eval_start = time.time()


g0_cold_primary_metrics, g0_cold_primary_ranks = (
    evaluate_filtered_ranking(
        model=model,
        edge_index=edge_index,
        edge_type=edge_type,
        evaluation_pairs=primary_cold_pairs,
        drug_node_ids=drug_node_ids,
        known_positive_mask=known_positive_mask,
        device=device,
        batch_size=128
    )
)


torch.cuda.synchronize()

eval_seconds = (
    time.time() - eval_start
)


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert (
    g0_cold_primary_metrics[
        "num_queries"
    ]
    == 17_176
)

assert bool(
    (g0_cold_primary_ranks >= 1).all()
)

assert bool(
    (
        g0_cold_primary_ranks
        <= 4_278
    ).all()
)


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\nPRIMARY COLD-START RESULTS")
print("-" * 75)

print(
    "MRR:",
    f"{g0_cold_primary_metrics['mean_reciprocal_rank']:.6f}"
)

print(
    "Hits@1:",
    f"{g0_cold_primary_metrics['hits_at_1']:.6f}"
)

print(
    "Hits@5:",
    f"{g0_cold_primary_metrics['hits_at_5']:.6f}"
)

print(
    "Hits@10:",
    f"{g0_cold_primary_metrics['hits_at_10']:.6f}"
)

print(
    "Mean rank:",
    f"{g0_cold_primary_metrics['mean_rank']:.2f}"
)

print(
    "Median rank:",
    f"{g0_cold_primary_metrics['median_rank']:.2f}"
)

print(
    "Minimum rank:",
    int(g0_cold_primary_ranks.min())
)

print(
    "Maximum rank:",
    int(g0_cold_primary_ranks.max())
)

print(
    "\nEvaluation time:",
    f"{eval_seconds:.2f} sec"
)

print(
    "\nPASS: full G0-cold primary evaluation completed."
)

In [ ]:
# ============================================================
# STEP 36 — Save G0-cold seed-42 primary evaluation
# ============================================================

import json
import torch

primary_result_path = (
    COLD_RESULT_DIR
    / "primary_cold_evaluation.json"
)

primary_ranks_path = (
    COLD_RESULT_DIR
    / "primary_cold_ranks.pt"
)


primary_result = {
    "experiment": "ddi_edge_cold_start",
    "graph_variant": "G0",

    "split_seed": 42,
    "model_seed": 42,

    "evaluation": "primary_one_sided_cold",
    "direction": "cold_query_to_warm_target",

    "num_queries":
        int(
            g0_cold_primary_metrics[
                "num_queries"
            ]
        ),

    "candidate_drugs":
        int(drug_node_ids.numel()),

    "filtering":
        "all_known_positive_ddis_plus_self",

    "mean_reciprocal_rank":
        float(
            g0_cold_primary_metrics[
                "mean_reciprocal_rank"
            ]
        ),

    "hits_at_1":
        float(
            g0_cold_primary_metrics[
                "hits_at_1"
            ]
        ),

    "hits_at_5":
        float(
            g0_cold_primary_metrics[
                "hits_at_5"
            ]
        ),

    "hits_at_10":
        float(
            g0_cold_primary_metrics[
                "hits_at_10"
            ]
        ),

    "mean_rank":
        float(
            g0_cold_primary_metrics[
                "mean_rank"
            ]
        ),

    "median_rank":
        float(
            g0_cold_primary_metrics[
                "median_rank"
            ]
        ),

    "min_rank":
        int(
            g0_cold_primary_ranks.min()
        ),

    "max_rank":
        int(
            g0_cold_primary_ranks.max()
        ),
}


with open(
    primary_result_path,
    "w"
) as f:

    json.dump(
        primary_result,
        f,
        indent=2
    )


torch.save(
    {
        "ranks":
            g0_cold_primary_ranks,

        "evaluation_pairs":
            primary_cold_pairs,

        "direction":
            "cold_query_to_warm_target",

        "split_seed":
            42,

        "model_seed":
            42,

        "graph_variant":
            "G0",
    },
    primary_ranks_path
)


print("G0 PRIMARY COLD-START RESULT SAVED")
print("=" * 70)

print(
    "Metrics:",
    primary_result_path
)

print(
    "Ranks:",
    primary_ranks_path
)

print()

print(
    "MRR:",
    f"{primary_result['mean_reciprocal_rank']:.6f}"
)

print(
    "Hits@1:",
    f"{primary_result['hits_at_1']:.6f}"
)

print(
    "Hits@5:",
    f"{primary_result['hits_at_5']:.6f}"
)

print(
    "Hits@10:",
    f"{primary_result['hits_at_10']:.6f}"
)

print(
    "\nPASS: G0-cold seed-42 primary result saved."
)

In [ ]:
# ============================================================
# STEP 36B — Reload G0 evaluation result
# ============================================================

with open(
    primary_result_path,
    "r"
) as f:

    saved_primary = json.load(f)

saved_rank_data = torch.load(
    primary_ranks_path,
    map_location="cpu"
)


assert (
    saved_primary["num_queries"]
    == 17_176
)

assert (
    saved_primary["direction"]
    == "cold_query_to_warm_target"
)

assert abs(
    saved_primary["mean_reciprocal_rank"]
    - 0.167135
) < 1e-6

assert (
    saved_rank_data["ranks"].numel()
    == 17_176
)

assert torch.equal(
    saved_rank_data["ranks"],
    g0_cold_primary_ranks
)

assert torch.equal(
    saved_rank_data["evaluation_pairs"],
    primary_cold_pairs
)


print("G0 RESULT RELOAD VERIFICATION")
print("=" * 70)

print(
    "Queries:",
    f"{saved_primary['num_queries']:,}"
)

print(
    "Direction:",
    saved_primary["direction"]
)

print(
    "MRR:",
    f"{saved_primary['mean_reciprocal_rank']:.6f}"
)

print(
    "Hits@1:",
    f"{saved_primary['hits_at_1']:.6f}"
)

print(
    "Hits@5:",
    f"{saved_primary['hits_at_5']:.6f}"
)

print(
    "Hits@10:",
    f"{saved_primary['hits_at_10']:.6f}"
)

print(
    "Exact rank tensor match:",
    True
)

print(
    "\nPASS: G0 primary evaluation "
    "is safely stored."
)


In [ ]:
# ============================================================
# STEP 37 — Initialize pristine G3-cold seed-42 model
# ============================================================

import torch

print("G3-COLD SEED-42 INITIALIZATION")
print("=" * 70)


# ------------------------------------------------------------
# G3 cold-start graph -> GPU
# ------------------------------------------------------------

g3_edge_index = (
    g3_cold["edge_index"]
    .to(device)
)

g3_edge_type = (
    g3_cold["edge_type"]
    .to(device)
)


# ------------------------------------------------------------
# Verify graph structure
# ------------------------------------------------------------

g3_ddi_count = int(
    (g3_edge_type == 0).sum().item()
)

g3_bio_count = int(
    (g3_edge_type != 0).sum().item()
)

assert g3_ddi_count == 1_855_564
assert g3_bio_count == 136_568

assert (
    g3_edge_index.shape[1]
    == 1_992_132
)


# ------------------------------------------------------------
# Verify cold drugs still have ZERO DDI edges
# ------------------------------------------------------------

cold_gpu = cold_drug_ids.to(device)

ddi_edges = g3_edge_index[
    :,
    g3_edge_type == 0
]

cold_in_g3_ddi = bool(
    torch.isin(
        ddi_edges.reshape(-1),
        cold_gpu
    ).any()
)

assert cold_in_g3_ddi is False


# ------------------------------------------------------------
# Recreate model from EXACT SAME seed
# ------------------------------------------------------------

MODEL_SEED = 42

reset_seed(
    MODEL_SEED
)

g3_model = RGCNDDIModel(
    num_nodes=NUM_NODES,
    num_relations=NUM_RELATIONS,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT
).to(device)


g3_optimizer = torch.optim.Adam(
    g3_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)


# ------------------------------------------------------------
# Parameter check
# ------------------------------------------------------------

g3_total_params = sum(
    p.numel()
    for p in g3_model.parameters()
)

assert g3_total_params == 2_200_704


# ------------------------------------------------------------
# Forward smoke test
# ------------------------------------------------------------

g3_model.eval()

with torch.no_grad():

    g3_z = g3_model.encode(
        g3_edge_index,
        g3_edge_type
    )

    g3_smoke_logits = g3_model.decode(
        g3_z,
        warm_train_pairs[:, :2048].to(device)
    )


assert g3_z.shape == (
    13_094,
    128
)

assert bool(
    torch.isfinite(
        g3_smoke_logits
    ).all()
)


print("GRAPH")
print("-" * 70)

print(
    "Directed DDI edges:",
    f"{g3_ddi_count:,}"
)

print(
    "Directed biomedical edges:",
    f"{g3_bio_count:,}"
)

print(
    "Total directed edges:",
    f"{g3_edge_index.shape[1]:,}"
)

print(
    "Cold drug appears in DDI edges:",
    cold_in_g3_ddi
)


print("\nMODEL")
print("-" * 70)

print(
    "Model seed:",
    MODEL_SEED
)

print(
    "Parameters:",
    f"{g3_total_params:,}"
)

print(
    "Node representation:",
    tuple(g3_z.shape)
)

print(
    "Forward logits finite:",
    bool(
        torch.isfinite(
            g3_smoke_logits
        ).all()
    )
)


print("\nGPU MEMORY")
print("-" * 70)

print(
    "Allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    "Reserved:",
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)

print(
    "Free:",
    f"{free_bytes / 1024**3:.2f} GB"
)


print(
    "\nPASS: pristine G3-cold seed-42 "
    "model and graph are ready."
)

In [ ]:
# ============================================================
# STEP 38 — G3-cold seed-42 real training epoch 1
# ============================================================

import time
import torch

print("G3-COLD — REAL TRAINING EPOCH 1")
print("=" * 70)

start_time = time.time()


g3_train_loss_epoch1 = train_one_epoch(
    model=g3_model,
    optimizer=g3_optimizer,

    edge_index=g3_edge_index,
    edge_type=g3_edge_type,

    train_positive_pairs=warm_train_pairs,
    train_negative_pairs=warm_train_negatives,

    epoch=1,
    model_seed=MODEL_SEED,
    device=device
)

torch.cuda.synchronize()

after_train = time.time()


g3_val_loss_epoch1 = validation_loss(
    model=g3_model,

    edge_index=g3_edge_index,
    edge_type=g3_edge_type,

    val_positive_pairs=warm_val_pairs,
    val_negative_pairs=warm_val_negatives,

    device=device
)

torch.cuda.synchronize()

end_time = time.time()


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert torch.isfinite(
    torch.tensor(
        g3_train_loss_epoch1
    )
)

assert torch.isfinite(
    torch.tensor(
        g3_val_loss_epoch1
    )
)

assert all(
    bool(torch.isfinite(p).all())
    for p in g3_model.parameters()
)


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print(
    "Training loss:",
    f"{g3_train_loss_epoch1:.6f}"
)

print(
    "Warm validation loss:",
    f"{g3_val_loss_epoch1:.6f}"
)

print(
    "Training time:",
    f"{after_train - start_time:.2f} sec"
)

print(
    "Validation time:",
    f"{end_time - after_train:.2f} sec"
)

print(
    "Total epoch time:",
    f"{end_time - start_time:.2f} sec"
)


print("\nGPU MEMORY")
print("-" * 70)

print(
    "Allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    "Reserved:",
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)

print(
    "Free:",
    f"{free_bytes / 1024**3:.2f} GB"
)


print(
    "\nPASS: real G3-cold epoch 1 completed."
)


In [ ]:
# ============================================================
# STEP 39 — Safe CUDA cache cleanup after G3 epoch 1
# ============================================================

import gc
import torch

# Remove temporary forward-smoke tensors from Step 37.
# They are not needed for training.
if "g3_z" in globals():
    del g3_z

if "g3_smoke_logits" in globals():
    del g3_smoke_logits

gc.collect()

# Releases UNUSED cached memory only.
# Does NOT delete model parameters or optimizer state.
torch.cuda.empty_cache()
torch.cuda.synchronize()


# ------------------------------------------------------------
# Verify model is still healthy
# ------------------------------------------------------------

assert all(
    bool(torch.isfinite(p).all())
    for p in g3_model.parameters()
)

assert len(g3_optimizer.state) > 0


free_bytes, total_bytes = torch.cuda.mem_get_info()

print("G3 CUDA MEMORY AFTER SAFE CACHE CLEANUP")
print("=" * 70)

print(
    "Allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    "Reserved:",
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

print(
    "Free:",
    f"{free_bytes / 1024**3:.2f} GB"
)

print(
    "Optimizer state exists:",
    len(g3_optimizer.state) > 0
)

print(
    "Model parameters finite:",
    True
)

print(
    "\nPASS: G3 epoch-1 training state preserved "
    "and unused CUDA cache released."
)

In [ ]:
# ============================================================
# STEP 40 — Continue G3-cold seed-42 training
#            Epoch 2 -> max 500
# ============================================================

import copy
import time
import math

print("G3-COLD — FULL TRAINING")
print("=" * 70)

print("Continuing from completed epoch 1.")
print("Maximum epoch:", MAX_EPOCHS)
print("Early stopping patience:", PATIENCE)
print("Model seed:", MODEL_SEED)
print("Validation: warm-only")
print()


# ------------------------------------------------------------
# Initialize early stopping from the REAL epoch 1
# ------------------------------------------------------------

g3_best_val_loss = g3_val_loss_epoch1
g3_best_epoch = 1

g3_best_model_state = copy.deepcopy(
    g3_model.state_dict()
)

g3_patience_counter = 0


g3_history = [
    {
        "epoch": 1,
        "train_loss": g3_train_loss_epoch1,
        "val_loss": g3_val_loss_epoch1
    }
]


g3_training_start = time.time()


# ------------------------------------------------------------
# Continue epoch 2 -> 500
# ------------------------------------------------------------

for epoch in range(2, MAX_EPOCHS + 1):

    epoch_start = time.time()

    g3_train_loss = train_one_epoch(
        model=g3_model,
        optimizer=g3_optimizer,

        edge_index=g3_edge_index,
        edge_type=g3_edge_type,

        train_positive_pairs=warm_train_pairs,
        train_negative_pairs=warm_train_negatives,

        epoch=epoch,
        model_seed=MODEL_SEED,
        device=device
    )


    g3_val_loss = validation_loss(
        model=g3_model,

        edge_index=g3_edge_index,
        edge_type=g3_edge_type,

        val_positive_pairs=warm_val_pairs,
        val_negative_pairs=warm_val_negatives,

        device=device
    )


    torch.cuda.synchronize()

    epoch_seconds = (
        time.time() - epoch_start
    )


    # --------------------------------------------------------
    # Safety checks
    # --------------------------------------------------------

    if not math.isfinite(g3_train_loss):
        raise RuntimeError(
            f"Non-finite training loss "
            f"at epoch {epoch}: "
            f"{g3_train_loss}"
        )

    if not math.isfinite(g3_val_loss):
        raise RuntimeError(
            f"Non-finite validation loss "
            f"at epoch {epoch}: "
            f"{g3_val_loss}"
        )


    g3_history.append(
        {
            "epoch": epoch,
            "train_loss": g3_train_loss,
            "val_loss": g3_val_loss
        }
    )


    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    improved = (
        g3_val_loss
        < g3_best_val_loss
    )


    if improved:

        g3_best_val_loss = (
            g3_val_loss
        )

        g3_best_epoch = epoch

        g3_best_model_state = (
            copy.deepcopy(
                g3_model.state_dict()
            )
        )

        g3_patience_counter = 0

    else:

        g3_patience_counter += 1


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (
        epoch <= 10
        or epoch % 10 == 0
        or improved
    ):

        marker = "*" if improved else ""

        print(
            f"Epoch {epoch:03d} | "
            f"train={g3_train_loss:.6f} | "
            f"val={g3_val_loss:.6f} | "
            f"best={g3_best_val_loss:.6f} "
            f"(epoch {g3_best_epoch}) | "
            f"patience="
            f"{g3_patience_counter}/{PATIENCE} | "
            f"{epoch_seconds:.2f}s "
            f"{marker}"
        )


    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if (
        g3_patience_counter
        >= PATIENCE
    ):

        print()

        print(
            f"EARLY STOPPING "
            f"at epoch {epoch}."
        )

        break


# ------------------------------------------------------------
# Restore BEST G3 model
# ------------------------------------------------------------

g3_model.load_state_dict(
    g3_best_model_state
)

g3_model.eval()


g3_training_seconds = (
    time.time()
    - g3_training_start
)

g3_last_epoch = (
    g3_history[-1]["epoch"]
)


# ------------------------------------------------------------
# Final safety check
# ------------------------------------------------------------

assert all(
    bool(torch.isfinite(p).all())
    for p in g3_model.parameters()
)


print("\n" + "=" * 70)

print(
    "G3-COLD TRAINING COMPLETE"
)

print("=" * 70)

print(
    "Last epoch executed:",
    g3_last_epoch
)

print(
    "Best epoch:",
    g3_best_epoch
)

print(
    "Best warm validation loss:",
    f"{g3_best_val_loss:.6f}"
)

print(
    "Total epochs represented:",
    len(g3_history)
)

print(
    "Continuation training time:",
    f"{g3_training_seconds:.2f} sec"
)

print(
    "Best model restored:",
    True
)

print(
    "All restored parameters finite:",
    True
)


print("\nGPU MEMORY")
print("-" * 70)

print(
    "Allocated:",
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    "Reserved:",
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)

free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)

print(
    "Free:",
    f"{free_bytes / 1024**3:.2f} GB"
)


print(
    "\nPASS: G3-cold seed-42 training "
    "completed and best model restored."
)

In [ ]:
# ============================================================
# STEP 41 — Save G3-cold seed-42 checkpoint + history
# ============================================================

from pathlib import Path
import json
import torch


G3_CHECKPOINT_DIR = (
    PROJECT_DIR
    / "checkpoints"
    / "cold_start"
    / "split_seed_42"
    / "G3"
    / "model_seed_42"
)

G3_RESULT_DIR = (
    PROJECT_DIR
    / "results"
    / "cold_start"
    / "split_seed_42"
    / "G3"
    / "model_seed_42"
)

G3_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

G3_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Save checkpoint
# ------------------------------------------------------------

g3_checkpoint_path = (
    G3_CHECKPOINT_DIR
    / "best_model.pt"
)

torch.save(
    {
        "model_state_dict":
            g3_model.state_dict(),

        "experiment":
            "ddi_edge_cold_start",

        "graph_variant":
            "G3",

        "split_seed":
            42,

        "model_seed":
            MODEL_SEED,

        "best_epoch":
            g3_best_epoch,

        "best_val_loss":
            g3_best_val_loss,

        "num_nodes":
            NUM_NODES,

        "num_relations":
            NUM_RELATIONS,

        "embedding_dim":
            EMBEDDING_DIM,

        "hidden_dim":
            HIDDEN_DIM,

        "dropout":
            DROPOUT,

        "learning_rate":
            1e-3,

        "weight_decay":
            1e-5,

        "max_epochs":
            MAX_EPOCHS,

        "patience":
            PATIENCE,

        "train_sample_size":
            TRAIN_SAMPLE_SIZE,

        "validation_protocol":
            "warm-only",

        "cold_start_scope":
            (
                "DDI-edge cold-start; cold drugs receive "
                "no supervised DDI positives or negatives; "
                "biomedical message-passing edges retained"
            ),
    },
    g3_checkpoint_path
)


# ------------------------------------------------------------
# Save history
# ------------------------------------------------------------

g3_history_path = (
    G3_RESULT_DIR
    / "training_history.json"
)

with open(
    g3_history_path,
    "w"
) as f:

    json.dump(
        g3_history,
        f,
        indent=2
    )


# ------------------------------------------------------------
# Save training summary
# ------------------------------------------------------------

g3_summary = {
    "experiment":
        "ddi_edge_cold_start",

    "graph_variant":
        "G3",

    "split_seed":
        42,

    "model_seed":
        MODEL_SEED,

    "best_epoch":
        int(g3_best_epoch),

    "best_val_loss":
        float(g3_best_val_loss),

    "last_epoch":
        int(g3_last_epoch),

    "warm_train_positive_count":
        int(warm_train_pairs.shape[1]),

    "warm_train_negative_count":
        int(warm_train_negatives.shape[1]),

    "warm_validation_positive_count":
        int(warm_val_pairs.shape[1]),

    "warm_validation_negative_count":
        int(warm_val_negatives.shape[1]),

    "cold_drug_count":
        int(cold_drug_ids.numel()),

    "directed_ddi_edges":
        int(g3_ddi_count),

    "directed_biomedical_edges":
        int(g3_bio_count),

    "validation_protocol":
        "warm-only",

    "test_evaluated":
        False,
}


g3_summary_path = (
    G3_RESULT_DIR
    / "training_summary.json"
)

with open(
    g3_summary_path,
    "w"
) as f:

    json.dump(
        g3_summary,
        f,
        indent=2
    )


print("G3-COLD SEED-42 ARTIFACTS SAVED")
print("=" * 70)

print("Checkpoint:")
print(g3_checkpoint_path)

print("\nTraining history:")
print(g3_history_path)

print("\nTraining summary:")
print(g3_summary_path)


In [ ]:
# ============================================================
# STEP 41B — Reload G3 checkpoint verification
# ============================================================

saved_g3_checkpoint = torch.load(
    g3_checkpoint_path,
    map_location="cpu"
)


assert (
    saved_g3_checkpoint["graph_variant"]
    == "G3"
)

assert (
    saved_g3_checkpoint["split_seed"]
    == 42
)

assert (
    saved_g3_checkpoint["model_seed"]
    == 42
)

assert (
    saved_g3_checkpoint["best_epoch"]
    == g3_best_epoch
)

assert abs(
    saved_g3_checkpoint["best_val_loss"]
    - g3_best_val_loss
) < 1e-12


# ------------------------------------------------------------
# Exact parameter verification
# ------------------------------------------------------------

current_g3_state = {
    name: tensor.detach().cpu()
    for name, tensor
    in g3_model.state_dict().items()
}

saved_g3_state = (
    saved_g3_checkpoint[
        "model_state_dict"
    ]
)

assert (
    current_g3_state.keys()
    == saved_g3_state.keys()
)

for name in current_g3_state:

    assert torch.equal(
        current_g3_state[name],
        saved_g3_state[name]
    ), f"Parameter mismatch: {name}"


# ------------------------------------------------------------
# History verification
# ------------------------------------------------------------

with open(
    g3_history_path,
    "r"
) as f:

    saved_g3_history = json.load(f)

assert len(saved_g3_history) == 500
assert saved_g3_history[0]["epoch"] == 1
assert saved_g3_history[-1]["epoch"] == 500


print("G3 CHECKPOINT RELOAD VERIFICATION")
print("=" * 70)

print(
    "Graph:",
    saved_g3_checkpoint["graph_variant"]
)

print(
    "Split seed:",
    saved_g3_checkpoint["split_seed"]
)

print(
    "Model seed:",
    saved_g3_checkpoint["model_seed"]
)

print(
    "Best epoch:",
    saved_g3_checkpoint["best_epoch"]
)

print(
    "Best validation loss:",
    f"{saved_g3_checkpoint['best_val_loss']:.6f}"
)

print(
    "History epochs:",
    len(saved_g3_history)
)

print(
    "Exact parameter match:",
    True
)

print(
    "\nPASS: G3-cold seed-42 checkpoint "
    "and history are safely stored."
)

In [ ]:
# ============================================================
# STEP 42 — Full primary G3-cold seed-42 evaluation
# ============================================================

import time

print("G3-COLD SEED-42 — PRIMARY COLD-START EVALUATION")
print("=" * 75)

print("Evaluation direction: cold drug -> warm target")
print(
    "Queries:",
    f"{primary_cold_pairs.shape[1]:,}"
)
print(
    "Candidate drugs:",
    f"{drug_node_ids.numel():,}"
)
print("Filtering: all known DDIs + self")
print()


eval_start = time.time()


g3_cold_primary_metrics, g3_cold_primary_ranks = (
    evaluate_filtered_ranking(
        model=g3_model,

        edge_index=g3_edge_index,
        edge_type=g3_edge_type,

        evaluation_pairs=primary_cold_pairs,

        drug_node_ids=drug_node_ids,
        known_positive_mask=known_positive_mask,

        device=device,
        batch_size=128
    )
)


torch.cuda.synchronize()

g3_eval_seconds = (
    time.time() - eval_start
)


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert (
    g3_cold_primary_metrics[
        "num_queries"
    ]
    == 17_176
)

assert bool(
    (g3_cold_primary_ranks >= 1).all()
)

assert bool(
    (
        g3_cold_primary_ranks
        <= 4_278
    ).all()
)


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print("\nPRIMARY COLD-START RESULTS")
print("-" * 75)

print(
    "MRR:",
    f"{g3_cold_primary_metrics['mean_reciprocal_rank']:.6f}"
)

print(
    "Hits@1:",
    f"{g3_cold_primary_metrics['hits_at_1']:.6f}"
)

print(
    "Hits@5:",
    f"{g3_cold_primary_metrics['hits_at_5']:.6f}"
)

print(
    "Hits@10:",
    f"{g3_cold_primary_metrics['hits_at_10']:.6f}"
)

print(
    "Mean rank:",
    f"{g3_cold_primary_metrics['mean_rank']:.2f}"
)

print(
    "Median rank:",
    f"{g3_cold_primary_metrics['median_rank']:.2f}"
)

print(
    "Minimum rank:",
    int(g3_cold_primary_ranks.min())
)

print(
    "Maximum rank:",
    int(g3_cold_primary_ranks.max())
)

print(
    "\nEvaluation time:",
    f"{g3_eval_seconds:.2f} sec"
)

print(
    "\nPASS: full G3-cold primary "
    "evaluation completed."
)

In [ ]:
# ============================================================
# STEP 43 — Save G3-cold seed-42 primary evaluation
# ============================================================

import json
import torch

g3_primary_result_path = (
    G3_RESULT_DIR
    / "primary_cold_evaluation.json"
)

g3_primary_ranks_path = (
    G3_RESULT_DIR
    / "primary_cold_ranks.pt"
)


g3_primary_result = {
    "experiment": "ddi_edge_cold_start",
    "graph_variant": "G3",

    "split_seed": 42,
    "model_seed": 42,

    "evaluation":
        "primary_one_sided_cold",

    "direction":
        "cold_query_to_warm_target",

    "num_queries":
        int(
            g3_cold_primary_metrics[
                "num_queries"
            ]
        ),

    "candidate_drugs":
        int(drug_node_ids.numel()),

    "filtering":
        "all_known_positive_ddis_plus_self",

    "mean_reciprocal_rank":
        float(
            g3_cold_primary_metrics[
                "mean_reciprocal_rank"
            ]
        ),

    "hits_at_1":
        float(
            g3_cold_primary_metrics[
                "hits_at_1"
            ]
        ),

    "hits_at_5":
        float(
            g3_cold_primary_metrics[
                "hits_at_5"
            ]
        ),

    "hits_at_10":
        float(
            g3_cold_primary_metrics[
                "hits_at_10"
            ]
        ),

    "mean_rank":
        float(
            g3_cold_primary_metrics[
                "mean_rank"
            ]
        ),

    "median_rank":
        float(
            g3_cold_primary_metrics[
                "median_rank"
            ]
        ),

    "min_rank":
        int(
            g3_cold_primary_ranks.min()
        ),

    "max_rank":
        int(
            g3_cold_primary_ranks.max()
        ),
}


with open(
    g3_primary_result_path,
    "w"
) as f:

    json.dump(
        g3_primary_result,
        f,
        indent=2
    )


torch.save(
    {
        "ranks":
            g3_cold_primary_ranks,

        "evaluation_pairs":
            primary_cold_pairs,

        "direction":
            "cold_query_to_warm_target",

        "split_seed":
            42,

        "model_seed":
            42,

        "graph_variant":
            "G3",
    },
    g3_primary_ranks_path
)


print("G3 PRIMARY COLD-START RESULT SAVED")
print("=" * 70)

print(
    "Metrics:",
    g3_primary_result_path
)

print(
    "Ranks:",
    g3_primary_ranks_path
)

print()

print(
    "MRR:",
    f"{g3_primary_result['mean_reciprocal_rank']:.6f}"
)

print(
    "Hits@1:",
    f"{g3_primary_result['hits_at_1']:.6f}"
)

print(
    "Hits@5:",
    f"{g3_primary_result['hits_at_5']:.6f}"
)

print(
    "Hits@10:",
    f"{g3_primary_result['hits_at_10']:.6f}"
)

print(
    "\nPASS: G3-cold seed-42 "
    "primary result saved."
)

In [ ]:
# ============================================================
# STEP 43B — Reload G3 primary result
# ============================================================

with open(
    g3_primary_result_path,
    "r"
) as f:

    saved_g3_primary = json.load(f)


saved_g3_rank_data = torch.load(
    g3_primary_ranks_path,
    map_location="cpu"
)


assert (
    saved_g3_primary["num_queries"]
    == 17_176
)

assert (
    saved_g3_primary["direction"]
    == "cold_query_to_warm_target"
)

assert abs(
    saved_g3_primary[
        "mean_reciprocal_rank"
    ]
    - 0.080328
) < 1e-6


assert (
    saved_g3_rank_data[
        "ranks"
    ].numel()
    == 17_176
)

assert torch.equal(
    saved_g3_rank_data["ranks"],
    g3_cold_primary_ranks
)

assert torch.equal(
    saved_g3_rank_data[
        "evaluation_pairs"
    ],
    primary_cold_pairs
)


# Exact paired-query check against G0
assert torch.equal(
    saved_rank_data[
        "evaluation_pairs"
    ],
    saved_g3_rank_data[
        "evaluation_pairs"
    ]
)


print("G3 RESULT RELOAD VERIFICATION")
print("=" * 70)

print(
    "Queries:",
    f"{saved_g3_primary['num_queries']:,}"
)

print(
    "Direction:",
    saved_g3_primary["direction"]
)

print(
    "MRR:",
    f"{saved_g3_primary['mean_reciprocal_rank']:.6f}"
)

print(
    "Hits@1:",
    f"{saved_g3_primary['hits_at_1']:.6f}"
)

print(
    "Hits@5:",
    f"{saved_g3_primary['hits_at_5']:.6f}"
)

print(
    "Hits@10:",
    f"{saved_g3_primary['hits_at_10']:.6f}"
)

print(
    "Exact G3 rank tensor match:",
    True
)

print(
    "G0/G3 evaluation pairs identical:",
    True
)

print(
    "\nPASS: G3 primary evaluation "
    "is safely stored."
)

## 12. Historical Diagnostics and Additional Model Seeds

The following cells contain pair-level diagnostics and the subsequent model-seed runs used during development.

They are preserved for experimental traceability. Historical stored outputs have been cleared.


In [ ]:
# ============================================================
# STEP 44 — Pair-by-pair G0 vs G3 cold-start diagnostic
# ============================================================

import torch
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# Load saved ranks
# ------------------------------------------------------------

g0_rank_data = torch.load(
    G0_RESULT_DIR / "primary_cold_ranks.pt",
    map_location="cpu"
)

g3_rank_data = torch.load(
    G3_RESULT_DIR / "primary_cold_ranks.pt",
    map_location="cpu"
)


g0_ranks = g0_rank_data["ranks"].long()
g3_ranks = g3_rank_data["ranks"].long()

pairs = g0_rank_data["evaluation_pairs"].long()


# ------------------------------------------------------------
# Exact comparability checks
# ------------------------------------------------------------

assert torch.equal(
    pairs,
    g3_rank_data["evaluation_pairs"]
)

assert torch.equal(
    pairs,
    primary_cold_pairs.cpu()
)

assert g0_ranks.numel() == 17_176
assert g3_ranks.numel() == 17_176

assert bool((g0_ranks >= 1).all())
assert bool((g3_ranks >= 1).all())


# ------------------------------------------------------------
# Rank difference
#
# Positive delta:
#     G3 rank is numerically larger -> WORSE
#
# Negative delta:
#     G3 rank is numerically smaller -> BETTER
# ------------------------------------------------------------

rank_delta = (
    g3_ranks - g0_ranks
)


improved = rank_delta < 0
worsened = rank_delta > 0
tied = rank_delta == 0


n = rank_delta.numel()

n_improved = int(improved.sum())
n_worsened = int(worsened.sum())
n_tied = int(tied.sum())


# ------------------------------------------------------------
# Reciprocal-rank difference
#
# Positive = G3 better
# Negative = G3 worse
# ------------------------------------------------------------

g0_rr = 1.0 / g0_ranks.float()
g3_rr = 1.0 / g3_ranks.float()

rr_delta = (
    g3_rr - g0_rr
)


# ------------------------------------------------------------
# Basic paired statistics
# ------------------------------------------------------------

print("G0 vs G3 — PAIR-BY-PAIR COLD-START DIAGNOSTIC")
print("=" * 75)

print(
    "Queries:",
    f"{n:,}"
)

print()

print("QUERY-LEVEL OUTCOMES")
print("-" * 75)

print(
    "G3 improved:",
    f"{n_improved:,}",
    f"({100*n_improved/n:.2f}%)"
)

print(
    "G3 worsened:",
    f"{n_worsened:,}",
    f"({100*n_worsened/n:.2f}%)"
)

print(
    "Exact ties:",
    f"{n_tied:,}",
    f"({100*n_tied/n:.2f}%)"
)


print("\nRANK CHANGE  (G3 rank - G0 rank)")
print("-" * 75)

print(
    "Mean:",
    f"{rank_delta.float().mean():.2f}"
)

print(
    "Median:",
    f"{rank_delta.float().median():.2f}"
)

for q in [
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95
]:

    value = torch.quantile(
        rank_delta.float(),
        q
    )

    print(
        f"{int(q*100):>2}% percentile:",
        f"{value.item():.2f}"
    )


print("\nRECIPROCAL-RANK CHANGE")
print("-" * 75)

print(
    "Mean RR delta:",
    f"{rr_delta.mean():.6f}"
)

print(
    "Median RR delta:",
    f"{rr_delta.median():.6f}"
)


# ------------------------------------------------------------
# Threshold movement
# ------------------------------------------------------------

print("\nHITS THRESHOLD MOVEMENT")
print("-" * 75)

for k in [1, 5, 10]:

    g0_hit = (
        g0_ranks <= k
    )

    g3_hit = (
        g3_ranks <= k
    )

    gained = int(
        ((~g0_hit) & g3_hit).sum()
    )

    lost = int(
        (g0_hit & (~g3_hit)).sum()
    )

    retained = int(
        (g0_hit & g3_hit).sum()
    )

    print(
        f"Hits@{k}: "
        f"gained={gained:,} | "
        f"lost={lost:,} | "
        f"retained={retained:,}"
    )


# ------------------------------------------------------------
# Cold-drug-level aggregation
# ------------------------------------------------------------

cold_query_ids = (
    pairs[0]
    .cpu()
    .numpy()
)

diagnostic_df = pd.DataFrame(
    {
        "cold_drug_id":
            cold_query_ids,

        "g0_rank":
            g0_ranks.cpu().numpy(),

        "g3_rank":
            g3_ranks.cpu().numpy(),

        "rank_delta":
            rank_delta.cpu().numpy(),

        "g0_rr":
            g0_rr.cpu().numpy(),

        "g3_rr":
            g3_rr.cpu().numpy(),

        "rr_delta":
            rr_delta.cpu().numpy(),
    }
)


per_drug = (
    diagnostic_df
    .groupby("cold_drug_id")
    .agg(
        queries=("cold_drug_id", "size"),

        g0_mean_rank=("g0_rank", "mean"),
        g3_mean_rank=("g3_rank", "mean"),

        mean_rank_delta=("rank_delta", "mean"),

        g0_mrr=("g0_rr", "mean"),
        g3_mrr=("g3_rr", "mean"),

        mean_rr_delta=("rr_delta", "mean"),
    )
    .reset_index()
)


per_drug["mrr_change"] = (
    per_drug["g3_mrr"]
    - per_drug["g0_mrr"]
)


drug_improved = int(
    (per_drug["mrr_change"] > 0).sum()
)

drug_worsened = int(
    (per_drug["mrr_change"] < 0).sum()
)

drug_tied = int(
    (per_drug["mrr_change"] == 0).sum()
)


print("\nCOLD-DRUG LEVEL")
print("-" * 75)

print(
    "Cold drugs evaluated:",
    f"{len(per_drug):,}"
)

print(
    "G3 higher per-drug MRR:",
    f"{drug_improved:,}",
    f"({100*drug_improved/len(per_drug):.2f}%)"
)

print(
    "G3 lower per-drug MRR:",
    f"{drug_worsened:,}",
    f"({100*drug_worsened/len(per_drug):.2f}%)"
)

print(
    "Equal per-drug MRR:",
    f"{drug_tied:,}"
)


# ------------------------------------------------------------
# Show worst and best cold drugs
# ------------------------------------------------------------

print("\n10 LARGEST PER-DRUG MRR DECREASES")
print("-" * 75)

display(
    per_drug
    .sort_values(
        "mrr_change",
        ascending=True
    )
    .head(10)
)


print("\n10 LARGEST PER-DRUG MRR IMPROVEMENTS")
print("-" * 75)

display(
    per_drug
    .sort_values(
        "mrr_change",
        ascending=False
    )
    .head(10)
)


print(
    "\nPASS: paired G0/G3 diagnostic completed."
)

In [ ]:
# ============================================================
# STEP 44A — Restore G0 result directory
# ============================================================

from pathlib import Path

G0_RESULT_DIR = (
    PROJECT_DIR
    / "results"
    / "cold_start"
    / "split_seed_42"
    / "G0"
    / "model_seed_42"
)

G3_RESULT_DIR = (
    PROJECT_DIR
    / "results"
    / "cold_start"
    / "split_seed_42"
    / "G3"
    / "model_seed_42"
)


g0_rank_path = (
    G0_RESULT_DIR
    / "primary_cold_ranks.pt"
)

g3_rank_path = (
    G3_RESULT_DIR
    / "primary_cold_ranks.pt"
)


print("G0 path:")
print(G0_RESULT_DIR)

print("\nG3 path:")
print(G3_RESULT_DIR)

print("\nG0 ranks exist:", g0_rank_path.exists())
print("G3 ranks exist:", g3_rank_path.exists())


assert g0_rank_path.exists(), (
    f"G0 rank file not found: {g0_rank_path}"
)

assert g3_rank_path.exists(), (
    f"G3 rank file not found: {g3_rank_path}"
)


print(
    "\nPASS: G0 and G3 saved rank files found."
)

In [ ]:
# ============================================================
# STEP 45 — Initialize pristine G0-cold model seed 43
#            Cold split remains FIXED at seed 42
# ============================================================

import gc
import random
import numpy as np
import torch


# ------------------------------------------------------------
# Experiment seeds
# ------------------------------------------------------------

SPLIT_SEED = 42
MODEL_SEED = 43


print("G0-COLD — MODEL SEED 43 INITIALIZATION")
print("=" * 75)

print("Cold split seed:", SPLIT_SEED)
print("Model seed:", MODEL_SEED)


# ------------------------------------------------------------
# Verify frozen cold-start data
# ------------------------------------------------------------

assert cold_drug_ids.numel() == 196

assert warm_train_pairs.shape[1] == 927_782
assert warm_train_negatives.shape[1] == 927_782

assert warm_val_pairs.shape[1] == 115_889
assert warm_val_negatives.shape[1] == 115_889

assert warm_test.shape[1] == 115_787

assert primary_cold_pairs.shape[1] == 17_176
assert cold_test_two_sided.shape[1] == 651


# Cold drugs must not appear in supervised training pairs
cold_set_cpu = set(
    cold_drug_ids.cpu().tolist()
)

train_nodes = set(
    warm_train_pairs.cpu().flatten().tolist()
)

train_neg_nodes = set(
    warm_train_negatives.cpu().flatten().tolist()
)

assert cold_set_cpu.isdisjoint(train_nodes)
assert cold_set_cpu.isdisjoint(train_neg_nodes)


# ------------------------------------------------------------
# G0 graph verification
# ------------------------------------------------------------

g0_edge_index = (
    g0_cold["edge_index"]
    .to(device)
)

g0_edge_type = (
    g0_cold["edge_type"]
    .to(device)
)


assert g0_edge_index.shape[1] == 1_855_564

assert bool(
    (g0_edge_type == 0).all()
)


cold_gpu = cold_drug_ids.to(device)

cold_in_g0 = torch.isin(
    g0_edge_index,
    cold_gpu
).any()

assert not bool(cold_in_g0)


# ------------------------------------------------------------
# Remove old model objects if they still exist
#
# Saved checkpoints/results are NOT affected.
# ------------------------------------------------------------

for object_name in [
    "model",
    "optimizer"
]:

    if object_name in globals():
        del globals()[object_name]


gc.collect()
torch.cuda.empty_cache()


# ------------------------------------------------------------
# Deterministic model seed 43
# ------------------------------------------------------------

random.seed(MODEL_SEED)
np.random.seed(MODEL_SEED)

torch.manual_seed(MODEL_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        MODEL_SEED
    )


# ------------------------------------------------------------
# Fresh G0 model
# ------------------------------------------------------------

g0_seed43_model = RGCNDDIModel(
    num_nodes=NUM_NODES,
    num_relations=NUM_RELATIONS,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT
).to(device)


g0_seed43_optimizer = torch.optim.Adam(
    g0_seed43_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)


num_parameters = sum(
    p.numel()
    for p in g0_seed43_model.parameters()
)


assert num_parameters == 2_200_704

# Fresh Adam optimizer must have no state yet.
assert len(
    g0_seed43_optimizer.state
) == 0


# ------------------------------------------------------------
# Forward-only smoke test
# ------------------------------------------------------------

g0_seed43_model.eval()

with torch.no_grad():

    seed43_z = (
        g0_seed43_model.encode(
            g0_edge_index,
            g0_edge_type
        )
    )

    seed43_smoke_logits = (
        g0_seed43_model.decode(
            seed43_z,
            warm_train_pairs[
                :, :2048
            ].to(device)
        )
    )


assert seed43_z.shape == (
    13_094,
    128
)

assert bool(
    torch.isfinite(
        seed43_smoke_logits
    ).all()
)


# ------------------------------------------------------------
# Memory
# ------------------------------------------------------------

free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)


print("\nFROZEN DATA")
print("-" * 75)

print(
    "Cold drugs:",
    f"{cold_drug_ids.numel():,}"
)

print(
    "Warm train positives:",
    f"{warm_train_pairs.shape[1]:,}"
)

print(
    "Warm train negatives:",
    f"{warm_train_negatives.shape[1]:,}"
)

print(
    "Warm validation positives:",
    f"{warm_val_pairs.shape[1]:,}"
)

print(
    "Warm validation negatives:",
    f"{warm_val_negatives.shape[1]:,}"
)

print(
    "Primary cold queries:",
    f"{primary_cold_pairs.shape[1]:,}"
)


print("\nG0 GRAPH")
print("-" * 75)

print(
    "Directed edges:",
    f"{g0_edge_index.shape[1]:,}"
)

print(
    "Cold drug appears in graph:",
    bool(cold_in_g0)
)


print("\nMODEL")
print("-" * 75)

print(
    "Model seed:",
    MODEL_SEED
)

print(
    "Parameters:",
    f"{num_parameters:,}"
)

print(
    "Optimizer state entries:",
    len(g0_seed43_optimizer.state)
)

print(
    "Node representation:",
    tuple(seed43_z.shape)
)

print(
    "Forward logits finite:",
    bool(
        torch.isfinite(
            seed43_smoke_logits
        ).all()
    )
)


print("\nGPU MEMORY")
print("-" * 75)

print(
    "Allocated:",
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)

print(
    "Reserved:",
    f"{torch.cuda.memory_reserved()/1024**3:.2f} GB"
)

print(
    "Free:",
    f"{free_bytes/1024**3:.2f} GB"
)


print(
    "\nPASS: pristine G0-cold model seed 43 "
    "is ready. Cold split remains seed 42."
)


In [ ]:
# ============================================================
# STEP 45A — Restore frozen test partition variables
# ============================================================

from pathlib import Path
import torch

COLD_START_DIR = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "cold_start"
    / "split_seed_42"
)


warm_test_path = (
    COLD_START_DIR
    / "warm_test.pt"
)

two_sided_path = (
    COLD_START_DIR
    / "cold_test_two_sided.pt"
)


print("RESTORING FROZEN COLD-START VARIABLES")
print("=" * 70)

print("Warm test file exists:", warm_test_path.exists())
print("Two-sided test file exists:", two_sided_path.exists())

assert warm_test_path.exists()
assert two_sided_path.exists()


warm_test = torch.load(
    warm_test_path,
    map_location="cpu"
)

cold_test_two_sided = torch.load(
    two_sided_path,
    map_location="cpu"
)


# Handle either a raw tensor or saved dictionary,
# depending on how the artifact was serialized.
if isinstance(warm_test, dict):
    if "pair_index" in warm_test:
        warm_test = warm_test["pair_index"]

if isinstance(cold_test_two_sided, dict):
    if "pair_index" in cold_test_two_sided:
        cold_test_two_sided = (
            cold_test_two_sided["pair_index"]
        )


assert isinstance(warm_test, torch.Tensor)
assert isinstance(cold_test_two_sided, torch.Tensor)

assert warm_test.shape == (2, 115_787)
assert cold_test_two_sided.shape == (2, 651)


print()
print(
    "Warm test:",
    tuple(warm_test.shape)
)

print(
    "Two-sided cold test:",
    tuple(cold_test_two_sided.shape)
)

print(
    "\nPASS: frozen test partitions restored."
)

In [ ]:
# ============================================================
# STEP 46 — G0-cold model seed 43
#            REAL training epoch 1
# ============================================================

import time
import torch

print("G0-COLD SEED-43 — REAL TRAINING EPOCH 1")
print("=" * 75)

start_time = time.time()


# ------------------------------------------------------------
# Train epoch 1
# ------------------------------------------------------------

g0_s43_train_loss_epoch1 = train_one_epoch(
    model=g0_seed43_model,
    optimizer=g0_seed43_optimizer,

    edge_index=g0_edge_index,
    edge_type=g0_edge_type,

    train_positive_pairs=warm_train_pairs,
    train_negative_pairs=warm_train_negatives,

    epoch=1,
    model_seed=MODEL_SEED,
    device=device
)

torch.cuda.synchronize()

after_train = time.time()


# ------------------------------------------------------------
# Warm-only validation
# ------------------------------------------------------------

g0_s43_val_loss_epoch1 = validation_loss(
    model=g0_seed43_model,

    edge_index=g0_edge_index,
    edge_type=g0_edge_type,

    val_positive_pairs=warm_val_pairs,
    val_negative_pairs=warm_val_negatives,

    device=device
)

torch.cuda.synchronize()

end_time = time.time()


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert torch.isfinite(
    torch.tensor(
        g0_s43_train_loss_epoch1
    )
)

assert torch.isfinite(
    torch.tensor(
        g0_s43_val_loss_epoch1
    )
)

assert all(
    bool(torch.isfinite(p).all())
    for p in g0_seed43_model.parameters()
)

# Adam state should now exist because one optimizer step occurred.
assert len(
    g0_seed43_optimizer.state
) > 0


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print(
    "Training loss:",
    f"{g0_s43_train_loss_epoch1:.6f}"
)

print(
    "Warm validation loss:",
    f"{g0_s43_val_loss_epoch1:.6f}"
)

print(
    "Training time:",
    f"{after_train - start_time:.2f} sec"
)

print(
    "Validation time:",
    f"{end_time - after_train:.2f} sec"
)

print(
    "Total epoch time:",
    f"{end_time - start_time:.2f} sec"
)


print("\nSTATE")
print("-" * 75)

print(
    "Cold split seed:",
    SPLIT_SEED
)

print(
    "Model seed:",
    MODEL_SEED
)

print(
    "Optimizer state entries:",
    len(g0_seed43_optimizer.state)
)

print(
    "Parameters finite:",
    True
)


print("\nGPU MEMORY")
print("-" * 75)

print(
    "Allocated:",
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)

print(
    "Reserved:",
    f"{torch.cuda.memory_reserved()/1024**3:.2f} GB"
)

free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)

print(
    "Free:",
    f"{free_bytes/1024**3:.2f} GB"
)


print(
    "\nPASS: real G0-cold seed-43 epoch 1 completed."
)

In [ ]:
# ============================================================
# STEP 47 — Continue G0-cold seed-43 training
#            Epoch 2 -> max 500
# ============================================================

import copy
import time
import math
import torch

print("G0-COLD SEED-43 — FULL TRAINING")
print("=" * 75)

print("Cold split seed:", SPLIT_SEED)
print("Model seed:", MODEL_SEED)
print("Continuing from completed epoch 1.")
print("Maximum epoch:", MAX_EPOCHS)
print("Early stopping patience:", PATIENCE)
print("Validation: warm-only")
print()


# ------------------------------------------------------------
# Initialize early stopping from REAL epoch 1
# ------------------------------------------------------------

g0_s43_best_val_loss = (
    g0_s43_val_loss_epoch1
)

g0_s43_best_epoch = 1

g0_s43_best_model_state = copy.deepcopy(
    g0_seed43_model.state_dict()
)

g0_s43_patience_counter = 0


g0_s43_history = [
    {
        "epoch": 1,
        "train_loss":
            g0_s43_train_loss_epoch1,
        "val_loss":
            g0_s43_val_loss_epoch1,
    }
]


training_start = time.time()


# ------------------------------------------------------------
# Epoch 2 -> 500
# ------------------------------------------------------------

for epoch in range(
    2,
    MAX_EPOCHS + 1
):

    epoch_start = time.time()


    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    train_loss = train_one_epoch(
        model=g0_seed43_model,
        optimizer=g0_seed43_optimizer,

        edge_index=g0_edge_index,
        edge_type=g0_edge_type,

        train_positive_pairs=
            warm_train_pairs,

        train_negative_pairs=
            warm_train_negatives,

        epoch=epoch,
        model_seed=MODEL_SEED,
        device=device
    )


    # --------------------------------------------------------
    # Warm-only validation
    # --------------------------------------------------------

    val_loss = validation_loss(
        model=g0_seed43_model,

        edge_index=g0_edge_index,
        edge_type=g0_edge_type,

        val_positive_pairs=
            warm_val_pairs,

        val_negative_pairs=
            warm_val_negatives,

        device=device
    )


    torch.cuda.synchronize()

    epoch_seconds = (
        time.time()
        - epoch_start
    )


    # --------------------------------------------------------
    # Numerical safety
    # --------------------------------------------------------

    if not math.isfinite(train_loss):
        raise RuntimeError(
            f"Non-finite train loss "
            f"at epoch {epoch}: "
            f"{train_loss}"
        )

    if not math.isfinite(val_loss):
        raise RuntimeError(
            f"Non-finite validation loss "
            f"at epoch {epoch}: "
            f"{val_loss}"
        )


    g0_s43_history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
        }
    )


    # --------------------------------------------------------
    # Early stopping
    # Exact same strict improvement rule as previous runs
    # --------------------------------------------------------

    improved = (
        val_loss
        < g0_s43_best_val_loss
    )


    if improved:

        g0_s43_best_val_loss = (
            val_loss
        )

        g0_s43_best_epoch = epoch

        g0_s43_best_model_state = (
            copy.deepcopy(
                g0_seed43_model.state_dict()
            )
        )

        g0_s43_patience_counter = 0

    else:

        g0_s43_patience_counter += 1


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (
        epoch <= 10
        or epoch % 10 == 0
        or improved
    ):

        marker = (
            "*"
            if improved
            else ""
        )

        print(
            f"Epoch {epoch:03d} | "
            f"train={train_loss:.6f} | "
            f"val={val_loss:.6f} | "
            f"best={g0_s43_best_val_loss:.6f} "
            f"(epoch {g0_s43_best_epoch}) | "
            f"patience="
            f"{g0_s43_patience_counter}/{PATIENCE} | "
            f"{epoch_seconds:.2f}s "
            f"{marker}"
        )


    # --------------------------------------------------------
    # Early stopping
    # --------------------------------------------------------

    if (
        g0_s43_patience_counter
        >= PATIENCE
    ):

        print()

        print(
            "EARLY STOPPING "
            f"at epoch {epoch}."
        )

        break


# ------------------------------------------------------------
# Restore best model
# ------------------------------------------------------------

g0_seed43_model.load_state_dict(
    g0_s43_best_model_state
)

g0_seed43_model.eval()


training_seconds = (
    time.time()
    - training_start
)

g0_s43_last_epoch = (
    g0_s43_history[-1]["epoch"]
)


# ------------------------------------------------------------
# Final verification
# ------------------------------------------------------------

assert all(
    bool(torch.isfinite(p).all())
    for p in g0_seed43_model.parameters()
)

assert (
    len(g0_s43_history)
    == g0_s43_last_epoch
)


print("\n" + "=" * 75)

print(
    "G0-COLD SEED-43 TRAINING COMPLETE"
)

print("=" * 75)

print(
    "Cold split seed:",
    SPLIT_SEED
)

print(
    "Model seed:",
    MODEL_SEED
)

print(
    "Last epoch executed:",
    g0_s43_last_epoch
)

print(
    "Best epoch:",
    g0_s43_best_epoch
)

print(
    "Best warm validation loss:",
    f"{g0_s43_best_val_loss:.6f}"
)

print(
    "Total epochs represented:",
    len(g0_s43_history)
)

print(
    "Continuation training time:",
    f"{training_seconds:.2f} sec"
)

print(
    "Best model restored:",
    True
)

print(
    "All restored parameters finite:",
    True
)


print("\nGPU MEMORY")
print("-" * 75)

print(
    "Allocated:",
    f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
)

print(
    "Reserved:",
    f"{torch.cuda.memory_reserved()/1024**3:.2f} GB"
)

free_bytes, total_bytes = (
    torch.cuda.mem_get_info()
)

print(
    "Free:",
    f"{free_bytes/1024**3:.2f} GB"
)


print(
    "\nPASS: G0-cold seed-43 training "
    "completed and best model restored."
)

In [ ]:
# ============================================================
# STEP 48 — Save + evaluate G0-cold model seed 43
# ============================================================

from pathlib import Path
import json
import torch
import time


SPLIT_SEED = 42
MODEL_SEED = 43


# ------------------------------------------------------------
# Directories
# ------------------------------------------------------------

G0_S43_CHECKPOINT_DIR = (
    PROJECT_DIR
    / "checkpoints"
    / "cold_start"
    / "split_seed_42"
    / "G0"
    / "model_seed_43"
)

G0_S43_RESULT_DIR = (
    PROJECT_DIR
    / "results"
    / "cold_start"
    / "split_seed_42"
    / "G0"
    / "model_seed_43"
)

G0_S43_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

G0_S43_RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Save best model
# ------------------------------------------------------------

checkpoint_path = (
    G0_S43_CHECKPOINT_DIR
    / "best_model.pt"
)

torch.save(
    {
        "model_state_dict":
            g0_seed43_model.state_dict(),

        "experiment":
            "ddi_edge_cold_start",

        "graph_variant":
            "G0",

        "split_seed":
            SPLIT_SEED,

        "model_seed":
            MODEL_SEED,

        "best_epoch":
            g0_s43_best_epoch,

        "best_val_loss":
            g0_s43_best_val_loss,

        "num_nodes":
            NUM_NODES,

        "num_relations":
            NUM_RELATIONS,

        "embedding_dim":
            EMBEDDING_DIM,

        "hidden_dim":
            HIDDEN_DIM,

        "dropout":
            DROPOUT,

        "learning_rate":
            1e-3,

        "weight_decay":
            1e-5,

        "max_epochs":
            MAX_EPOCHS,

        "patience":
            PATIENCE,

        "train_sample_size":
            TRAIN_SAMPLE_SIZE,

        "validation_protocol":
            "warm-only",

        "cold_start_scope":
            "DDI-edge cold-start",
    },
    checkpoint_path
)


# ------------------------------------------------------------
# Save training history
# ------------------------------------------------------------

history_path = (
    G0_S43_RESULT_DIR
    / "training_history.json"
)

with open(history_path, "w") as f:
    json.dump(
        g0_s43_history,
        f,
        indent=2
    )


# ------------------------------------------------------------
# Evaluate EXACT same primary queries
# ------------------------------------------------------------

print("G0 SEED-43 PRIMARY EVALUATION")
print("=" * 70)

eval_start = time.time()

g0_s43_metrics, g0_s43_ranks = (
    evaluate_filtered_ranking(
        model=g0_seed43_model,

        edge_index=g0_edge_index,
        edge_type=g0_edge_type,

        evaluation_pairs=
            primary_cold_pairs,

        drug_node_ids=
            drug_node_ids,

        known_positive_mask=
            known_positive_mask,

        device=device,

        batch_size=128
    )
)

torch.cuda.synchronize()

eval_seconds = (
    time.time() - eval_start
)


assert (
    g0_s43_metrics["num_queries"]
    == 17_176
)

assert g0_s43_ranks.numel() == 17_176

assert bool(
    (g0_s43_ranks >= 1).all()
)

assert bool(
    (g0_s43_ranks <= 4_278).all()
)


# ------------------------------------------------------------
# Save evaluation metrics
# ------------------------------------------------------------

evaluation_result = {
    "experiment":
        "ddi_edge_cold_start",

    "graph_variant":
        "G0",

    "split_seed":
        SPLIT_SEED,

    "model_seed":
        MODEL_SEED,

    "evaluation":
        "primary_one_sided_cold",

    "direction":
        "cold_query_to_warm_target",

    "num_queries":
        int(
            g0_s43_metrics[
                "num_queries"
            ]
        ),

    "candidate_drugs":
        int(drug_node_ids.numel()),

    "filtering":
        "all_known_positive_ddis_plus_self",

    "mean_reciprocal_rank":
        float(
            g0_s43_metrics[
                "mean_reciprocal_rank"
            ]
        ),

    "hits_at_1":
        float(
            g0_s43_metrics[
                "hits_at_1"
            ]
        ),

    "hits_at_5":
        float(
            g0_s43_metrics[
                "hits_at_5"
            ]
        ),

    "hits_at_10":
        float(
            g0_s43_metrics[
                "hits_at_10"
            ]
        ),

    "mean_rank":
        float(
            g0_s43_metrics[
                "mean_rank"
            ]
        ),

    "median_rank":
        float(
            g0_s43_metrics[
                "median_rank"
            ]
        ),

    "min_rank":
        int(g0_s43_ranks.min()),

    "max_rank":
        int(g0_s43_ranks.max()),

    "best_epoch":
        int(g0_s43_best_epoch),

    "best_warm_val_loss":
        float(g0_s43_best_val_loss),
}


evaluation_path = (
    G0_S43_RESULT_DIR
    / "primary_cold_evaluation.json"
)

with open(evaluation_path, "w") as f:
    json.dump(
        evaluation_result,
        f,
        indent=2
    )


# ------------------------------------------------------------
# Save ranks
# ------------------------------------------------------------

ranks_path = (
    G0_S43_RESULT_DIR
    / "primary_cold_ranks.pt"
)

torch.save(
    {
        "ranks":
            g0_s43_ranks.cpu(),

        "evaluation_pairs":
            primary_cold_pairs.cpu(),

        "direction":
            "cold_query_to_warm_target",

        "split_seed":
            SPLIT_SEED,

        "model_seed":
            MODEL_SEED,

        "graph_variant":
            "G0",
    },
    ranks_path
)


# ------------------------------------------------------------
# Reload verification
# ------------------------------------------------------------

saved_checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu"
)

with open(evaluation_path, "r") as f:
    saved_eval = json.load(f)

saved_ranks = torch.load(
    ranks_path,
    map_location="cpu"
)


assert saved_checkpoint["model_seed"] == 43
assert saved_checkpoint["graph_variant"] == "G0"

assert saved_eval["num_queries"] == 17_176
assert saved_eval["model_seed"] == 43

assert torch.equal(
    saved_ranks["evaluation_pairs"],
    primary_cold_pairs.cpu()
)

assert torch.equal(
    saved_ranks["ranks"],
    g0_s43_ranks.cpu()
)


# ------------------------------------------------------------
# Final output
# ------------------------------------------------------------

print("\nG0-COLD SEED-43 RESULT")
print("-" * 70)

print(
    "Best epoch:",
    g0_s43_best_epoch
)

print(
    "Best warm val loss:",
    f"{g0_s43_best_val_loss:.6f}"
)

print(
    "MRR:",
    f"{saved_eval['mean_reciprocal_rank']:.6f}"
)

print(
    "Hits@1:",
    f"{saved_eval['hits_at_1']:.6f}"
)

print(
    "Hits@5:",
    f"{saved_eval['hits_at_5']:.6f}"
)

print(
    "Hits@10:",
    f"{saved_eval['hits_at_10']:.6f}"
)

print(
    "Mean rank:",
    f"{saved_eval['mean_rank']:.2f}"
)

print(
    "Median rank:",
    f"{saved_eval['median_rank']:.2f}"
)

print(
    "Evaluation time:",
    f"{eval_seconds:.2f} sec"
)

print("\nSaved:")
print(checkpoint_path)
print(evaluation_path)
print(ranks_path)

print(
    "\nPASS: G0-cold seed-43 model, "
    "history, metrics and ranks safely stored."
)

In [ ]:
# ============================================================
# STEP 49 — AUTOMATED REMAINING COLD-START EXPERIMENTS
#
# Runs sequentially:
#   1. G3 model seed 43
#   2. G0 model seed 44
#   3. G3 model seed 44
#
# Fixed:
#   cold split seed = 42
#   cold drugs = 196
#   train/val negatives
#   primary evaluation queries = 17,176
#
# Each run:
#   fresh model
#   fresh optimizer
#   train <= 500 epochs
#   warm-only validation
#   restore best model
#   save checkpoint/history
#   primary cold->warm evaluation
#   save metrics/ranks
#   verify saved artifacts
#   release GPU memory
# ============================================================

from pathlib import Path
import copy
import gc
import json
import math
import random
import time

import numpy as np
import torch


SPLIT_SEED = 42

REMAINING_RUNS = [
    ("G3", 43),
    ("G0", 44),
    ("G3", 44),
]


# ============================================================
# 1. Frozen-data safety checks
# ============================================================

print("VERIFYING FROZEN COLD-START EXPERIMENT")
print("=" * 78)

assert cold_drug_ids.numel() == 196

assert warm_train_pairs.shape == (
    2, 927_782
)

assert warm_train_negatives.shape == (
    2, 927_782
)

assert warm_val_pairs.shape == (
    2, 115_889
)

assert warm_val_negatives.shape == (
    2, 115_889
)

assert primary_cold_pairs.shape == (
    2, 17_176
)

assert drug_node_ids.numel() == 4_278


# Cold drugs must receive no supervised DDI training
cold_cpu = cold_drug_ids.cpu()

assert not bool(
    torch.isin(
        warm_train_pairs.cpu(),
        cold_cpu
    ).any()
)

assert not bool(
    torch.isin(
        warm_train_negatives.cpu(),
        cold_cpu
    ).any()
)


print("Cold split seed:", SPLIT_SEED)
print("Cold drugs:", f"{cold_drug_ids.numel():,}")
print(
    "Warm train positives:",
    f"{warm_train_pairs.shape[1]:,}"
)
print(
    "Warm train negatives:",
    f"{warm_train_negatives.shape[1]:,}"
)
print(
    "Warm validation positives:",
    f"{warm_val_pairs.shape[1]:,}"
)
print(
    "Warm validation negatives:",
    f"{warm_val_negatives.shape[1]:,}"
)
print(
    "Primary cold queries:",
    f"{primary_cold_pairs.shape[1]:,}"
)

print("\nPASS: frozen data verified.")


# ============================================================
# 2. Helper: deterministic initialization
# ============================================================

def set_experiment_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ============================================================
# 3. Helper: get frozen graph
# ============================================================

def get_cold_graph(graph_variant):

    if graph_variant == "G0":

        graph = g0_cold

        expected_total = 1_855_564
        expected_ddi = 1_855_564
        expected_bio = 0

    elif graph_variant == "G3":

        graph = g3_cold

        expected_total = 1_992_132
        expected_ddi = 1_855_564
        expected_bio = 136_568

    else:

        raise ValueError(
            f"Unsupported graph: {graph_variant}"
        )


    edge_index = (
        graph["edge_index"]
        .to(device)
    )

    edge_type = (
        graph["edge_type"]
        .to(device)
    )


    ddi_count = int(
        (edge_type == 0)
        .sum()
        .item()
    )

    bio_count = int(
        (edge_type != 0)
        .sum()
        .item()
    )


    assert edge_index.shape[1] == expected_total
    assert ddi_count == expected_ddi
    assert bio_count == expected_bio


    # No cold drug may participate in DDI message passing.
    ddi_edges = edge_index[
        :,
        edge_type == 0
    ]

    assert not bool(
        torch.isin(
            ddi_edges,
            cold_drug_ids.to(device)
        ).any()
    )


    return (
        edge_index,
        edge_type,
        ddi_count,
        bio_count
    )


# ============================================================
# 4. Helper: train one complete run
# ============================================================

def run_cold_experiment(
    graph_variant,
    model_seed
):

    print("\n\n")
    print("#" * 78)
    print(
        f"STARTING {graph_variant}-COLD "
        f"MODEL SEED {model_seed}"
    )
    print("#" * 78)

    print(
        "Cold split seed:",
        SPLIT_SEED
    )

    print(
        "Model seed:",
        model_seed
    )


    # --------------------------------------------------------
    # Clear only unused Python/CUDA objects
    # --------------------------------------------------------

    gc.collect()
    torch.cuda.empty_cache()


    # --------------------------------------------------------
    # Frozen graph
    # --------------------------------------------------------

    (
        edge_index,
        edge_type,
        ddi_count,
        bio_count
    ) = get_cold_graph(
        graph_variant
    )


    print("\nGRAPH")
    print("-" * 78)

    print(
        "Directed DDI edges:",
        f"{ddi_count:,}"
    )

    print(
        "Directed biomedical edges:",
        f"{bio_count:,}"
    )

    print(
        "Total directed edges:",
        f"{edge_index.shape[1]:,}"
    )


    # --------------------------------------------------------
    # Fresh deterministic model
    # --------------------------------------------------------

    set_experiment_seed(
        model_seed
    )


    model = RGCNDDIModel(
        num_nodes=NUM_NODES,
        num_relations=NUM_RELATIONS,
        embedding_dim=EMBEDDING_DIM,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT
    ).to(device)


    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-5
    )


    num_parameters = sum(
        p.numel()
        for p in model.parameters()
    )


    assert num_parameters == 2_200_704
    assert len(optimizer.state) == 0


    print("\nMODEL")
    print("-" * 78)

    print(
        "Fresh model parameters:",
        f"{num_parameters:,}"
    )

    print(
        "Fresh optimizer state:",
        len(optimizer.state)
    )


    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    best_val_loss = float("inf")
    best_epoch = None
    best_model_state = None

    patience_counter = 0

    history = []

    training_start = time.time()


    print("\nTRAINING")
    print("-" * 78)


    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):

        epoch_start = time.time()


        train_loss = train_one_epoch(
            model=model,
            optimizer=optimizer,

            edge_index=edge_index,
            edge_type=edge_type,

            train_positive_pairs=
                warm_train_pairs,

            train_negative_pairs=
                warm_train_negatives,

            epoch=epoch,
            model_seed=model_seed,
            device=device
        )


        val_loss = validation_loss(
            model=model,

            edge_index=edge_index,
            edge_type=edge_type,

            val_positive_pairs=
                warm_val_pairs,

            val_negative_pairs=
                warm_val_negatives,

            device=device
        )


        torch.cuda.synchronize()


        if not math.isfinite(train_loss):
            raise RuntimeError(
                f"{graph_variant} seed "
                f"{model_seed}: non-finite "
                f"training loss at epoch {epoch}"
            )


        if not math.isfinite(val_loss):
            raise RuntimeError(
                f"{graph_variant} seed "
                f"{model_seed}: non-finite "
                f"validation loss at epoch {epoch}"
            )


        history.append(
            {
                "epoch":
                    int(epoch),

                "train_loss":
                    float(train_loss),

                "val_loss":
                    float(val_loss),
            }
        )


        improved = (
            val_loss
            < best_val_loss
        )


        if improved:

            best_val_loss = (
                float(val_loss)
            )

            best_epoch = (
                int(epoch)
            )

            best_model_state = (
                copy.deepcopy(
                    model.state_dict()
                )
            )

            patience_counter = 0

        else:

            patience_counter += 1


        epoch_seconds = (
            time.time()
            - epoch_start
        )


        if (
            epoch == 1
            or epoch % 25 == 0
            or epoch == MAX_EPOCHS
        ):

            print(
                f"{graph_variant} "
                f"seed {model_seed} | "
                f"epoch {epoch:03d} | "
                f"train={train_loss:.6f} | "
                f"val={val_loss:.6f} | "
                f"best={best_val_loss:.6f} "
                f"@ {best_epoch} | "
                f"patience="
                f"{patience_counter}/{PATIENCE} | "
                f"{epoch_seconds:.2f}s"
            )


        if (
            patience_counter
            >= PATIENCE
        ):

            print(
                f"Early stopping at "
                f"epoch {epoch}."
            )

            break


    # --------------------------------------------------------
    # Restore best model
    # --------------------------------------------------------

    assert best_model_state is not None
    assert best_epoch is not None


    model.load_state_dict(
        best_model_state
    )

    model.eval()


    last_epoch = history[-1]["epoch"]

    training_seconds = (
        time.time()
        - training_start
    )


    assert all(
        bool(torch.isfinite(p).all())
        for p in model.parameters()
    )


    print("\nTRAINING COMPLETE")
    print("-" * 78)

    print(
        "Last epoch:",
        last_epoch
    )

    print(
        "Best epoch:",
        best_epoch
    )

    print(
        "Best warm val loss:",
        f"{best_val_loss:.6f}"
    )

    print(
        "Training time:",
        f"{training_seconds:.2f} sec"
    )


    # --------------------------------------------------------
    # Paths
    # --------------------------------------------------------

    checkpoint_dir = (
        PROJECT_DIR
        / "checkpoints"
        / "cold_start"
        / f"split_seed_{SPLIT_SEED}"
        / graph_variant
        / f"model_seed_{model_seed}"
    )

    result_dir = (
        PROJECT_DIR
        / "results"
        / "cold_start"
        / f"split_seed_{SPLIT_SEED}"
        / graph_variant
        / f"model_seed_{model_seed}"
    )


    checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    result_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    checkpoint_path = (
        checkpoint_dir
        / "best_model.pt"
    )

    history_path = (
        result_dir
        / "training_history.json"
    )

    summary_path = (
        result_dir
        / "training_summary.json"
    )


    # --------------------------------------------------------
    # Save checkpoint BEFORE evaluation
    # --------------------------------------------------------

    torch.save(
        {
            "model_state_dict":
                model.state_dict(),

            "experiment":
                "ddi_edge_cold_start",

            "graph_variant":
                graph_variant,

            "split_seed":
                SPLIT_SEED,

            "model_seed":
                model_seed,

            "best_epoch":
                best_epoch,

            "best_val_loss":
                best_val_loss,

            "num_nodes":
                NUM_NODES,

            "num_relations":
                NUM_RELATIONS,

            "embedding_dim":
                EMBEDDING_DIM,

            "hidden_dim":
                HIDDEN_DIM,

            "dropout":
                DROPOUT,

            "learning_rate":
                1e-3,

            "weight_decay":
                1e-5,

            "max_epochs":
                MAX_EPOCHS,

            "patience":
                PATIENCE,

            "train_sample_size":
                TRAIN_SAMPLE_SIZE,

            "validation_protocol":
                "warm-only",

            "cold_start_scope":
                "DDI-edge cold-start",
        },
        checkpoint_path
    )


    with open(
        history_path,
        "w"
    ) as f:

        json.dump(
            history,
            f,
            indent=2
        )


    training_summary = {
        "experiment":
            "ddi_edge_cold_start",

        "graph_variant":
            graph_variant,

        "split_seed":
            SPLIT_SEED,

        "model_seed":
            model_seed,

        "best_epoch":
            best_epoch,

        "best_val_loss":
            best_val_loss,

        "last_epoch":
            last_epoch,

        "training_seconds":
            training_seconds,

        "warm_train_positive_count":
            int(
                warm_train_pairs.shape[1]
            ),

        "warm_train_negative_count":
            int(
                warm_train_negatives.shape[1]
            ),

        "warm_validation_positive_count":
            int(
                warm_val_pairs.shape[1]
            ),

        "warm_validation_negative_count":
            int(
                warm_val_negatives.shape[1]
            ),

        "cold_drug_count":
            int(cold_drug_ids.numel()),

        "directed_ddi_edges":
            ddi_count,

        "directed_biomedical_edges":
            bio_count,

        "validation_protocol":
            "warm-only",
    }


    with open(
        summary_path,
        "w"
    ) as f:

        json.dump(
            training_summary,
            f,
            indent=2
        )


    print(
        "\nCheckpoint saved before evaluation."
    )


    # --------------------------------------------------------
    # Primary cold -> warm evaluation
    # --------------------------------------------------------

    print("\nPRIMARY EVALUATION")
    print("-" * 78)

    eval_start = time.time()


    metrics, ranks = (
        evaluate_filtered_ranking(
            model=model,

            edge_index=edge_index,
            edge_type=edge_type,

            evaluation_pairs=
                primary_cold_pairs,

            drug_node_ids=
                drug_node_ids,

            known_positive_mask=
                known_positive_mask,

            device=device,

            batch_size=128
        )
    )


    torch.cuda.synchronize()


    eval_seconds = (
        time.time()
        - eval_start
    )


    assert (
        metrics["num_queries"]
        == 17_176
    )

    assert ranks.numel() == 17_176

    assert bool(
        (ranks >= 1).all()
    )

    assert bool(
        (ranks <= 4_278).all()
    )


    # --------------------------------------------------------
    # Save evaluation
    # --------------------------------------------------------

    evaluation_result = {
        "experiment":
            "ddi_edge_cold_start",

        "graph_variant":
            graph_variant,

        "split_seed":
            SPLIT_SEED,

        "model_seed":
            model_seed,

        "evaluation":
            "primary_one_sided_cold",

        "direction":
            "cold_query_to_warm_target",

        "num_queries":
            int(metrics["num_queries"]),

        "candidate_drugs":
            int(drug_node_ids.numel()),

        "filtering":
            "all_known_positive_ddis_plus_self",

        "mean_reciprocal_rank":
            float(
                metrics[
                    "mean_reciprocal_rank"
                ]
            ),

        "hits_at_1":
            float(
                metrics["hits_at_1"]
            ),

        "hits_at_5":
            float(
                metrics["hits_at_5"]
            ),

        "hits_at_10":
            float(
                metrics["hits_at_10"]
            ),

        "mean_rank":
            float(
                metrics["mean_rank"]
            ),

        "median_rank":
            float(
                metrics["median_rank"]
            ),

        "min_rank":
            int(ranks.min()),

        "max_rank":
            int(ranks.max()),

        "best_epoch":
            best_epoch,

        "best_warm_val_loss":
            best_val_loss,

        "evaluation_seconds":
            eval_seconds,
    }


    evaluation_path = (
        result_dir
        / "primary_cold_evaluation.json"
    )

    ranks_path = (
        result_dir
        / "primary_cold_ranks.pt"
    )


    with open(
        evaluation_path,
        "w"
    ) as f:

        json.dump(
            evaluation_result,
            f,
            indent=2
        )


    torch.save(
        {
            "ranks":
                ranks.cpu(),

            "evaluation_pairs":
                primary_cold_pairs.cpu(),

            "direction":
                "cold_query_to_warm_target",

            "split_seed":
                SPLIT_SEED,

            "model_seed":
                model_seed,

            "graph_variant":
                graph_variant,
        },
        ranks_path
    )


    # --------------------------------------------------------
    # Reload verification
    # --------------------------------------------------------

    saved_checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu"
    )

    with open(
        evaluation_path,
        "r"
    ) as f:

        saved_eval = json.load(f)


    saved_rank_data = torch.load(
        ranks_path,
        map_location="cpu"
    )


    assert (
        saved_checkpoint["graph_variant"]
        == graph_variant
    )

    assert (
        saved_checkpoint["model_seed"]
        == model_seed
    )

    assert (
        saved_checkpoint["split_seed"]
        == SPLIT_SEED
    )

    assert (
        saved_eval["num_queries"]
        == 17_176
    )

    assert torch.equal(
        saved_rank_data[
            "evaluation_pairs"
        ],
        primary_cold_pairs.cpu()
    )

    assert torch.equal(
        saved_rank_data["ranks"],
        ranks.cpu()
    )


    # --------------------------------------------------------
    # Run result
    # --------------------------------------------------------

    print(
        "MRR:",
        f"{evaluation_result['mean_reciprocal_rank']:.6f}"
    )

    print(
        "Hits@1:",
        f"{evaluation_result['hits_at_1']:.6f}"
    )

    print(
        "Hits@5:",
        f"{evaluation_result['hits_at_5']:.6f}"
    )

    print(
        "Hits@10:",
        f"{evaluation_result['hits_at_10']:.6f}"
    )

    print(
        "Mean rank:",
        f"{evaluation_result['mean_rank']:.2f}"
    )

    print(
        "Median rank:",
        f"{evaluation_result['median_rank']:.2f}"
    )

    print(
        "Evaluation time:",
        f"{eval_seconds:.2f} sec"
    )


    print(
        f"\nPASS: {graph_variant}-cold "
        f"seed-{model_seed} safely stored."
    )


    result_copy = dict(
        evaluation_result
    )


    # --------------------------------------------------------
    # Free this run before next model
    # --------------------------------------------------------

    del ranks
    del model
    del optimizer
    del best_model_state
    del edge_index
    del edge_type

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


    free_bytes, _ = (
        torch.cuda.mem_get_info()
    )

    print(
        "GPU free after cleanup:",
        f"{free_bytes/1024**3:.2f} GB"
    )


    return result_copy


# ============================================================
# 5. Run remaining experiments sequentially
# ============================================================

remaining_results = []


for graph_variant, model_seed in REMAINING_RUNS:

    result = run_cold_experiment(
        graph_variant=graph_variant,
        model_seed=model_seed
    )

    remaining_results.append(
        result
    )


# ============================================================
# 6. Final summary of the THREE new runs
# ============================================================

print("\n\n")
print("=" * 78)
print("ALL REMAINING RUNS COMPLETE")
print("=" * 78)

for result in remaining_results:

    print(
        f"{result['graph_variant']} "
        f"seed {result['model_seed']} | "
        f"MRR="
        f"{result['mean_reciprocal_rank']:.6f} | "
        f"H@1="
        f"{result['hits_at_1']:.6f} | "
        f"H@5="
        f"{result['hits_at_5']:.6f} | "
        f"H@10="
        f"{result['hits_at_10']:.6f}"
    )


print(
    "\nPASS: G3-43, G0-44 and G3-44 "
    "completed sequentially."
)

## 13. Historical Three-Seed Aggregation

This section contains the original aggregation logic.

**Important reproducibility note:** the original G0 rank vectors were later reproduced exactly from the preserved checkpoints and frozen graph. The original G3 rank vectors were not reproduced from the preserved checkpoints and frozen G3 graph. Therefore the original G3 aggregate values from this historical section are not used as final results.


In [ ]:
# ============================================================
# STEP 50 — FINAL 3-SEED COLD-START SUMMARY
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd


SPLIT_SEED = 42
MODEL_SEEDS = [42, 43, 44]
GRAPH_VARIANTS = ["G0", "G3"]

BASE_RESULT_DIR = (
    PROJECT_DIR
    / "results"
    / "cold_start"
    / f"split_seed_{SPLIT_SEED}"
)


# ------------------------------------------------------------
# Load all six saved evaluation files
# ------------------------------------------------------------

rows = []

for graph in GRAPH_VARIANTS:

    for seed in MODEL_SEEDS:

        path = (
            BASE_RESULT_DIR
            / graph
            / f"model_seed_{seed}"
            / "primary_cold_evaluation.json"
        )

        assert path.exists(), (
            f"Missing result: {path}"
        )

        with open(path, "r") as f:
            result = json.load(f)

        assert result["graph_variant"] == graph
        assert result["model_seed"] == seed
        assert result["split_seed"] == SPLIT_SEED
        assert result["num_queries"] == 17_176
        assert (
            result["direction"]
            == "cold_query_to_warm_target"
        )

        rows.append(
            {
                "graph":
                    graph,

                "seed":
                    seed,

                "mrr":
                    result[
                        "mean_reciprocal_rank"
                    ],

                "hits_at_1":
                    result["hits_at_1"],

                "hits_at_5":
                    result["hits_at_5"],

                "hits_at_10":
                    result["hits_at_10"],

                "mean_rank":
                    result["mean_rank"],

                "median_rank":
                    result["median_rank"],

                "best_epoch":
    result.get("best_epoch", None),

"best_warm_val_loss":
    result.get(
        "best_warm_val_loss",
        None
    ),
            }
        )


results_df = pd.DataFrame(rows)


# ------------------------------------------------------------
# Per-seed table
# ------------------------------------------------------------

print("PER-SEED COLD-START RESULTS")
print("=" * 90)

display(
    results_df.sort_values(
        ["graph", "seed"]
    ).reset_index(drop=True)
)


# ------------------------------------------------------------
# Mean ± sample SD
# ------------------------------------------------------------

metric_columns = [
    "mrr",
    "hits_at_1",
    "hits_at_5",
    "hits_at_10",
    "mean_rank",
    "median_rank",
]


summary_rows = []

for graph in GRAPH_VARIANTS:

    subset = (
        results_df[
            results_df["graph"] == graph
        ]
    )

    row = {
        "graph": graph
    }

    for metric in metric_columns:

        values = (
            subset[metric]
            .astype(float)
            .to_numpy()
        )

        row[f"{metric}_mean"] = (
            values.mean()
        )

        row[f"{metric}_sd"] = (
            values.std(ddof=1)
        )

    summary_rows.append(row)


summary_df = pd.DataFrame(
    summary_rows
)


print("\nMEAN ± SAMPLE SD")
print("=" * 90)

for _, row in summary_df.iterrows():

    print(f"\n{row['graph']}")

    print(
        "MRR:",
        f"{row['mrr_mean']:.6f} "
        f"± {row['mrr_sd']:.6f}"
    )

    print(
        "Hits@1:",
        f"{row['hits_at_1_mean']:.6f} "
        f"± {row['hits_at_1_sd']:.6f}"
    )

    print(
        "Hits@5:",
        f"{row['hits_at_5_mean']:.6f} "
        f"± {row['hits_at_5_sd']:.6f}"
    )

    print(
        "Hits@10:",
        f"{row['hits_at_10_mean']:.6f} "
        f"± {row['hits_at_10_sd']:.6f}"
    )

    print(
        "Mean rank:",
        f"{row['mean_rank_mean']:.2f} "
        f"± {row['mean_rank_sd']:.2f}"
    )

    print(
        "Median rank:",
        f"{row['median_rank_mean']:.2f} "
        f"± {row['median_rank_sd']:.2f}"
    )


# ------------------------------------------------------------
# Paired G3 - G0 differences
# ------------------------------------------------------------

g0 = (
    results_df[
        results_df["graph"] == "G0"
    ]
    .set_index("seed")
    .sort_index()
)

g3 = (
    results_df[
        results_df["graph"] == "G3"
    ]
    .set_index("seed")
    .sort_index()
)


assert list(g0.index) == MODEL_SEEDS
assert list(g3.index) == MODEL_SEEDS


delta_rows = []

for seed in MODEL_SEEDS:

    delta_rows.append(
        {
            "seed":
                seed,

            "delta_mrr":
                g3.loc[seed, "mrr"]
                - g0.loc[seed, "mrr"],

            "delta_hits_at_1":
                g3.loc[seed, "hits_at_1"]
                - g0.loc[seed, "hits_at_1"],

            "delta_hits_at_5":
                g3.loc[seed, "hits_at_5"]
                - g0.loc[seed, "hits_at_5"],

            "delta_hits_at_10":
                g3.loc[seed, "hits_at_10"]
                - g0.loc[seed, "hits_at_10"],

            # Positive rank delta = worse
            "delta_mean_rank":
                g3.loc[seed, "mean_rank"]
                - g0.loc[seed, "mean_rank"],

            "delta_median_rank":
                g3.loc[seed, "median_rank"]
                - g0.loc[seed, "median_rank"],
        }
    )


delta_df = pd.DataFrame(
    delta_rows
)


print("\nPAIRED G3 - G0 DELTAS")
print("=" * 90)

display(delta_df)


# ------------------------------------------------------------
# Delta summary
# ------------------------------------------------------------

print("\nPAIRED DELTA SUMMARY")
print("=" * 90)

for metric in [
    "delta_mrr",
    "delta_hits_at_1",
    "delta_hits_at_5",
    "delta_hits_at_10",
    "delta_mean_rank",
    "delta_median_rank",
]:

    values = (
        delta_df[metric]
        .astype(float)
        .to_numpy()
    )

    print(
        f"{metric}: "
        f"{values.mean():.6f} "
        f"± {values.std(ddof=1):.6f}"
    )


# ------------------------------------------------------------
# Consistency
# ------------------------------------------------------------

mrr_better = int(
    (delta_df["delta_mrr"] > 0).sum()
)

mrr_worse = int(
    (delta_df["delta_mrr"] < 0).sum()
)


print("\nCONSISTENCY")
print("=" * 90)

print(
    "G3 higher MRR than G0:",
    f"{mrr_better}/3 seeds"
)

print(
    "G3 lower MRR than G0:",
    f"{mrr_worse}/3 seeds"
)


# ------------------------------------------------------------
# Relative mean-MRR change
# ------------------------------------------------------------

g0_mean_mrr = float(
    summary_df.loc[
        summary_df["graph"] == "G0",
        "mrr_mean"
    ].iloc[0]
)

g3_mean_mrr = float(
    summary_df.loc[
        summary_df["graph"] == "G3",
        "mrr_mean"
    ].iloc[0]
)

relative_change = (
    (g3_mean_mrr - g0_mean_mrr)
    / g0_mean_mrr
    * 100
)


print("\nMEAN MRR COMPARISON")
print("=" * 90)

print(
    "G0 mean MRR:",
    f"{g0_mean_mrr:.6f}"
)

print(
    "G3 mean MRR:",
    f"{g3_mean_mrr:.6f}"
)

print(
    "Absolute G3-G0:",
    f"{g3_mean_mrr - g0_mean_mrr:.6f}"
)

print(
    "Relative change:",
    f"{relative_change:.2f}%"
)


# ------------------------------------------------------------
# Save final CSVs
# ------------------------------------------------------------

FINAL_DIR = (
    BASE_RESULT_DIR
    / "final"
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


per_seed_path = (
    FINAL_DIR
    / "cold_start_3seed_per_seed.csv"
)

summary_path = (
    FINAL_DIR
    / "cold_start_3seed_summary.csv"
)

delta_path = (
    FINAL_DIR
    / "cold_start_3seed_paired_deltas.csv"
)


results_df.to_csv(
    per_seed_path,
    index=False
)

summary_df.to_csv(
    summary_path,
    index=False
)

delta_df.to_csv(
    delta_path,
    index=False
)


print("\nSAVED")
print("=" * 90)

print(per_seed_path)
print(summary_path)
print(delta_path)

print(
    "\nPASS: final 3-seed cold-start "
    "summary generated."
)

## 14. Historical Thesis Figure Code

This self-contained figure/table code is retained for traceability but does not define the final validated cold-start results.

The failed earlier figure/table attempts were removed from this cleaned copy.


In [ ]:
# ============================================================
# STEP 51 — FINAL THESIS FIGURE + TABLE
# SELF-CONTAINED VERSION
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


print("CREATING FINAL COLD-START FIGURE + TABLE")
print("=" * 75)


# ============================================================
# 1. Paths
# ============================================================

FINAL_DIR = (
    PROJECT_DIR
    / "results"
    / "cold_start"
    / "split_seed_42"
    / "final"
)

FIGURE_DIR = (
    PROJECT_DIR
    / "figures"
    / "cold_start"
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


per_seed_path = (
    FINAL_DIR
    / "cold_start_3seed_per_seed.csv"
)

summary_path = (
    FINAL_DIR
    / "cold_start_3seed_summary.csv"
)


assert per_seed_path.exists(), per_seed_path
assert summary_path.exists(), summary_path


# ============================================================
# 2. Load authoritative final results
# ============================================================

per_seed_df = pd.read_csv(
    per_seed_path
)

summary_df = pd.read_csv(
    summary_path
)


assert len(per_seed_df) == 6
assert len(summary_df) == 2

assert set(
    per_seed_df["graph"]
) == {"G0", "G3"}

assert set(
    per_seed_df["seed"]
) == {42, 43, 44}


print("\nLoaded:")
print(per_seed_path)
print(summary_path)

print(
    "\nPer-seed rows:",
    len(per_seed_df)
)

print(
    "Summary rows:",
    len(summary_df)
)


# ============================================================
# 3. Extract MRR values
# ============================================================

graph_order = [
    "G0",
    "G3"
]

seeds = [
    42,
    43,
    44
]


seed_values = {}
means = []
stds = []


for graph in graph_order:

    subset = (
        per_seed_df[
            per_seed_df["graph"] == graph
        ]
        .sort_values("seed")
    )

    assert (
        subset["seed"].tolist()
        == seeds
    )

    values = (
        subset["mrr"]
        .astype(float)
        .to_numpy()
    )

    seed_values[graph] = values

    means.append(
        values.mean()
    )

    stds.append(
        values.std(ddof=1)
    )


means = np.asarray(means)
stds = np.asarray(stds)


# ============================================================
# 4. Verify final numbers
# ============================================================

assert np.isclose(
    means[0],
    0.140603,
    atol=1e-6
)

assert np.isclose(
    means[1],
    0.062437,
    atol=1e-6
)


print("\nAUTHORITATIVE MRR")
print("-" * 75)

print(
    "G0:",
    f"{means[0]:.6f} ± {stds[0]:.6f}"
)

print(
    "G3:",
    f"{means[1]:.6f} ± {stds[1]:.6f}"
)


# ============================================================
# 5. Create figure
# ============================================================

fig, ax = plt.subplots(
    figsize=(7.2, 5.4)
)


x = np.arange(
    len(graph_order)
)


# ------------------------------------------------------------
# Mean bars + sample SD
# ------------------------------------------------------------

ax.bar(
    x,
    means,
    width=0.55,
    yerr=stds,
    capsize=7,
    alpha=0.72,
    edgecolor="black",
    linewidth=1.0
)


# ------------------------------------------------------------
# Individual model seeds
# ------------------------------------------------------------

seed_offsets = [
    -0.08,
    0.00,
    0.08
]


for graph_index, graph in enumerate(
    graph_order
):

    values = seed_values[graph]

    for offset, seed, value in zip(
        seed_offsets,
        seeds,
        values
    ):

        ax.scatter(
            graph_index + offset,
            value,
            s=55,
            zorder=5,
            edgecolor="black",
            linewidth=0.7
        )

        ax.annotate(
            str(seed),
            (
                graph_index + offset,
                value
            ),
            xytext=(0, 7),
            textcoords="offset points",
            ha="center",
            fontsize=8
        )


# ------------------------------------------------------------
# Mean value labels
# ------------------------------------------------------------

for i, mean_value in enumerate(
    means
):

    ax.text(
        i,
        0.006,
        f"{mean_value:.3f}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )


# ------------------------------------------------------------
# Figure formatting
# ------------------------------------------------------------

ax.set_xticks(x)

ax.set_xticklabels(
    [
        "G0\nDDI only",
        "G3\nDDI + biomedical context"
    ]
)

ax.set_ylabel(
    "Mean Reciprocal Rank (MRR)"
)

ax.set_xlabel(
    "Graph configuration"
)

ax.set_title(
    "Primary DDI-Edge Cold-Start Performance"
)

upper_limit = (
    max(means + stds)
    * 1.30
)

ax.set_ylim(
    0,
    upper_limit
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.30
)

ax.set_axisbelow(True)


fig.text(
    0.5,
    0.01,
    "Bars show mean ± sample SD; points show model seeds 42, 43, and 44.",
    ha="center",
    fontsize=8.5
)


fig.tight_layout(
    rect=[0, 0.04, 1, 1]
)


# ============================================================
# 6. Save figure
# ============================================================

png_path = (
    FIGURE_DIR
    / "cold_start_mrr_3seed.png"
)

svg_path = (
    FIGURE_DIR
    / "cold_start_mrr_3seed.svg"
)


fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    svg_path,
    format="svg",
    bbox_inches="tight"
)


plt.show()


# ============================================================
# 7. Build thesis-ready table
# ============================================================

table_rows = []


for graph in graph_order:

    row = (
        summary_df[
            summary_df["graph"] == graph
        ]
        .iloc[0]
    )


    table_rows.append(
        {
            "Model":
                graph,

            "MRR":
                (
                    f"{row['mrr_mean']:.4f} "
                    f"± {row['mrr_sd']:.4f}"
                ),

            "Hits@1":
                (
                    f"{row['hits_at_1_mean']:.4f} "
                    f"± {row['hits_at_1_sd']:.4f}"
                ),

            "Hits@5":
                (
                    f"{row['hits_at_5_mean']:.4f} "
                    f"± {row['hits_at_5_sd']:.4f}"
                ),

            "Hits@10":
                (
                    f"{row['hits_at_10_mean']:.4f} "
                    f"± {row['hits_at_10_sd']:.4f}"
                ),

            "Mean Rank":
                (
                    f"{row['mean_rank_mean']:.1f} "
                    f"± {row['mean_rank_sd']:.1f}"
                ),

            "Median Rank":
                (
                    f"{row['median_rank_mean']:.1f} "
                    f"± {row['median_rank_sd']:.1f}"
                ),
        }
    )


thesis_table_df = pd.DataFrame(
    table_rows
)


print("\nTHESIS-READY TABLE")
print("=" * 75)

display(
    thesis_table_df
)


# ============================================================
# 8. Save table CSV
# ============================================================

table_csv_path = (
    FINAL_DIR
    / "cold_start_results_thesis.csv"
)


thesis_table_df.to_csv(
    table_csv_path,
    index=False
)


# ============================================================
# 9. Create LaTeX table
# ============================================================

latex_path = (
    FINAL_DIR
    / "cold_start_results_thesis.tex"
)


latex_text = (
    "\\begin{table}[t]\n"
    "\\centering\n"
    "\\caption{Primary one-sided DDI-edge cold-start "
    "performance across three model seeds. Values are "
    "reported as mean $\\pm$ sample standard deviation. "
    "Higher is better for MRR and Hits@K; lower is better "
    "for rank metrics.}\n"
    "\\label{tab:cold_start_results}\n"
    "\\begin{tabular}{lcccccc}\n"
    "\\hline\n"
    "Model & MRR & Hits@1 & Hits@5 & Hits@10 "
    "& Mean Rank & Median Rank \\\\\n"
    "\\hline\n"
)


for _, row in thesis_table_df.iterrows():

    mrr_text = str(
        row["MRR"]
    ).replace(
        "±",
        "$\\pm$"
    )

    h1_text = str(
        row["Hits@1"]
    ).replace(
        "±",
        "$\\pm$"
    )

    h5_text = str(
        row["Hits@5"]
    ).replace(
        "±",
        "$\\pm$"
    )

    h10_text = str(
        row["Hits@10"]
    ).replace(
        "±",
        "$\\pm$"
    )

    mean_rank_text = str(
        row["Mean Rank"]
    ).replace(
        "±",
        "$\\pm$"
    )

    median_rank_text = str(
        row["Median Rank"]
    ).replace(
        "±",
        "$\\pm$"
    )


    latex_text += (
        str(row["Model"])
        + " & "
        + mrr_text
        + " & "
        + h1_text
        + " & "
        + h5_text
        + " & "
        + h10_text
        + " & "
        + mean_rank_text
        + " & "
        + median_rank_text
        + " \\\\\n"
    )


latex_text += (
    "\\hline\n"
    "\\end{tabular}\n"
    "\\end{table}\n"
)


with open(
    latex_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        latex_text
    )


# ============================================================
# 10. Verify generated files
# ============================================================

assert png_path.exists()
assert svg_path.exists()
assert table_csv_path.exists()
assert latex_path.exists()

assert png_path.stat().st_size > 0
assert svg_path.stat().st_size > 0
assert table_csv_path.stat().st_size > 0
assert latex_path.stat().st_size > 0


# ============================================================
# 11. Final output
# ============================================================

print("\nSAVED")
print("=" * 75)

print(
    "PNG:",
    png_path
)

print(
    "SVG:",
    svg_path
)

print(
    "CSV:",
    table_csv_path
)

print(
    "LaTeX:",
    latex_path
)


print(
    "\nPASS: thesis-ready cold-start "
    "figure and table generated successfully."
)

## 15. Secondary Cold↔Cold Evaluation

The secondary evaluation uses the 651 held-out cold–cold test pairs in both directions, producing 1,302 directional ranking queries.

The historical evaluation implementation is retained below. Final reported values are taken from the canonical checkpoint-based reproduction.


In [ ]:
# ============================================================
# STEP 52 — PREPARE SECONDARY TWO-SIDED COLD-START EVALUATION
#
# 651 cold-cold test pairs
# evaluated in BOTH directions
# = 1,302 ranking queries
#
# NO TRAINING
# NO NEW NEGATIVE SAMPLING
# ============================================================

from pathlib import Path
import torch
import json


SPLIT_SEED = 42
MODEL_SEEDS = [42, 43, 44]
GRAPH_VARIANTS = ["G0", "G3"]


print("PREPARING TWO-SIDED COLD-COLD EVALUATION")
print("=" * 78)


# ------------------------------------------------------------
# 1. Verify frozen cold-cold pairs
# ------------------------------------------------------------

assert cold_test_two_sided.shape == (
    2,
    651
)

cold_set_cpu = cold_drug_ids.cpu()

two_sided_cpu = (
    cold_test_two_sided.cpu()
)


# Both endpoints MUST be cold.
src_is_cold = torch.isin(
    two_sided_cpu[0],
    cold_set_cpu
)

dst_is_cold = torch.isin(
    two_sided_cpu[1],
    cold_set_cpu
)


assert bool(src_is_cold.all())
assert bool(dst_is_cold.all())


print(
    "Frozen cold-cold pairs:",
    f"{two_sided_cpu.shape[1]:,}"
)

print(
    "Both endpoints cold:",
    True
)


# ------------------------------------------------------------
# 2. Construct BOTH evaluation directions
# ------------------------------------------------------------

forward_pairs = (
    two_sided_cpu.clone()
)

reverse_pairs = torch.stack(
    [
        two_sided_cpu[1],
        two_sided_cpu[0]
    ],
    dim=0
)


cold_cold_bidirectional_pairs = torch.cat(
    [
        forward_pairs,
        reverse_pairs
    ],
    dim=1
)


assert cold_cold_bidirectional_pairs.shape == (
    2,
    1_302
)


# Every query and every target must be cold.
assert bool(
    torch.isin(
        cold_cold_bidirectional_pairs[0],
        cold_set_cpu
    ).all()
)

assert bool(
    torch.isin(
        cold_cold_bidirectional_pairs[1],
        cold_set_cpu
    ).all()
)


print(
    "Directional ranking queries:",
    f"{cold_cold_bidirectional_pairs.shape[1]:,}"
)

print(
    "Query drugs all cold:",
    True
)

print(
    "Target drugs all cold:",
    True
)


# ------------------------------------------------------------
# 3. Verify all six checkpoints
# ------------------------------------------------------------

checkpoint_records = []


for graph in GRAPH_VARIANTS:

    for model_seed in MODEL_SEEDS:

        checkpoint_path = (
            PROJECT_DIR
            / "checkpoints"
            / "cold_start"
            / f"split_seed_{SPLIT_SEED}"
            / graph
            / f"model_seed_{model_seed}"
            / "best_model.pt"
        )


        assert checkpoint_path.exists(), (
            f"Missing checkpoint:\n"
            f"{checkpoint_path}"
        )


        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu"
        )


        assert (
            checkpoint["graph_variant"]
            == graph
        )

        assert (
            checkpoint["model_seed"]
            == model_seed
        )

        assert (
            checkpoint["split_seed"]
            == SPLIT_SEED
        )


        checkpoint_records.append(
            {
                "graph":
                    graph,

                "model_seed":
                    model_seed,

                "path":
                    str(checkpoint_path),

                "best_epoch":
                    checkpoint.get(
                        "best_epoch",
                        None
                    ),

                "best_val_loss":
                    checkpoint.get(
                        "best_val_loss",
                        None
                    ),
            }
        )


print("\nCHECKPOINTS")
print("-" * 78)

for record in checkpoint_records:

    print(
        f"{record['graph']} "
        f"seed {record['model_seed']} | "
        f"best epoch="
        f"{record['best_epoch']} | "
        f"val="
        f"{record['best_val_loss']}"
    )


assert len(checkpoint_records) == 6


# ------------------------------------------------------------
# 4. Verify known-positive filtering mask
# ------------------------------------------------------------

assert known_positive_mask.shape == (
    4_278,
    4_278
)

assert known_positive_mask.dtype == torch.bool


print("\nFILTERING")
print("-" * 78)

print(
    "Candidate drugs:",
    f"{drug_node_ids.numel():,}"
)

print(
    "Known-positive mask:",
    tuple(known_positive_mask.shape)
)

print(
    "Filtering protocol:",
    "all known positive DDIs + self"
)


# ------------------------------------------------------------
# 5. Final preparation summary
# ------------------------------------------------------------

print("\n" + "=" * 78)

print("TWO-SIDED EVALUATION READY")

print("=" * 78)

print(
    "Cold drugs:",
    f"{cold_drug_ids.numel():,}"
)

print(
    "Cold-cold test pairs:",
    "651"
)

print(
    "Directions per pair:",
    "2"
)

print(
    "Total ranking queries:",
    "1,302"
)

print(
    "Models to evaluate:",
    "6"
)

print(
    "Retraining required:",
    "NO"
)

print(
    "New negatives required:",
    "NO"
)

print(
    "\nPASS: secondary cold-cold evaluation "
    "is ready."
)

In [ ]:
# ============================================================
# STEP 53 — SECONDARY TWO-SIDED COLD-COLD EVALUATION
#
# Evaluates:
#   G0 seeds 42, 43, 44
#   G3 seeds 42, 43, 44
#
# Frozen evaluation:
#   651 cold-cold test pairs
#   both directions
#   1,302 ranking queries
#
# NO TRAINING
# ============================================================

from pathlib import Path
import gc
import json

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt


SPLIT_SEED = 42
MODEL_SEEDS = [42, 43, 44]
GRAPH_VARIANTS = ["G0", "G3"]


RESULT_BASE = (
    PROJECT_DIR
    / "results"
    / "cold_start"
    / f"split_seed_{SPLIT_SEED}"
)

FINAL_DIR = (
    RESULT_BASE
    / "final"
)

FIGURE_DIR = (
    PROJECT_DIR
    / "figures"
    / "cold_start"
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("SECONDARY TWO-SIDED COLD-COLD EVALUATION")
print("=" * 80)

print(
    "Cold-cold pairs:",
    f"{cold_test_two_sided.shape[1]:,}"
)

print(
    "Directional queries:",
    f"{cold_cold_bidirectional_pairs.shape[1]:,}"
)

print(
    "Candidate drugs:",
    f"{drug_node_ids.numel():,}"
)


# ============================================================
# Helper — load frozen graph
# ============================================================

def load_eval_graph(graph_variant):

    if graph_variant == "G0":
        graph = g0_cold

        expected_total = 1_855_564
        expected_bio = 0

    elif graph_variant == "G3":
        graph = g3_cold

        expected_total = 1_992_132
        expected_bio = 136_568

    else:
        raise ValueError(
            f"Unknown graph: {graph_variant}"
        )


    edge_index = (
        graph["edge_index"]
        .to(device)
    )

    edge_type = (
        graph["edge_type"]
        .to(device)
    )


    assert (
        edge_index.shape[1]
        == expected_total
    )

    assert int(
        (edge_type == 0)
        .sum()
        .item()
    ) == 1_855_564

    assert int(
        (edge_type != 0)
        .sum()
        .item()
    ) == expected_bio


    return edge_index, edge_type


# ============================================================
# Evaluate all six checkpoints
# ============================================================

evaluation_rows = []


for graph_variant in GRAPH_VARIANTS:

    for model_seed in MODEL_SEEDS:

        print("\n" + "-" * 80)

        print(
            f"EVALUATING "
            f"{graph_variant} "
            f"SEED {model_seed}"
        )

        print("-" * 80)


        # ----------------------------------------------------
        # Load graph
        # ----------------------------------------------------

        edge_index, edge_type = (
            load_eval_graph(
                graph_variant
            )
        )


        # ----------------------------------------------------
        # Fresh model container
        # ----------------------------------------------------

        model = RGCNDDIModel(
            num_nodes=NUM_NODES,
            num_relations=NUM_RELATIONS,
            embedding_dim=EMBEDDING_DIM,
            hidden_dim=HIDDEN_DIM,
            dropout=DROPOUT
        ).to(device)


        # ----------------------------------------------------
        # Load saved BEST checkpoint
        # ----------------------------------------------------

        checkpoint_path = (
            PROJECT_DIR
            / "checkpoints"
            / "cold_start"
            / f"split_seed_{SPLIT_SEED}"
            / graph_variant
            / f"model_seed_{model_seed}"
            / "best_model.pt"
        )


        assert checkpoint_path.exists()


        checkpoint = torch.load(
            checkpoint_path,
            map_location=device
        )


        assert (
            checkpoint["graph_variant"]
            == graph_variant
        )

        assert (
            checkpoint["model_seed"]
            == model_seed
        )

        assert (
            checkpoint["split_seed"]
            == SPLIT_SEED
        )


        model.load_state_dict(
            checkpoint[
                "model_state_dict"
            ]
        )

        model.eval()


        # ----------------------------------------------------
        # Evaluate 1,302 queries
        # ----------------------------------------------------

        metrics, ranks = (
            evaluate_filtered_ranking(
                model=model,

                edge_index=edge_index,
                edge_type=edge_type,

                evaluation_pairs=
                    cold_cold_bidirectional_pairs,

                drug_node_ids=
                    drug_node_ids,

                known_positive_mask=
                    known_positive_mask,

                device=device,

                batch_size=128
            )
        )


        assert (
            metrics["num_queries"]
            == 1_302
        )

        assert ranks.numel() == 1_302

        assert bool(
            (ranks >= 1).all()
        )

        assert bool(
            (ranks <= 4_278).all()
        )


        # ----------------------------------------------------
        # Result row
        # ----------------------------------------------------

        row = {
            "graph":
                graph_variant,

            "seed":
                model_seed,

            "num_pairs":
                651,

            "num_queries":
                1_302,

            "mrr":
                float(
                    metrics[
                        "mean_reciprocal_rank"
                    ]
                ),

            "hits_at_1":
                float(
                    metrics["hits_at_1"]
                ),

            "hits_at_5":
                float(
                    metrics["hits_at_5"]
                ),

            "hits_at_10":
                float(
                    metrics["hits_at_10"]
                ),

            "mean_rank":
                float(
                    metrics["mean_rank"]
                ),

            "median_rank":
                float(
                    metrics["median_rank"]
                ),

            "min_rank":
                int(ranks.min()),

            "max_rank":
                int(ranks.max()),

            "best_epoch":
                checkpoint.get(
                    "best_epoch",
                    None
                ),

            "best_warm_val_loss":
                checkpoint.get(
                    "best_val_loss",
                    None
                ),
        }


        evaluation_rows.append(
            row
        )


        # ----------------------------------------------------
        # Save individual evaluation
        # ----------------------------------------------------

        model_result_dir = (
            RESULT_BASE
            / graph_variant
            / f"model_seed_{model_seed}"
        )

        model_result_dir.mkdir(
            parents=True,
            exist_ok=True
        )


        eval_path = (
            model_result_dir
            / "secondary_cold_cold_evaluation.json"
        )

        rank_path = (
            model_result_dir
            / "secondary_cold_cold_ranks.pt"
        )


        with open(
            eval_path,
            "w"
        ) as f:

            json.dump(
                {
                    "experiment":
                        "ddi_edge_cold_start",

                    "evaluation":
                        "secondary_two_sided_cold_cold",

                    "direction":
                        "both_directions",

                    "graph_variant":
                        graph_variant,

                    "split_seed":
                        SPLIT_SEED,

                    "model_seed":
                        model_seed,

                    **row
                },
                f,
                indent=2
            )


        torch.save(
            {
                "ranks":
                    ranks.cpu(),

                "evaluation_pairs":
                    cold_cold_bidirectional_pairs.cpu(),

                "original_undirected_pairs":
                    cold_test_two_sided.cpu(),

                "graph_variant":
                    graph_variant,

                "split_seed":
                    SPLIT_SEED,

                "model_seed":
                    model_seed,

                "direction":
                    "both_directions",
            },
            rank_path
        )


        print(
            "MRR:",
            f"{row['mrr']:.6f}"
        )

        print(
            "H@1:",
            f"{row['hits_at_1']:.6f}"
        )

        print(
            "H@5:",
            f"{row['hits_at_5']:.6f}"
        )

        print(
            "H@10:",
            f"{row['hits_at_10']:.6f}"
        )

        print(
            "Mean rank:",
            f"{row['mean_rank']:.2f}"
        )

        print(
            "Median rank:",
            f"{row['median_rank']:.2f}"
        )


        # ----------------------------------------------------
        # Free this model before next checkpoint
        # ----------------------------------------------------

        del ranks
        del metrics
        del model
        del checkpoint
        del edge_index
        del edge_type

        gc.collect()

        torch.cuda.empty_cache()


# ============================================================
# Build per-seed dataframe
# ============================================================

secondary_df = pd.DataFrame(
    evaluation_rows
)

secondary_df = (
    secondary_df
    .sort_values(
        ["graph", "seed"]
    )
    .reset_index(drop=True)
)


assert len(secondary_df) == 6


print("\n\nPER-SEED SECONDARY RESULTS")
print("=" * 80)

display(
    secondary_df
)


# ============================================================
# Mean ± sample SD
# ============================================================

metric_columns = [
    "mrr",
    "hits_at_1",
    "hits_at_5",
    "hits_at_10",
    "mean_rank",
    "median_rank",
]


summary_rows = []


for graph in GRAPH_VARIANTS:

    subset = secondary_df[
        secondary_df["graph"] == graph
    ]


    summary_row = {
        "graph": graph
    }


    for metric in metric_columns:

        values = (
            subset[metric]
            .astype(float)
            .to_numpy()
        )

        summary_row[
            f"{metric}_mean"
        ] = values.mean()

        summary_row[
            f"{metric}_sd"
        ] = values.std(
            ddof=1
        )


    summary_rows.append(
        summary_row
    )


secondary_summary_df = (
    pd.DataFrame(
        summary_rows
    )
)


print("\nMEAN ± SAMPLE SD")
print("=" * 80)


for _, row in (
    secondary_summary_df
    .iterrows()
):

    print(
        f"\n{row['graph']}"
    )

    print(
        "MRR:",
        f"{row['mrr_mean']:.6f} "
        f"± {row['mrr_sd']:.6f}"
    )

    print(
        "Hits@1:",
        f"{row['hits_at_1_mean']:.6f} "
        f"± {row['hits_at_1_sd']:.6f}"
    )

    print(
        "Hits@5:",
        f"{row['hits_at_5_mean']:.6f} "
        f"± {row['hits_at_5_sd']:.6f}"
    )

    print(
        "Hits@10:",
        f"{row['hits_at_10_mean']:.6f} "
        f"± {row['hits_at_10_sd']:.6f}"
    )

    print(
        "Mean rank:",
        f"{row['mean_rank_mean']:.2f} "
        f"± {row['mean_rank_sd']:.2f}"
    )

    print(
        "Median rank:",
        f"{row['median_rank_mean']:.2f} "
        f"± {row['median_rank_sd']:.2f}"
    )


# ============================================================
# Paired G3 - G0 differences
# ============================================================

g0 = (
    secondary_df[
        secondary_df["graph"] == "G0"
    ]
    .set_index("seed")
    .sort_index()
)

g3 = (
    secondary_df[
        secondary_df["graph"] == "G3"
    ]
    .set_index("seed")
    .sort_index()
)


assert (
    list(g0.index)
    == MODEL_SEEDS
)

assert (
    list(g3.index)
    == MODEL_SEEDS
)


delta_rows = []


for seed in MODEL_SEEDS:

    delta_rows.append(
        {
            "seed":
                seed,

            "delta_mrr":
                g3.loc[seed, "mrr"]
                - g0.loc[seed, "mrr"],

            "delta_hits_at_1":
                g3.loc[seed, "hits_at_1"]
                - g0.loc[seed, "hits_at_1"],

            "delta_hits_at_5":
                g3.loc[seed, "hits_at_5"]
                - g0.loc[seed, "hits_at_5"],

            "delta_hits_at_10":
                g3.loc[seed, "hits_at_10"]
                - g0.loc[seed, "hits_at_10"],

            # Positive rank delta means G3 is worse.
            "delta_mean_rank":
                g3.loc[seed, "mean_rank"]
                - g0.loc[seed, "mean_rank"],

            "delta_median_rank":
                g3.loc[seed, "median_rank"]
                - g0.loc[seed, "median_rank"],
        }
    )


secondary_delta_df = (
    pd.DataFrame(
        delta_rows
    )
)


print("\nPAIRED G3 - G0 DELTAS")
print("=" * 80)

display(
    secondary_delta_df
)


# ============================================================
# Delta summary
# ============================================================

print("\nPAIRED DELTA SUMMARY")
print("=" * 80)


for metric in [
    "delta_mrr",
    "delta_hits_at_1",
    "delta_hits_at_5",
    "delta_hits_at_10",
    "delta_mean_rank",
    "delta_median_rank",
]:

    values = (
        secondary_delta_df[
            metric
        ]
        .astype(float)
        .to_numpy()
    )

    print(
        f"{metric}: "
        f"{values.mean():.6f} "
        f"± "
        f"{values.std(ddof=1):.6f}"
    )


# ============================================================
# Consistency
# ============================================================

g3_better = int(
    (
        secondary_delta_df[
            "delta_mrr"
        ] > 0
    ).sum()
)

g3_worse = int(
    (
        secondary_delta_df[
            "delta_mrr"
        ] < 0
    ).sum()
)


print("\nCONSISTENCY")
print("=" * 80)

print(
    "G3 higher MRR:",
    f"{g3_better}/3 seeds"
)

print(
    "G3 lower MRR:",
    f"{g3_worse}/3 seeds"
)


# ============================================================
# Relative mean MRR change
# ============================================================

g0_mean = float(
    secondary_summary_df.loc[
        secondary_summary_df[
            "graph"
        ] == "G0",
        "mrr_mean"
    ].iloc[0]
)

g3_mean = float(
    secondary_summary_df.loc[
        secondary_summary_df[
            "graph"
        ] == "G3",
        "mrr_mean"
    ].iloc[0]
)


relative_change = (
    (g3_mean - g0_mean)
    / g0_mean
    * 100
)


print("\nMEAN MRR COMPARISON")
print("=" * 80)

print(
    "G0 mean MRR:",
    f"{g0_mean:.6f}"
)

print(
    "G3 mean MRR:",
    f"{g3_mean:.6f}"
)

print(
    "Absolute G3-G0:",
    f"{g3_mean - g0_mean:.6f}"
)

print(
    "Relative change:",
    f"{relative_change:.2f}%"
)


# ============================================================
# Save authoritative CSVs
# ============================================================

per_seed_csv = (
    FINAL_DIR
    / "cold_cold_3seed_per_seed.csv"
)

summary_csv = (
    FINAL_DIR
    / "cold_cold_3seed_summary.csv"
)

delta_csv = (
    FINAL_DIR
    / "cold_cold_3seed_paired_deltas.csv"
)


secondary_df.to_csv(
    per_seed_csv,
    index=False
)

secondary_summary_df.to_csv(
    summary_csv,
    index=False
)

secondary_delta_df.to_csv(
    delta_csv,
    index=False
)


# ============================================================
# Thesis-ready MRR figure
# ============================================================

graph_order = [
    "G0",
    "G3"
]


means = []
stds = []
individual_values = {}


for graph in graph_order:

    values = (
        secondary_df[
            secondary_df["graph"]
            == graph
        ]
        .sort_values("seed")
        ["mrr"]
        .to_numpy()
    )

    individual_values[
        graph
    ] = values

    means.append(
        values.mean()
    )

    stds.append(
        values.std(ddof=1)
    )


means = np.asarray(means)
stds = np.asarray(stds)


fig, ax = plt.subplots(
    figsize=(7.2, 5.4)
)


x = np.arange(2)


ax.bar(
    x,
    means,
    width=0.55,
    yerr=stds,
    capsize=7,
    alpha=0.72,
    edgecolor="black",
    linewidth=1.0
)


offsets = [
    -0.08,
    0.0,
    0.08
]


for graph_index, graph in enumerate(
    graph_order
):

    values = individual_values[
        graph
    ]

    for offset, seed, value in zip(
        offsets,
        MODEL_SEEDS,
        values
    ):

        ax.scatter(
            graph_index + offset,
            value,
            s=55,
            zorder=5,
            edgecolor="black",
            linewidth=0.7
        )

        ax.annotate(
            str(seed),
            (
                graph_index + offset,
                value
            ),
            xytext=(0, 7),
            textcoords="offset points",
            ha="center",
            fontsize=8
        )


for i, value in enumerate(
    means
):

    ax.text(
        i,
        max(
            0.0001,
            value * 0.08
        ),
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold"
    )


ax.set_xticks(x)

ax.set_xticklabels(
    [
        "G0\nDDI only",
        "G3\nDDI + biomedical context"
    ]
)

ax.set_ylabel(
    "Mean Reciprocal Rank (MRR)"
)

ax.set_xlabel(
    "Graph configuration"
)

ax.set_title(
    "Secondary Cold–Cold DDI-Edge "
    "Cold-Start Performance"
)

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.30
)

ax.set_axisbelow(True)


upper = max(
    means + stds
)

ax.set_ylim(
    0,
    max(
        upper * 1.30,
        0.01
    )
)


fig.text(
    0.5,
    0.01,
    "651 cold–cold test pairs evaluated in both directions "
    "(1,302 ranking queries); bars show mean ± sample SD.",
    ha="center",
    fontsize=8.5
)


fig.tight_layout(
    rect=[
        0,
        0.04,
        1,
        1
    ]
)


png_path = (
    FIGURE_DIR
    / "cold_cold_mrr_3seed.png"
)

svg_path = (
    FIGURE_DIR
    / "cold_cold_mrr_3seed.svg"
)


fig.savefig(
    png_path,
    dpi=300,
    bbox_inches="tight"
)

fig.savefig(
    svg_path,
    format="svg",
    bbox_inches="tight"
)


plt.show()


# ============================================================
# Final verification
# ============================================================

assert per_seed_csv.exists()
assert summary_csv.exists()
assert delta_csv.exists()

assert png_path.exists()
assert svg_path.exists()


print("\nSAVED")
print("=" * 80)

print(per_seed_csv)
print(summary_csv)
print(delta_csv)

print(png_path)
print(svg_path)


print("\n" + "=" * 80)

print(
    "PASS: secondary cold-cold "
    "3-seed evaluation complete."
)

print("=" * 80)

## 16. Canonical Checkpoint-Based Reproduction

A separate fresh-process reproducibility audit reloaded:

- the frozen G0/G3 cold-start graphs,
- the frozen evaluation pairs,
- the complete known-positive mask,
- the preserved model checkpoints for seeds 42, 43, and 44,
- and the canonical filtered-ranking implementation.

The control result was decisive:

- **G0 reproduced 100% of saved ranks for all three seeds** in both primary and secondary evaluation.
- **G3 did not reproduce the original saved rank vectors** from the preserved checkpoints and frozen G3 graph.

Small CUDA forward-pass nondeterminism was investigated and was far too small to explain the G3 discrepancy. The final cold-start results therefore use the fresh canonical checkpoint-based reproduction files rather than the historical G3 evaluation artifacts.

The original result files are preserved and were not overwritten.


In [ ]:
from pathlib import Path
import pandas as pd
import json

PROJECT_DIR = Path('/workspace/primekg_ddi_rgcn')
REPRO_DIR = PROJECT_DIR / 'results' / 'cold_start' / 'split_seed_42' / 'reproduced'

validated_per_seed = pd.read_csv(REPRO_DIR / 'cold_start_validated_per_seed.csv')
validated_summary = pd.read_csv(REPRO_DIR / 'cold_start_validated_summary.csv')
validated_deltas = pd.read_csv(REPRO_DIR / 'cold_start_validated_paired_deltas.csv')

with open(REPRO_DIR / 'cold_start_validated_manifest.json', 'r') as f:
    validated_manifest = json.load(f)

display(validated_summary)
display(validated_deltas[['evaluation', 'model_seed', 'delta_mrr']])


### Validated Primary Result: Cold → Warm

Across model seeds 42–44:

- **G0:** MRR = **0.140603 ± 0.058159**
- **G3:** MRR = **0.010184 ± 0.007833**

The paired G3−G0 MRR difference is negative for all three model seeds. Under this fixed context-available DDI-edge cold-start cohort, G0 therefore performs substantially better than G3 for the primary cold→warm ranking task.

This result does **not** establish that biomedical knowledge generally harms DDI prediction. In the separate transductive graph-composition experiment, G3 performs slightly better than G0. The cold-start result is specific to this evaluation protocol and cohort.


### Validated Secondary Result: Cold ↔ Cold

Across model seeds 42–44:

- **G0:** MRR = **0.000990 ± 0.000177**
- **G3:** MRR = **0.008430 ± 0.004076**

The paired G3−G0 MRR difference is positive for all three model seeds.

A plausible interpretation is that when both DDI endpoints are cold, the retained biomedical context in G3 provides useful message-passing information that is unavailable in DDI-only G0. This is an interpretation of the observed result, not a demonstrated causal mechanism.

Absolute ranking performance remains low, so the appropriate conclusion is that G3 **outperforms G0 in this secondary setting**, not that G3 solves cold-start DDI prediction.


## 17. Limitations

1. This experiment evaluates **DDI-edge cold-start**, not fully inductive unseen-node generalization. Cold drugs remain graph nodes and have learned node embeddings.
2. G3 retains biomedical edges for cold drugs, whereas their DDI edges are removed.
3. The cold cohort is fixed using split seed 42. Seeds 42–44 are model seeds only, so the reported standard deviation reflects model variation conditional on one cohort.
4. Cohort eligibility uses total DDI degree and existing test DDI degree; therefore cohort construction is not test-blind.
5. Sample standard deviations across three model seeds are descriptive. No statistical-significance claim is made.
6. Unobserved drug pairs are used as negatives and should not be interpreted as clinically confirmed non-interactions.
7. The original G3 cold-start evaluation artifacts were not reproducible from the preserved checkpoints and frozen graph. Final reporting uses the canonical checkpoint-based reproduction while preserving the historical artifacts for auditability.
